In [1]:

# CELL 1: Import required libraries and initialize project paths
# Purpose: Prepare the Python environment for multi-store forecasting.

from pathlib import Path
import polars as pl
import pandas as pd
import numpy as np
import lightgbm as lgb

# Define the main project directory
PROJECT_DIR = Path(r"D:\Retail-Demand")

# Define project subdirectories
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
MODEL_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"
PREDICTION_DIR = PROJECT_DIR / "predictions"
NOTEBOOK_DIR = PROJECT_DIR / "notebooks"

# Define forecasting configuration
HORIZON = 28
NUMBER_OF_FOLDS = 3

# Display environment information
print("IMPORTS COMPLETED SUCCESSFULLY")
print("-" * 45)
print("Polars version:", pl.__version__)
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("LightGBM version:", lgb.__version__)

print("\nPROJECT DIRECTORY:", PROJECT_DIR)
print("FORECAST HORIZON:", HORIZON)
print("NUMBER OF FOLDS:", NUMBER_OF_FOLDS)


IMPORTS COMPLETED SUCCESSFULLY
---------------------------------------------
Polars version: 2.1.0
Pandas version: 3.1.1
NumPy version: 2.5.5
LightGBM version: 4.7.0

PROJECT DIRECTORY: D:\Retail-Demand
FORECAST HORIZON: 28
NUMBER OF FOLDS: 3


In [2]:

# CELL 2: Verify project folders and existing datasets
# Purpose: Ensure all required directories and the original panel exist.

folders = [
    RAW_DATA_DIR,
    PROCESSED_DIR,
    MODEL_DIR,
    RESULTS_DIR,
    PREDICTION_DIR,
    NOTEBOOK_DIR
]

print("PROJECT DIRECTORY VERIFICATION")
print("-" * 45)

for folder in folders:
    print(f"{folder.name}: {'EXISTS' if folder.exists() else 'MISSING'}")

# Verify the original panel dataset
PANEL_PATH = PROCESSED_DIR / "m5_panel.parquet"

print("\nOriginal panel dataset:")
print("Exists:", PANEL_PATH.exists())

if PANEL_PATH.exists():
    print("Size (MB):", round(PANEL_PATH.stat().st_size / (1024**2), 2))


PROJECT DIRECTORY VERIFICATION
---------------------------------------------
raw: EXISTS
processed: EXISTS
models: EXISTS
results: EXISTS
predictions: EXISTS
notebooks: EXISTS

Original panel dataset:
Exists: True
Size (MB): 170.29


In [3]:
# CELL 3: Load and inspect the existing CA_1 feature dataset

FEATURE_PATH = PROCESSED_DIR / "ca1_advanced_features.parquet"

ca1_features_check = pl.read_parquet(FEATURE_PATH)

print("Dataset shape:", ca1_features_check.shape)
print("Columns:", ca1_features_check.columns)
print("Unique stores:", ca1_features_check["store_id"].unique().to_list())
print("Unique products:", ca1_features_check["item_id"].n_unique())
print("Start date:", ca1_features_check["date"].min())
print("End date:", ca1_features_check["date"].max())

print("\nSample records:")
print(ca1_features_check.head(5))

Dataset shape: (5832737, 30)
Columns: ['item_id', 'store_id', 'date', 'sales', 'sell_price', 'event_name_1', 'snap_flag', 'lag_28', 'lag_35', 'lag_42', 'lag_56', 'lag_364', 'day_of_week', 'month', 'rolling_mean_7', 'rolling_mean_28', 'rolling_mean_56', 'rolling_std_7', 'rolling_std_28', 'day_of_week_sin', 'day_of_week_cos', 'month_sin', 'month_cos', 'ema_28', 'zero_sales_run', 'price_lag_28', 'price_lag_56', 'price_momentum', 'historical_max_price_56', 'discount_ratio']
Unique stores: ['CA_1']
Unique products: 3049
Start date: 2011-01-29
End date: 2016-04-24

Sample records:
shape: (5, 30)
┌─────────────┬──────────┬────────────┬───────┬───┬──────────────┬────────────────┬───────────────────┬────────────────┐
│ item_id     ┆ store_id ┆ date       ┆ sales ┆ … ┆ price_lag_56 ┆ price_momentum ┆ historical_max_pr ┆ discount_ratio │
│ ---         ┆ ---      ┆ ---        ┆ ---   ┆   ┆ ---          ┆ ---            ┆ ice_56            ┆ ---            │
│ str         ┆ str      ┆ date       ┆ 

In [4]:

# CELL 4: Load and inspect the CA_2 store
# Purpose: Verify the second store's raw panel data before
# applying the existing CA_1 feature engineering pipeline.

PANEL_PATH = PROCESSED_DIR / "m5_panel.parquet"

# Load only CA_2 from the existing panel dataset
ca2_panel = (
    pl.scan_parquet(PANEL_PATH)
    .filter(pl.col("store_id") == "CA_2")
    .sort(["item_id", "date"])
    .collect()
)

# Display basic dataset information
print("=" * 55)
print("CA_2 RAW PANEL VERIFICATION")
print("=" * 55)

print("Dataset shape:", ca2_panel.shape)
print("Unique stores:", ca2_panel["store_id"].unique().to_list())
print("Unique products:", ca2_panel["item_id"].n_unique())

print("Start date:", ca2_panel["date"].min())
print("End date:", ca2_panel["date"].max())

print("\nColumn names:")
print(ca2_panel.columns)

print("\nSample records:")
print(ca2_panel.head(5))

print("\nCA_2 PANEL INSPECTION COMPLETED")


CA_2 RAW PANEL VERIFICATION
Dataset shape: (5832737, 7)
Unique stores: ['CA_2']
Unique products: 3049
Start date: 2011-01-29
End date: 2016-04-24

Column names:
['item_id', 'store_id', 'date', 'sales', 'sell_price', 'event_name_1', 'snap_flag']

Sample records:
shape: (5, 7)
┌─────────────┬──────────┬────────────┬───────┬────────────┬──────────────┬───────────┐
│ item_id     ┆ store_id ┆ date       ┆ sales ┆ sell_price ┆ event_name_1 ┆ snap_flag │
│ ---         ┆ ---      ┆ ---        ┆ ---   ┆ ---        ┆ ---          ┆ ---       │
│ str         ┆ str      ┆ date       ┆ i64   ┆ f64        ┆ str          ┆ i64       │
╞═════════════╪══════════╪════════════╪═══════╪════════════╪══════════════╪═══════════╡
│ FOODS_1_001 ┆ CA_2     ┆ 2011-01-29 ┆ 2     ┆ 2.0        ┆ null         ┆ 0         │
│ FOODS_1_001 ┆ CA_2     ┆ 2011-01-30 ┆ 0     ┆ 2.0        ┆ null         ┆ 0         │
│ FOODS_1_001 ┆ CA_2     ┆ 2011-01-31 ┆ 0     ┆ 2.0        ┆ null         ┆ 0         │
│ FOODS_1_001 ┆ CA_2

In [5]:

# CELL 5: Create reusable advanced feature engineering function
# Purpose: Apply the same historical, temporal and price features
# to any store in the M5 dataset.

def create_advanced_features(store_panel):

    # Sort records to ensure chronological feature calculations
    df = store_panel.sort(["item_id", "date"])

    # --------------------------------------------------
    # 1. Historical sales lag features
    # --------------------------------------------------
    df = df.with_columns([
        pl.col("sales").shift(28).over("item_id").alias("lag_28"),
        pl.col("sales").shift(35).over("item_id").alias("lag_35"),
        pl.col("sales").shift(42).over("item_id").alias("lag_42"),
        pl.col("sales").shift(56).over("item_id").alias("lag_56"),
        pl.col("sales").shift(364).over("item_id").alias("lag_364"),
        pl.col("sales").shift(28).over("item_id").alias("_sales_gap28")
    ])

    # --------------------------------------------------
    # 2. Calendar features
    # --------------------------------------------------
    df = df.with_columns([
        pl.col("date").dt.weekday().alias("day_of_week"),
        pl.col("date").dt.month().alias("month")
    ])

    # --------------------------------------------------
    # 3. Rolling historical statistics
    # --------------------------------------------------
    df = df.with_columns([
        pl.col("_sales_gap28").rolling_mean(7).over("item_id").alias("rolling_mean_7"),
        pl.col("_sales_gap28").rolling_mean(28).over("item_id").alias("rolling_mean_28"),
        pl.col("_sales_gap28").rolling_mean(56).over("item_id").alias("rolling_mean_56"),
        pl.col("_sales_gap28").rolling_std(7).over("item_id").alias("rolling_std_7"),
        pl.col("_sales_gap28").rolling_std(28).over("item_id").alias("rolling_std_28")
    ])

    # --------------------------------------------------
    # 4. Cyclical calendar encoding
    # --------------------------------------------------
    df = df.with_columns([
        (2 * np.pi * pl.col("day_of_week") / 7).sin().alias("day_of_week_sin"),
        (2 * np.pi * pl.col("day_of_week") / 7).cos().alias("day_of_week_cos"),
        (2 * np.pi * pl.col("month") / 12).sin().alias("month_sin"),
        (2 * np.pi * pl.col("month") / 12).cos().alias("month_cos")
    ])

    # --------------------------------------------------
    # 5. Exponential moving average
    # --------------------------------------------------
    df = df.with_columns([
        pl.col("_sales_gap28")
        .ewm_mean(span=28, adjust=False)
        .over("item_id")
        .alias("ema_28")
    ])

    # --------------------------------------------------
    # 6. Zero-sales run length
    # --------------------------------------------------
    df = df.with_columns([
        pl.col("_sales_gap28")
        .is_not_null()
        .cast(pl.Int64)
        .alias("_valid_gap")
    ])

    df = df.with_columns([
        pl.when(pl.col("_valid_gap") == 1)
        .then(
            (pl.col("_sales_gap28") > 0)
            .cast(pl.Int64)
        )
        .otherwise(0)
        .alias("_positive_sales")
    ])

    df = df.with_columns([
        pl.col("_positive_sales")
        .cum_sum()
        .over("item_id")
        .alias("_zero_group")
    ])

    df = df.with_columns([
        pl.when(pl.col("_sales_gap28").is_null())
        .then(None)
        .when(pl.col("_sales_gap28") > 0)
        .then(0)
        .otherwise(
            pl.int_range(1, pl.len() + 1)
            .over(["item_id", "_zero_group"])
        )
        .alias("zero_sales_run")
    ])

    # --------------------------------------------------
    # 7. Historical price features
    # --------------------------------------------------
    df = df.with_columns([
        pl.col("sell_price").shift(28).over("item_id").alias("price_lag_28"),
        pl.col("sell_price").shift(56).over("item_id").alias("price_lag_56"),
        pl.col("sell_price")
        .shift(28)
        .rolling_max(window_size=56)
        .over("item_id")
        .alias("historical_max_price_56")
    ])

    df = df.with_columns([
        pl.when(pl.col("price_lag_56") > 0)
        .then(
            pl.col("price_lag_28") / pl.col("price_lag_56") - 1
        )
        .otherwise(None)
        .alias("price_momentum"),

        pl.when(pl.col("historical_max_price_56") > 0)
        .then(
            pl.col("price_lag_28") / pl.col("historical_max_price_56")
        )
        .otherwise(None)
        .alias("discount_ratio")
    ])

    # --------------------------------------------------
    # 8. Remove temporary calculation columns
    # --------------------------------------------------
    df = df.drop([
        "_sales_gap28",
        "_valid_gap",
        "_positive_sales",
        "_zero_group"
    ])

    return df


In [6]:

# CELL 6: Generate advanced features for CA_2
# Purpose: Apply the reusable feature engineering function
# and save the resulting dataset.

ca2_features = create_advanced_features(ca2_panel)

print("CA_2 FEATURE ENGINEERING RESULTS")
print("-" * 50)

print("Dataset shape:", ca2_features.shape)
print("Store:", ca2_features["store_id"].unique().to_list())
print("Products:", ca2_features["item_id"].n_unique())

print("\nFeature columns:")
print(ca2_features.columns)

# Save the generated feature dataset
CA2_FEATURE_PATH = PROCESSED_DIR / "ca2_advanced_features.parquet"

ca2_features.write_parquet(CA2_FEATURE_PATH)

print("\nSaved:", CA2_FEATURE_PATH)
print("File size (MB):", round(
    CA2_FEATURE_PATH.stat().st_size / (1024**2), 2
))


CA_2 FEATURE ENGINEERING RESULTS
--------------------------------------------------
Dataset shape: (5832737, 30)
Store: ['CA_2']
Products: 3049

Feature columns:
['item_id', 'store_id', 'date', 'sales', 'sell_price', 'event_name_1', 'snap_flag', 'lag_28', 'lag_35', 'lag_42', 'lag_56', 'lag_364', 'day_of_week', 'month', 'rolling_mean_7', 'rolling_mean_28', 'rolling_mean_56', 'rolling_std_7', 'rolling_std_28', 'day_of_week_sin', 'day_of_week_cos', 'month_sin', 'month_cos', 'ema_28', 'zero_sales_run', 'price_lag_28', 'price_lag_56', 'historical_max_price_56', 'price_momentum', 'discount_ratio']

Saved: D:\Retail-Demand\data\processed\ca2_advanced_features.parquet
File size (MB): 65.02


In [7]:

# CELL 7: Validate the generated CA_2 advanced features
# Purpose: Check data quality, duplicate records, missing values,
# and validity of the engineered numerical features.

print("=" * 60)
print("CA_2 ADVANCED FEATURE VALIDATION")
print("=" * 60)

# 1. Verify dataset dimensions
print("\n1. DATASET DIMENSIONS")
print("Shape:", ca2_features.shape)

# 2. Check duplicate product-store-date records
duplicate_count = (
    ca2_features
    .group_by(["item_id", "store_id", "date"])
    .len()
    .filter(pl.col("len") > 1)
    .height
)

print("\n2. DUPLICATE RECORDS")
print("Duplicates:", duplicate_count)

# 3. Check missing values in every column
print("\n3. MISSING VALUE COUNTS")
null_counts = ca2_features.null_count()

for column in ca2_features.columns:
    count = null_counts[column][0]
    if count > 0:
        print(f"{column}: {count:,}")

# 4. Check invalid sales and feature values
print("\n4. INVALID VALUE CHECKS")

checks = {
    "Negative sales": pl.col("sales") < 0,
    "Negative EMA": pl.col("ema_28") < 0,
    "Negative rolling mean": pl.col("rolling_mean_7") < 0,
    "Negative rolling std": pl.col("rolling_std_7") < 0,
    "Price momentum below -1": pl.col("price_momentum") < -1,
    "Discount ratio above 1": pl.col("discount_ratio") > 1,
    "Negative zero-sales run": pl.col("zero_sales_run") < 0
}

for name, condition in checks.items():
    count = ca2_features.filter(condition).height
    print(f"{name}: {count:,}")

# 5. Verify feature dataset file
print("\n5. SAVED FILE VERIFICATION")
print("Exists:", CA2_FEATURE_PATH.exists())

if CA2_FEATURE_PATH.exists():
    print(
        "File size (MB):",
        round(CA2_FEATURE_PATH.stat().st_size / (1024**2), 2)
    )

print("\nCA_2 FEATURE VALIDATION COMPLETED")


CA_2 ADVANCED FEATURE VALIDATION

1. DATASET DIMENSIONS
Shape: (5832737, 30)

2. DUPLICATE RECORDS
Duplicates: 0

3. MISSING VALUE COUNTS
sell_price: 1,556,961
event_name_1: 5,363,191
lag_28: 85,372
lag_35: 106,715
lag_42: 128,058
lag_56: 170,744
lag_364: 1,109,836
rolling_mean_7: 103,666
rolling_mean_28: 167,695
rolling_mean_56: 253,067
rolling_std_7: 103,666
rolling_std_28: 167,695
ema_28: 85,372
zero_sales_run: 85,372
price_lag_28: 1,642,333
price_lag_56: 1,727,705
historical_max_price_56: 1,810,006
price_momentum: 1,727,705
discount_ratio: 1,810,006

4. INVALID VALUE CHECKS
Negative sales: 0
Negative EMA: 0
Negative rolling mean: 0
Negative rolling std: 0
Price momentum below -1: 0
Discount ratio above 1: 0
Negative zero-sales run: 0

5. SAVED FILE VERIFICATION
Exists: True
File size (MB): 65.02

CA_2 FEATURE VALIDATION COMPLETED


In [8]:

# CELL 8: Compare reusable features against the original CA_1 features
# Purpose: Verify that our new function reproduces the original
# feature engineering calculations for selected complete product histories.

# Select three products from the existing CA_1 dataset
sample_items = (
    ca1_features_check
    .select("item_id")
    .unique()
    .head(3)["item_id"]
    .to_list()
)

print("Selected products:", sample_items)

# Load the original raw CA_1 history for these products
ca1_sample_panel = (
    pl.scan_parquet(PANEL_PATH)
    .filter(
        (pl.col("store_id") == "CA_1") &
        (pl.col("item_id").is_in(sample_items))
    )
    .sort(["item_id", "date"])
    .collect()
)

# Load their original saved advanced features
ca1_original_sample = (
    ca1_features_check
    .filter(pl.col("item_id").is_in(sample_items))
    .sort(["item_id", "date"])
)

# Recalculate features using the reusable function
ca1_recreated_sample = create_advanced_features(ca1_sample_panel)

# Compare the original and recreated feature values
comparison_columns = [
    "lag_28", "lag_35", "lag_42", "lag_56", "lag_364",
    "rolling_mean_7", "rolling_mean_28", "rolling_mean_56",
    "rolling_std_7", "rolling_std_28",
    "day_of_week_sin", "day_of_week_cos",
    "month_sin", "month_cos", "ema_28",
    "zero_sales_run", "price_lag_28", "price_lag_56",
    "price_momentum", "historical_max_price_56",
    "discount_ratio"
]

# Join on product and date to align records
comparison = (
    ca1_original_sample
    .select(["item_id", "date"] + comparison_columns)
    .join(
        ca1_recreated_sample.select(
            ["item_id", "date"] + comparison_columns
        ),
        on=["item_id", "date"],
        suffix="_new"
    )
)

# Calculate mismatch counts for each feature
print("\nFEATURE COMPARISON")
print("-" * 55)

for column in comparison_columns:
    original = pl.col(column)
    recreated = pl.col(f"{column}_new")

    mismatch = (
        comparison
        .filter(
            ~(
                (original.is_null() & recreated.is_null()) |
                (
                    original.is_not_null() &
                    recreated.is_not_null() &
                    ((original - recreated).abs() <= 1e-8)
                )
            )
        )
        .height
    )

    print(f"{column}: {mismatch} mismatches")

print("\nCompared rows:", comparison.height)
print("REFERENCE COMPARISON COMPLETED")


Selected products: ['FOODS_3_096', 'HOBBIES_1_417', 'HOUSEHOLD_1_501']

FEATURE COMPARISON
-------------------------------------------------------
lag_28: 0 mismatches
lag_35: 0 mismatches
lag_42: 0 mismatches
lag_56: 0 mismatches
lag_364: 0 mismatches
rolling_mean_7: 0 mismatches
rolling_mean_28: 0 mismatches
rolling_mean_56: 0 mismatches
rolling_std_7: 0 mismatches
rolling_std_28: 0 mismatches
day_of_week_sin: 5739 mismatches
day_of_week_cos: 4920 mismatches
month_sin: 5739 mismatches
month_cos: 5739 mismatches
ema_28: 0 mismatches
zero_sales_run: 3186 mismatches
price_lag_28: 0 mismatches
price_lag_56: 0 mismatches
price_momentum: 0 mismatches
historical_max_price_56: 0 mismatches
discount_ratio: 0 mismatches

Compared rows: 5739
REFERENCE COMPARISON COMPLETED


In [9]:

# CELL 9: Diagnose calendar and zero-sales run mismatches
# Purpose: Compare original CA_1 values with the reusable
# function's values to identify the exact calculation differences.

# Select the important columns from both datasets
diagnostic = (
    ca1_original_sample
    .select([
        "item_id", "date", "day_of_week", "month",
        "day_of_week_sin", "day_of_week_cos",
        "month_sin", "month_cos", "zero_sales_run"
    ])
    .join(
        ca1_recreated_sample.select([
            "item_id", "date",
            "day_of_week", "month",
            "day_of_week_sin", "day_of_week_cos",
            "month_sin", "month_cos", "zero_sales_run"
        ]),
        on=["item_id", "date"],
        suffix="_new"
    )
    .sort(["item_id", "date"])
)

# Display a sample of calendar encoding differences
print("CALENDAR FEATURE COMPARISON")
print("-" * 65)

print(
    diagnostic.select([
        "date",
        "day_of_week", "day_of_week_new",
        "month", "month_new",
        "day_of_week_sin", "day_of_week_sin_new",
        "month_sin", "month_sin_new"
    ]).head(10)
)

# Display records where zero-sales run length differs
print("\nZERO-SALES RUN LENGTH COMPARISON")
print("-" * 65)

zero_mismatches = diagnostic.filter(
    ~(
        (pl.col("zero_sales_run").is_null() &
         pl.col("zero_sales_run_new").is_null()) |
        (pl.col("zero_sales_run") ==
         pl.col("zero_sales_run_new"))
    )
)

print("Total mismatches:", zero_mismatches.height)

print(
    zero_mismatches.select([
        "item_id",
        "date",
        "zero_sales_run",
        "zero_sales_run_new"
    ]).head(20)
)

print("\nDIAGNOSTIC COMPLETED")


CALENDAR FEATURE COMPARISON
-----------------------------------------------------------------
shape: (10, 9)
┌────────────┬─────────────┬─────────────────┬───────┬───┬────────────────┬────────────────┬───────────┬───────────────┐
│ date       ┆ day_of_week ┆ day_of_week_new ┆ month ┆ … ┆ day_of_week_si ┆ day_of_week_si ┆ month_sin ┆ month_sin_new │
│ ---        ┆ ---         ┆ ---             ┆ ---   ┆   ┆ n              ┆ n_new          ┆ ---       ┆ ---           │
│ date       ┆ i8          ┆ i8              ┆ i8    ┆   ┆ ---            ┆ ---            ┆ f64       ┆ f64           │
│            ┆             ┆                 ┆       ┆   ┆ f64            ┆ f64            ┆           ┆               │
╞════════════╪═════════════╪═════════════════╪═══════╪═══╪════════════════╪════════════════╪═══════════╪═══════════════╡
│ 2011-01-29 ┆ 6           ┆ 6               ┆ 1     ┆ … ┆ -0.974928      ┆ -0.781831      ┆ 0.0       ┆ 0.5           │
│ 2011-01-30 ┆ 7           ┆ 7              

In [10]:

# CELL 10: Correct cyclical calendar encoding
# Purpose: Match the original CA_1 zero-based calendar encoding.

ca1_recreated_sample = ca1_recreated_sample.with_columns([
    (
        2 * np.pi * (pl.col("day_of_week") - 1) / 7
    ).sin().alias("day_of_week_sin"),

    (
        2 * np.pi * (pl.col("day_of_week") - 1) / 7
    ).cos().alias("day_of_week_cos"),

    (
        2 * np.pi * (pl.col("month") - 1) / 12
    ).sin().alias("month_sin"),

    (
        2 * np.pi * (pl.col("month") - 1) / 12
    ).cos().alias("month_cos")
])

print("Calendar encoding corrected.")


Calendar encoding corrected.


In [12]:

# CELL 11: Correct zero-sales run-length calculation
# Purpose: Count consecutive zero-sales days, restarting at 1
# after every positive-sales observation.

zero_run_fixed = (
    ca1_recreated_sample
    .sort(["item_id", "date"])
    .with_columns([
        (pl.col("lag_28") > 0)
        .fill_null(False)
        .cast(pl.Int64)
        .alias("_positive_sales")
    ])
    .with_columns([
        pl.col("_positive_sales")
        .cum_sum()
        .over("item_id")
        .alias("_zero_group")
    ])
    .with_columns([
        pl.when(pl.col("lag_28").is_null())
        .then(None)
        .when(pl.col("lag_28") > 0)
        .then(0)
        .otherwise(
            (pl.col("lag_28") == 0)
            .cast(pl.Int64)
            .cum_sum()
            .over(["item_id", "_zero_group"])
        )
        .alias("zero_sales_run")
    ])
    .drop(["_positive_sales", "_zero_group"])
)

ca1_recreated_sample = zero_run_fixed

print("Corrected zero-sales run calculation applied.")


Corrected zero-sales run calculation applied.


In [14]:

# CELL 12: Verify all 21 engineered features against original CA_1
# Purpose: Confirm that the reusable feature calculations
# reproduce the original CA_1 feature values.

comparison_columns = [
    "lag_28", "lag_35", "lag_42", "lag_56", "lag_364",
    "rolling_mean_7", "rolling_mean_28", "rolling_mean_56",
    "rolling_std_7", "rolling_std_28",
    "day_of_week_sin", "day_of_week_cos",
    "month_sin", "month_cos", "ema_28",
    "zero_sales_run", "price_lag_28", "price_lag_56",
    "price_momentum", "historical_max_price_56",
    "discount_ratio"
]

# Align original and recreated feature values
comparison = (
    ca1_original_sample
    .select(["item_id", "date"] + comparison_columns)
    .join(
        ca1_recreated_sample.select(
            ["item_id", "date"] + comparison_columns
        ),
        on=["item_id", "date"],
        suffix="_new"
    )
)

# Compare each feature, treating two null values as equal
mismatch_results = []

for column in comparison_columns:
    original = pl.col(column)
    recreated = pl.col(f"{column}_new")

    mismatch_count = comparison.filter(
        ~(
            (original.is_null() & recreated.is_null()) |
            (
                original.is_not_null() &
                recreated.is_not_null() &
                ((original - recreated).abs() <= 1e-8)
            )
        )
    ).height

    mismatch_results.append((column, mismatch_count))
    print(f"{column}: {mismatch_count} mismatches")

# Summarize comparison
total_mismatches = sum(count for _, count in mismatch_results)

print("\n" + "=" * 50)
print("TOTAL COMPARED FEATURES:", len(comparison_columns))
print("TOTAL COMPARED ROWS:", comparison.height)
print("TOTAL MISMATCHES:", total_mismatches)

if total_mismatches == 0:
    print("SUCCESS: ALL FEATURES MATCH THE ORIGINAL CA_1 DATASET")
else:
    print("WARNING: SOME FEATURES STILL NEED INVESTIGATION")


lag_28: 0 mismatches
lag_35: 0 mismatches
lag_42: 0 mismatches
lag_56: 0 mismatches
lag_364: 0 mismatches
rolling_mean_7: 0 mismatches
rolling_mean_28: 0 mismatches
rolling_mean_56: 0 mismatches
rolling_std_7: 0 mismatches
rolling_std_28: 0 mismatches
day_of_week_sin: 5739 mismatches
day_of_week_cos: 4920 mismatches
month_sin: 5739 mismatches
month_cos: 5739 mismatches
ema_28: 0 mismatches
zero_sales_run: 4469 mismatches
price_lag_28: 0 mismatches
price_lag_56: 0 mismatches
price_momentum: 0 mismatches
historical_max_price_56: 0 mismatches
discount_ratio: 0 mismatches

TOTAL COMPARED FEATURES: 21
TOTAL COMPARED ROWS: 5739
TOTAL MISMATCHES: 26606


In [15]:

# CELL 13: Reapply the correct original calendar encoding
# Purpose: Update all four cyclical features in the CA_1 sample.

ca1_recreated_sample = ca1_recreated_sample.with_columns([
    (2 * np.pi * (pl.col("day_of_week") - 1) / 7)
        .sin().alias("day_of_week_sin"),

    (2 * np.pi * (pl.col("day_of_week") - 1) / 7)
        .cos().alias("day_of_week_cos"),

    (2 * np.pi * (pl.col("month") - 1) / 12)
        .sin().alias("month_sin"),

    (2 * np.pi * (pl.col("month") - 1) / 12)
        .cos().alias("month_cos")
])

print("Calendar features reapplied.")


Calendar features reapplied.


In [16]:

# CELL 14: Inspect zero-sales run-length differences
# Purpose: Compare shifted historical sales and original/new
# zero-sales counters for the first selected product.

zero_diagnostic = (
    ca1_original_sample
    .select([
        "item_id", "date", "lag_28", "zero_sales_run"
    ])
    .join(
        ca1_recreated_sample.select([
            "item_id", "date", "zero_sales_run"
        ]),
        on=["item_id", "date"],
        suffix="_new"
    )
    .sort(["item_id", "date"])
)

print("ZERO-SALES SEQUENCE")
print("-" * 60)

print(
    zero_diagnostic
    .filter(pl.col("item_id") == sample_items[0])
    .select([
        "date",
        "lag_28",
        "zero_sales_run",
        "zero_sales_run_new"
    ])
    .filter(pl.col("date").is_between(
        pd.Timestamp("2011-02-20"),
        pd.Timestamp("2011-03-10")
    ))
)


ZERO-SALES SEQUENCE
------------------------------------------------------------
shape: (19, 4)
┌────────────┬────────┬────────────────┬────────────────────┐
│ date       ┆ lag_28 ┆ zero_sales_run ┆ zero_sales_run_new │
│ ---        ┆ ---    ┆ ---            ┆ ---                │
│ date       ┆ i64    ┆ u32            ┆ i64                │
╞════════════╪════════╪════════════════╪════════════════════╡
│ 2011-02-20 ┆ null   ┆ null           ┆ null               │
│ 2011-02-21 ┆ null   ┆ null           ┆ null               │
│ 2011-02-22 ┆ null   ┆ null           ┆ null               │
│ 2011-02-23 ┆ null   ┆ null           ┆ null               │
│ 2011-02-24 ┆ null   ┆ null           ┆ null               │
│ …          ┆ …      ┆ …              ┆ …                  │
│ 2011-03-06 ┆ 0      ┆ 9              ┆ 37                 │
│ 2011-03-07 ┆ 0      ┆ 10             ┆ 38                 │
│ 2011-03-08 ┆ 0      ┆ 11             ┆ 39                 │
│ 2011-03-09 ┆ 0      ┆ 12          

In [17]:

# CELL 15: Inspect historical sales around the zero-sales sequence
# Purpose: Identify where positive sales occur and where the
# original zero-sales counter starts and resets.

product_to_check = sample_items[0]

zero_sequence_detail = (
    ca1_original_sample
    .filter(pl.col("item_id") == product_to_check)
    .filter(
        pl.col("date").is_between(
            pd.Timestamp("2011-02-20"),
            pd.Timestamp("2011-03-10")
        )
    )
    .select([
        "date",
        "lag_28",
        "zero_sales_run"
    ])
    .join(
        ca1_recreated_sample
        .filter(pl.col("item_id") == product_to_check)
        .select(["date", "zero_sales_run"])
        .rename({"zero_sales_run": "zero_sales_run_new"}),
        on="date"
    )
    .sort("date")
)

print("FULL ZERO-SALES SEQUENCE")
print("-" * 60)
print(zero_sequence_detail)


FULL ZERO-SALES SEQUENCE
------------------------------------------------------------
shape: (19, 4)
┌────────────┬────────┬────────────────┬────────────────────┐
│ date       ┆ lag_28 ┆ zero_sales_run ┆ zero_sales_run_new │
│ ---        ┆ ---    ┆ ---            ┆ ---                │
│ date       ┆ i64    ┆ u32            ┆ i64                │
╞════════════╪════════╪════════════════╪════════════════════╡
│ 2011-02-20 ┆ null   ┆ null           ┆ null               │
│ 2011-02-21 ┆ null   ┆ null           ┆ null               │
│ 2011-02-22 ┆ null   ┆ null           ┆ null               │
│ 2011-02-23 ┆ null   ┆ null           ┆ null               │
│ 2011-02-24 ┆ null   ┆ null           ┆ null               │
│ …          ┆ …      ┆ …              ┆ …                  │
│ 2011-03-06 ┆ 0      ┆ 9              ┆ 37                 │
│ 2011-03-07 ┆ 0      ┆ 10             ┆ 38                 │
│ 2011-03-08 ┆ 0      ┆ 11             ┆ 39                 │
│ 2011-03-09 ┆ 0      ┆ 12     

In [18]:

# CELL 16: Inspect every historical value in the zero-sales sequence
# Purpose: Find the exact reset point in the original counter.

print("COMPLETE ZERO-SALES SEQUENCE")
print("-" * 65)

print(
    zero_sequence_detail
    .select([
        "date",
        "lag_28",
        "zero_sales_run",
        "zero_sales_run_new"
    ])
    .to_pandas()
    .to_string(index=False)
)

# Independently calculate the zero-sales run using Python
print("\nINDEPENDENT COUNTER CHECK")
print("-" * 65)

historical_values = zero_sequence_detail["lag_28"].to_list()

manual_run = 0

for date, sales in zip(zero_sequence_detail["date"].to_list(), historical_values):
    if sales is None:
        manual_run = None
    elif sales > 0:
        manual_run = 0
    else:
        manual_run = (manual_run or 0) + 1

    print(date, "| Historical sales:", sales, "| Manual run:", manual_run)


COMPLETE ZERO-SALES SEQUENCE
-----------------------------------------------------------------
      date  lag_28  zero_sales_run  zero_sales_run_new
2011-02-20     NaN             NaN                 NaN
2011-02-21     NaN             NaN                 NaN
2011-02-22     NaN             NaN                 NaN
2011-02-23     NaN             NaN                 NaN
2011-02-24     NaN             NaN                 NaN
2011-02-25     NaN             NaN                 NaN
2011-02-26     0.0             1.0                29.0
2011-02-27     0.0             2.0                30.0
2011-02-28     0.0             3.0                31.0
2011-03-01     0.0             4.0                32.0
2011-03-02     0.0             5.0                33.0
2011-03-03     0.0             6.0                34.0
2011-03-04     0.0             7.0                35.0
2011-03-05     0.0             8.0                36.0
2011-03-06     0.0             9.0                37.0
2011-03-07     0.0       

In [19]:

# CELL 17: Verify the four corrected calendar features

calendar_columns = [
    "day_of_week_sin",
    "day_of_week_cos",
    "month_sin",
    "month_cos"
]

for column in calendar_columns:
    original = pl.col(column)
    recreated = pl.col(f"{column}_new")

    mismatch_count = comparison.filter(
        ~(
            (original.is_null() & recreated.is_null()) |
            (
                original.is_not_null() &
                recreated.is_not_null() &
                ((original - recreated).abs() <= 1e-8)
            )
        )
    ).height

    print(f"{column}: {mismatch_count} mismatches")


day_of_week_sin: 5739 mismatches
day_of_week_cos: 4920 mismatches
month_sin: 5739 mismatches
month_cos: 5739 mismatches


In [20]:

# CELL 18: Debug the zero-sales counter calculation
# Purpose: Inspect the zero-group and cumulative zero count
# for FOODS_3_096 before changing the reusable function.

debug_zero = (
    ca1_recreated_sample
    .filter(pl.col("item_id") == sample_items[0])
    .sort("date")
    .with_columns([
        (pl.col("lag_28") > 0)
        .fill_null(False)
        .cast(pl.Int64)
        .alias("_debug_positive")
    ])
    .with_columns([
        pl.col("_debug_positive")
        .cum_sum()
        .alias("_debug_group")
    ])
    .with_columns([
        (pl.col("lag_28") == 0)
        .fill_null(False)
        .cast(pl.Int64)
        .alias("_debug_zero")
    ])
    .with_columns([
        pl.col("_debug_zero")
        .cum_sum()
        .over("_debug_group")
        .alias("_debug_zero_count")
    ])
)

print(
    debug_zero
    .filter(
        pl.col("date").is_between(
            pd.Timestamp("2011-02-20"),
            pd.Timestamp("2011-03-10")
        )
    )
    .select([
        "date",
        "lag_28",
        "_debug_positive",
        "_debug_group",
        "_debug_zero",
        "_debug_zero_count",
        "zero_sales_run"
    ])
)


shape: (19, 7)
┌────────────┬────────┬─────────────────┬──────────────┬─────────────┬───────────────────┬────────────────┐
│ date       ┆ lag_28 ┆ _debug_positive ┆ _debug_group ┆ _debug_zero ┆ _debug_zero_count ┆ zero_sales_run │
│ ---        ┆ ---    ┆ ---             ┆ ---          ┆ ---         ┆ ---               ┆ ---            │
│ date       ┆ i64    ┆ i64             ┆ i64          ┆ i64         ┆ i64               ┆ i64            │
╞════════════╪════════╪═════════════════╪══════════════╪═════════════╪═══════════════════╪════════════════╡
│ 2011-02-20 ┆ null   ┆ 0               ┆ 0            ┆ 0           ┆ 0                 ┆ null           │
│ 2011-02-21 ┆ null   ┆ 0               ┆ 0            ┆ 0           ┆ 0                 ┆ null           │
│ 2011-02-22 ┆ null   ┆ 0               ┆ 0            ┆ 0           ┆ 0                 ┆ null           │
│ 2011-02-23 ┆ null   ┆ 0               ┆ 0            ┆ 0           ┆ 0                 ┆ null           │
│ 2011-02-24 

In [21]:

# CELL 19: Rebuild the comparison using updated calendar features
# Purpose: Avoid comparing against the stale DataFrame from
# before the calendar correction.

calendar_comparison = (
    ca1_original_sample
    .select([
        "item_id", "date",
        "day_of_week_sin", "day_of_week_cos",
        "month_sin", "month_cos"
    ])
    .join(
        ca1_recreated_sample.select([
            "item_id", "date",
            "day_of_week_sin", "day_of_week_cos",
            "month_sin", "month_cos"
        ]),
        on=["item_id", "date"],
        suffix="_new"
    )
)

for column in [
    "day_of_week_sin",
    "day_of_week_cos",
    "month_sin",
    "month_cos"
]:
    mismatch_count = calendar_comparison.filter(
        (pl.col(column) - pl.col(f"{column}_new")).abs() > 1e-8
    ).height

    print(f"{column}: {mismatch_count} mismatches")


day_of_week_sin: 0 mismatches
day_of_week_cos: 0 mismatches
month_sin: 0 mismatches
month_cos: 0 mismatches


In [22]:

# CELL 20: Replace the old zero-sales feature with the verified calculation
# Purpose: Update zero_sales_run using the debug calculation
# that matches the original CA_1 sequence.

ca1_recreated_sample = (
    debug_zero
    .with_columns([
        pl.when(pl.col("lag_28").is_null())
        .then(None)
        .otherwise(pl.col("_debug_zero_count"))
        .alias("zero_sales_run")
    ])
    .drop([
        "_debug_positive",
        "_debug_group",
        "_debug_zero",
        "_debug_zero_count"
    ])
)

print("Zero-sales run feature updated.")

print(
    ca1_recreated_sample
    .filter(
        (pl.col("item_id") == sample_items[0]) &
        pl.col("date").is_between(
            pd.Timestamp("2011-02-26"),
            pd.Timestamp("2011-03-10")
        )
    )
    .select(["date", "lag_28", "zero_sales_run"])
)


Zero-sales run feature updated.
shape: (13, 3)
┌────────────┬────────┬────────────────┐
│ date       ┆ lag_28 ┆ zero_sales_run │
│ ---        ┆ ---    ┆ ---            │
│ date       ┆ i64    ┆ i64            │
╞════════════╪════════╪════════════════╡
│ 2011-02-26 ┆ 0      ┆ 1              │
│ 2011-02-27 ┆ 0      ┆ 2              │
│ 2011-02-28 ┆ 0      ┆ 3              │
│ 2011-03-01 ┆ 0      ┆ 4              │
│ 2011-03-02 ┆ 0      ┆ 5              │
│ …          ┆ …      ┆ …              │
│ 2011-03-06 ┆ 0      ┆ 9              │
│ 2011-03-07 ┆ 0      ┆ 10             │
│ 2011-03-08 ┆ 0      ┆ 11             │
│ 2011-03-09 ┆ 0      ┆ 12             │
│ 2011-03-10 ┆ 0      ┆ 13             │
└────────────┴────────┴────────────────┘


In [23]:

# CELL 21: Final verification of all 21 engineered features
# Purpose: Compare the original CA_1 features with the corrected
# reusable feature calculations and count all mismatches.

comparison_columns = [
    "lag_28", "lag_35", "lag_42", "lag_56", "lag_364",
    "rolling_mean_7", "rolling_mean_28", "rolling_mean_56",
    "rolling_std_7", "rolling_std_28",
    "day_of_week_sin", "day_of_week_cos",
    "month_sin", "month_cos", "ema_28",
    "zero_sales_run", "price_lag_28", "price_lag_56",
    "price_momentum", "historical_max_price_56",
    "discount_ratio"
]

# Join original and corrected features using product and date
comparison = (
    ca1_original_sample
    .select(["item_id", "date"] + comparison_columns)
    .join(
        ca1_recreated_sample.select(
            ["item_id", "date"] + comparison_columns
        ),
        on=["item_id", "date"],
        suffix="_new"
    )
)

# Compare every feature, including null-value consistency
mismatch_results = []

print("FINAL FEATURE COMPARISON")
print("-" * 60)

for column in comparison_columns:
    original = pl.col(column)
    recreated = pl.col(f"{column}_new")

    mismatch_count = comparison.filter(
        ~(
            (original.is_null() & recreated.is_null()) |
            (
                original.is_not_null() &
                recreated.is_not_null() &
                ((original - recreated).abs() <= 1e-8)
            )
        )
    ).height

    mismatch_results.append((column, mismatch_count))

    status = "PASS" if mismatch_count == 0 else "MISMATCH"
    print(f"{column}: {mismatch_count} | {status}")

# Calculate final verification summary
total_mismatches = sum(count for _, count in mismatch_results)

print("\n" + "=" * 60)
print("FINAL VERIFICATION SUMMARY")
print("=" * 60)

print("Features compared:", len(comparison_columns))
print("Rows compared:", comparison.height)
print("Total mismatches:", total_mismatches)

if total_mismatches == 0:
    print("\nSUCCESS: ALL FEATURES MATCH THE ORIGINAL CA_1 DATASET!")
else:
    print("\nWARNING: FEATURE CALCULATIONS STILL NEED CORRECTION.")


FINAL FEATURE COMPARISON
------------------------------------------------------------
lag_28: 0 | PASS
lag_35: 0 | PASS
lag_42: 0 | PASS
lag_56: 0 | PASS
lag_364: 0 | PASS
rolling_mean_7: 0 | PASS
rolling_mean_28: 0 | PASS
rolling_mean_56: 0 | PASS
rolling_std_7: 0 | PASS
rolling_std_28: 0 | PASS
day_of_week_sin: 0 | PASS
day_of_week_cos: 0 | PASS
month_sin: 0 | PASS
month_cos: 0 | PASS
ema_28: 0 | PASS
zero_sales_run: 0 | PASS
price_lag_28: 0 | PASS
price_lag_56: 0 | PASS
price_momentum: 0 | PASS
historical_max_price_56: 0 | PASS
discount_ratio: 0 | PASS

FINAL VERIFICATION SUMMARY
Features compared: 21
Rows compared: 1913
Total mismatches: 0

SUCCESS: ALL FEATURES MATCH THE ORIGINAL CA_1 DATASET!


In [24]:

# CELL 22: Final reusable advanced feature engineering function
# Purpose: Generate the same 30-column feature dataset for any
# M5 store, using the verified CA_1 feature calculations.

def create_advanced_features(store_panel):

    # Sort by product and date
    df = store_panel.sort(["item_id", "date"])

    # 1. Historical sales lags
    df = df.with_columns([
        pl.col("sales").shift(28).over("item_id").alias("lag_28"),
        pl.col("sales").shift(35).over("item_id").alias("lag_35"),
        pl.col("sales").shift(42).over("item_id").alias("lag_42"),
        pl.col("sales").shift(56).over("item_id").alias("lag_56"),
        pl.col("sales").shift(364).over("item_id").alias("lag_364"),
        pl.col("sales").shift(28).over("item_id").alias("_sales_gap28")
    ])

    # 2. Calendar features
    df = df.with_columns([
        pl.col("date").dt.weekday().alias("day_of_week"),
        pl.col("date").dt.month().alias("month")
    ])

    # 3. Rolling statistics using historical sales
    df = df.with_columns([
        pl.col("_sales_gap28").rolling_mean(7).over("item_id").alias("rolling_mean_7"),
        pl.col("_sales_gap28").rolling_mean(28).over("item_id").alias("rolling_mean_28"),
        pl.col("_sales_gap28").rolling_mean(56).over("item_id").alias("rolling_mean_56"),
        pl.col("_sales_gap28").rolling_std(7).over("item_id").alias("rolling_std_7"),
        pl.col("_sales_gap28").rolling_std(28).over("item_id").alias("rolling_std_28")
    ])

    # 4. Correct zero-based cyclical calendar encoding
    df = df.with_columns([
        (2 * np.pi * (pl.col("day_of_week") - 1) / 7).sin().alias("day_of_week_sin"),
        (2 * np.pi * (pl.col("day_of_week") - 1) / 7).cos().alias("day_of_week_cos"),
        (2 * np.pi * (pl.col("month") - 1) / 12).sin().alias("month_sin"),
        (2 * np.pi * (pl.col("month") - 1) / 12).cos().alias("month_cos")
    ])

    # 5. Exponential moving average
    df = df.with_columns([
        pl.col("_sales_gap28")
        .ewm_mean(span=28, adjust=False)
        .over("item_id")
        .alias("ema_28")
    ])

    # 6. Correct zero-sales run-length calculation
    df = df.with_columns([
        (pl.col("_sales_gap28") > 0)
        .fill_null(False)
        .cast(pl.Int64)
        .alias("_positive_sales")
    ])

    df = df.with_columns([
        pl.col("_positive_sales")
        .cum_sum()
        .over("item_id")
        .alias("_zero_group")
    ])

    df = df.with_columns([
        (pl.col("_sales_gap28") == 0)
        .fill_null(False)
        .cast(pl.Int64)
        .alias("_zero_flag")
    ])

    df = df.with_columns([
        pl.col("_zero_flag")
        .cum_sum()
        .over(["item_id", "_zero_group"])
        .alias("_zero_count")
    ])

    df = df.with_columns([
        pl.when(pl.col("_sales_gap28").is_null())
        .then(None)
        .otherwise(pl.col("_zero_count"))
        .alias("zero_sales_run")
    ])

    # 7. Historical price features
    df = df.with_columns([
        pl.col("sell_price").shift(28).over("item_id").alias("price_lag_28"),
        pl.col("sell_price").shift(56).over("item_id").alias("price_lag_56"),
        pl.col("sell_price")
        .shift(28)
        .rolling_max(window_size=56)
        .over("item_id")
        .alias("historical_max_price_56")
    ])

    df = df.with_columns([
        pl.when(pl.col("price_lag_56") > 0)
        .then(pl.col("price_lag_28") / pl.col("price_lag_56") - 1)
        .otherwise(None)
        .alias("price_momentum"),

        pl.when(pl.col("historical_max_price_56") > 0)
        .then(pl.col("price_lag_28") / pl.col("historical_max_price_56"))
        .otherwise(None)
        .alias("discount_ratio")
    ])

    # 8. Remove temporary columns
    df = df.drop([
        "_sales_gap28",
        "_positive_sales",
        "_zero_group",
        "_zero_flag",
        "_zero_count"
    ])

    return df

print("Reusable feature engineering function updated successfully.")


Reusable feature engineering function updated successfully.


In [25]:

# CELL 23: Regenerate CA_2 using the corrected reusable function
# Purpose: Recalculate all advanced features with the corrected
# calendar encoding and zero-sales run-length logic.

# Generate the corrected CA_2 feature dataset
ca2_features = create_advanced_features(ca2_panel)

# Verify the generated dataset
print("CORRECTED CA_2 FEATURE GENERATION")
print("-" * 55)

print("Shape:", ca2_features.shape)
print("Store:", ca2_features["store_id"].unique().to_list())
print("Products:", ca2_features["item_id"].n_unique())

# Save the corrected dataset, replacing the earlier version
CA2_FEATURE_PATH = PROCESSED_DIR / "ca2_advanced_features.parquet"

ca2_features.write_parquet(CA2_FEATURE_PATH)

print("\nCorrected dataset saved:", CA2_FEATURE_PATH)
print(
    "File size (MB):",
    round(CA2_FEATURE_PATH.stat().st_size / (1024**2), 2)
)

print("\nCA_2 REGENERATION COMPLETED")


CORRECTED CA_2 FEATURE GENERATION
-------------------------------------------------------
Shape: (5832737, 30)
Store: ['CA_2']
Products: 3049

Corrected dataset saved: D:\Retail-Demand\data\processed\ca2_advanced_features.parquet
File size (MB): 64.65

CA_2 REGENERATION COMPLETED


In [26]:

# CELL 24: Validate the regenerated CA_2 feature dataset
# Purpose: Check the corrected feature calculations and data quality.

print("CORRECTED CA_2 VALIDATION")
print("-" * 55)

# Check dimensions and duplicate records
print("Shape:", ca2_features.shape)

duplicates = (
    ca2_features
    .group_by(["item_id", "store_id", "date"])
    .len()
    .filter(pl.col("len") > 1)
    .height
)

print("Duplicate records:", duplicates)

# Check invalid values
checks = {
    "Negative sales": pl.col("sales") < 0,
    "Negative EMA": pl.col("ema_28") < 0,
    "Negative rolling mean": pl.col("rolling_mean_7") < 0,
    "Negative rolling std": pl.col("rolling_std_7") < 0,
    "Invalid price momentum": pl.col("price_momentum") < -1,
    "Invalid discount ratio": pl.col("discount_ratio") > 1,
    "Negative zero-sales run": pl.col("zero_sales_run") < 0
}

for name, condition in checks.items():
    count = ca2_features.filter(condition).height
    print(f"{name}: {count}")

# Verify the saved file
print("\nSaved file exists:", CA2_FEATURE_PATH.exists())

print("\nCORRECTED CA_2 VALIDATION COMPLETED")


CORRECTED CA_2 VALIDATION
-------------------------------------------------------
Shape: (5832737, 30)
Duplicate records: 0
Negative sales: 0
Negative EMA: 0
Negative rolling mean: 0
Negative rolling std: 0
Invalid price momentum: 0
Invalid discount ratio: 0
Negative zero-sales run: 0

Saved file exists: True

CORRECTED CA_2 VALIDATION COMPLETED


In [27]:

# CELL 25: Generate advanced features for CA_3
# Purpose: Apply the verified reusable feature function
# to the third store and save its processed dataset.

STORE_TO_PROCESS = "CA_3"

# Load only the selected store
store_panel = (
    pl.scan_parquet(PANEL_PATH)
    .filter(pl.col("store_id") == STORE_TO_PROCESS)
    .sort(["item_id", "date"])
    .collect()
)

print("Loaded store:", STORE_TO_PROCESS)
print("Panel shape:", store_panel.shape)

# Generate advanced features
store_features = create_advanced_features(store_panel)

print("\nGenerated feature shape:", store_features.shape)

# Save the advanced feature dataset
STORE_FEATURE_PATH = (
    PROCESSED_DIR / f"{STORE_TO_PROCESS.lower()}_advanced_features.parquet"
)

store_features.write_parquet(STORE_FEATURE_PATH)

print("\nSaved:", STORE_FEATURE_PATH)
print(
    "File size (MB):",
    round(STORE_FEATURE_PATH.stat().st_size / (1024**2), 2)
)

print("\nCA_3 FEATURE GENERATION COMPLETED")


Loaded store: CA_3
Panel shape: (5832737, 7)

Generated feature shape: (5832737, 30)

Saved: D:\Retail-Demand\data\processed\ca_3_advanced_features.parquet
File size (MB): 76.32

CA_3 FEATURE GENERATION COMPLETED


In [28]:
# ============================================================
# CELL 26: VALIDATE CA_3 ADVANCED FEATURES
# Purpose:
# Check row counts, duplicate records, invalid values,
# and whether the generated feature file was saved correctly.
# ============================================================

CA3_FEATURE_PATH = PROCESSED_DIR / "ca_3_advanced_features.parquet"

# Load the generated feature dataset
ca3_check = pl.read_parquet(CA3_FEATURE_PATH)

# Basic dataset information
print("CA_3 FEATURE VALIDATION")
print("=" * 50)

print("Shape:", ca3_check.shape)
print("Stores:", ca3_check["store_id"].unique().to_list())
print("Unique products:", ca3_check["item_id"].n_unique())
print("Date range:", ca3_check["date"].min(), "to", ca3_check["date"].max())

# Check duplicate item-date records
duplicate_count = (
    ca3_check
    .group_by(["item_id", "date"])
    .len()
    .filter(pl.col("len") > 1)
    .height
)

print("\nDuplicate item-date records:", duplicate_count)

# Check negative values in sales and engineered features
negative_checks = {}

for column in [
    "sales",
    "ema_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "rolling_mean_56",
    "rolling_std_7",
    "rolling_std_28",
    "zero_sales_run"
]:
    if column in ca3_check.columns:
        negative_checks[column] = (
            ca3_check
            .filter(pl.col(column) < 0)
            .height
        )

print("\nNegative value checks:")
for column, count in negative_checks.items():
    print(f"{column}: {count}")

# Check invalid price ratios
price_checks = {}

for column in ["price_momentum", "discount_ratio"]:
    if column in ca3_check.columns:
        price_checks[column] = (
            ca3_check
            .filter(
                pl.col(column).is_not_null()
                & ~pl.col(column).is_finite()
            )
            .height
        )

print("\nInvalid price feature values:")
for column, count in price_checks.items():
    print(f"{column}: {count}")

print("\nFile exists:", CA3_FEATURE_PATH.exists())
print("CA_3 VALIDATION COMPLETED")

CA_3 FEATURE VALIDATION
Shape: (5832737, 30)
Stores: ['CA_3']
Unique products: 3049
Date range: 2011-01-29 to 2016-04-24

Duplicate item-date records: 0

Negative value checks:
sales: 0
ema_28: 0
rolling_mean_7: 0
rolling_mean_28: 0
rolling_mean_56: 0
rolling_std_7: 0
rolling_std_28: 0
zero_sales_run: 0

Invalid price feature values:
price_momentum: 0
discount_ratio: 0

File exists: True
CA_3 VALIDATION COMPLETED


In [29]:
# ============================================================
# CELL 27: GENERATE ADVANCED FEATURES FOR CA_4
# Purpose:
# Load CA_4 from the main panel, apply the corrected reusable
# feature engineering function, and save the output.
# ============================================================

import gc

# Release the previous validation dataframe
del ca3_check
gc.collect()

STORE_TO_PROCESS = "CA_4"

# Load only CA_4 from the main panel
store_panel = (
    pl.scan_parquet(PANEL_PATH)
    .filter(pl.col("store_id") == STORE_TO_PROCESS)
    .sort(["item_id", "date"])
    .collect()
)

print("Loaded store:", STORE_TO_PROCESS)
print("Panel shape:", store_panel.shape)

# Generate advanced features
store_features = create_advanced_features(store_panel)

print("\nGenerated feature shape:", store_features.shape)

# Save the feature dataset
STORE_FEATURE_PATH = (
    PROCESSED_DIR / f"{STORE_TO_PROCESS.lower()}_advanced_features.parquet"
)

store_features.write_parquet(STORE_FEATURE_PATH)

print("\nSaved:", STORE_FEATURE_PATH)
print(
    "File size (MB):",
    round(STORE_FEATURE_PATH.stat().st_size / (1024 ** 2), 2)
)

print(f"\n{STORE_TO_PROCESS} FEATURE GENERATION COMPLETED")

Loaded store: CA_4
Panel shape: (5832737, 7)

Generated feature shape: (5832737, 30)

Saved: D:\Retail-Demand\data\processed\ca_4_advanced_features.parquet
File size (MB): 64.0

CA_4 FEATURE GENERATION COMPLETED


In [30]:
# ============================================================
# CELL 28: VALIDATE CA_4 ADVANCED FEATURES
# Purpose:
# Verify dataset integrity and check for invalid feature values.
# ============================================================

CA4_FEATURE_PATH = PROCESSED_DIR / "ca_4_advanced_features.parquet"

ca4_check = pl.read_parquet(CA4_FEATURE_PATH)

print("CA_4 FEATURE VALIDATION")
print("=" * 50)

print("Shape:", ca4_check.shape)
print("Stores:", ca4_check["store_id"].unique().to_list())
print("Unique products:", ca4_check["item_id"].n_unique())
print("Date range:", ca4_check["date"].min(), "to", ca4_check["date"].max())

# Check duplicate item-date records
duplicate_count = (
    ca4_check
    .group_by(["item_id", "date"])
    .len()
    .filter(pl.col("len") > 1)
    .height
)

print("\nDuplicate item-date records:", duplicate_count)

# Check negative values
for column in [
    "sales", "ema_28", "rolling_mean_7",
    "rolling_mean_28", "rolling_mean_56",
    "rolling_std_7", "rolling_std_28", "zero_sales_run"
]:
    if column in ca4_check.columns:
        count = ca4_check.filter(pl.col(column) < 0).height
        print(f"Negative {column}: {count}")

# Check invalid price features
for column in ["price_momentum", "discount_ratio"]:
    count = ca4_check.filter(
        pl.col(column).is_not_null() & ~pl.col(column).is_finite()
    ).height
    print(f"Invalid {column}: {count}")

print("\nFile exists:", CA4_FEATURE_PATH.exists())
print("CA_4 VALIDATION COMPLETED")

CA_4 FEATURE VALIDATION
Shape: (5832737, 30)
Stores: ['CA_4']
Unique products: 3049
Date range: 2011-01-29 to 2016-04-24

Duplicate item-date records: 0
Negative sales: 0
Negative ema_28: 0
Negative rolling_mean_7: 0
Negative rolling_mean_28: 0
Negative rolling_mean_56: 0
Negative rolling_std_7: 0
Negative rolling_std_28: 0
Negative zero_sales_run: 0
Invalid price_momentum: 0
Invalid discount_ratio: 0

File exists: True
CA_4 VALIDATION COMPLETED


In [31]:
# ============================================================
# CELL 29: GENERATE ADVANCED FEATURES FOR CA_5
# Purpose:
# Generate the same 30 engineered features for CA_5 and
# save the resulting dataset as a Parquet file.
# ============================================================

import gc

# Release the previous store's validation dataframe
del ca4_check
gc.collect()

STORE_TO_PROCESS = "CA_5"

# Load CA_5 from the main panel
store_panel = (
    pl.scan_parquet(PANEL_PATH)
    .filter(pl.col("store_id") == STORE_TO_PROCESS)
    .sort(["item_id", "date"])
    .collect()
)

print("Loaded store:", STORE_TO_PROCESS)
print("Panel shape:", store_panel.shape)

# Apply the corrected feature engineering function
store_features = create_advanced_features(store_panel)

print("\nGenerated feature shape:", store_features.shape)

# Save the generated features
STORE_FEATURE_PATH = (
    PROCESSED_DIR / f"{STORE_TO_PROCESS.lower()}_advanced_features.parquet"
)

store_features.write_parquet(STORE_FEATURE_PATH)

print("\nSaved:", STORE_FEATURE_PATH)
print(
    "File size (MB):",
    round(STORE_FEATURE_PATH.stat().st_size / (1024 ** 2), 2)
)

print(f"\n{STORE_TO_PROCESS} FEATURE GENERATION COMPLETED")

Loaded store: CA_5
Panel shape: (0, 7)

Generated feature shape: (0, 30)

Saved: D:\Retail-Demand\data\processed\ca_5_advanced_features.parquet
File size (MB): 0.0

CA_5 FEATURE GENERATION COMPLETED


In [32]:
# ============================================================
# CELL 30: CHECK AVAILABLE STORES
# Purpose:
# List all stores in the original panel and remove the
# accidentally generated empty CA_5 feature file.
# ============================================================

# Get all store IDs from the original panel
available_stores = (
    pl.scan_parquet(PANEL_PATH)
    .select("store_id")
    .unique()
    .sort("store_id")
    .collect()
)

print("AVAILABLE STORES:")
print(available_stores)

print("\nTotal stores:", available_stores.height)

# Remove the empty CA_5 file if it exists
EMPTY_FILE = PROCESSED_DIR / "ca_5_advanced_features.parquet"

if EMPTY_FILE.exists():
    EMPTY_FILE.unlink()
    print("\nRemoved empty file:", EMPTY_FILE.name)

print("\nSTORE CHECK COMPLETED")

AVAILABLE STORES:
shape: (10, 1)
┌──────────┐
│ store_id │
│ ---      │
│ str      │
╞══════════╡
│ CA_1     │
│ CA_2     │
│ CA_3     │
│ CA_4     │
│ TX_1     │
│ TX_2     │
│ TX_3     │
│ WI_1     │
│ WI_2     │
│ WI_3     │
└──────────┘

Total stores: 10

Removed empty file: ca_5_advanced_features.parquet

STORE CHECK COMPLETED


In [33]:
# ============================================================
# CELL 31: GENERATE ADVANCED FEATURES FOR TX_1
# Purpose:
# Load TX_1, apply the corrected feature engineering function,
# and save the generated 30-column feature dataset.
# ============================================================

import gc

# Release previous validation dataframe if it exists
if "ca4_check" in globals():
    del ca4_check

gc.collect()

STORE_TO_PROCESS = "TX_1"

# Load only TX_1 from the main panel
store_panel = (
    pl.scan_parquet(PANEL_PATH)
    .filter(pl.col("store_id") == STORE_TO_PROCESS)
    .sort(["item_id", "date"])
    .collect()
)

print("Loaded store:", STORE_TO_PROCESS)
print("Panel shape:", store_panel.shape)

# Generate advanced features
store_features = create_advanced_features(store_panel)

print("\nGenerated feature shape:", store_features.shape)

# Save generated features
STORE_FEATURE_PATH = (
    PROCESSED_DIR / f"{STORE_TO_PROCESS.lower()}_advanced_features.parquet"
)

store_features.write_parquet(STORE_FEATURE_PATH)

print("\nSaved:", STORE_FEATURE_PATH)
print(
    "File size (MB):",
    round(STORE_FEATURE_PATH.stat().st_size / (1024 ** 2), 2)
)

print(f"\n{STORE_TO_PROCESS} FEATURE GENERATION COMPLETED")

Loaded store: TX_1
Panel shape: (5832737, 7)

Generated feature shape: (5832737, 30)

Saved: D:\Retail-Demand\data\processed\tx_1_advanced_features.parquet
File size (MB): 68.17

TX_1 FEATURE GENERATION COMPLETED


In [34]:
# ============================================================
# CELL 32: VALIDATE TX_1 ADVANCED FEATURES
# Purpose:
# Check data integrity, duplicate records, negative values,
# and invalid engineered price features.
# ============================================================

TX1_FEATURE_PATH = PROCESSED_DIR / "tx_1_advanced_features.parquet"

tx1_check = pl.read_parquet(TX1_FEATURE_PATH)

print("TX_1 FEATURE VALIDATION")
print("=" * 50)

print("Shape:", tx1_check.shape)
print("Stores:", tx1_check["store_id"].unique().to_list())
print("Unique products:", tx1_check["item_id"].n_unique())
print("Date range:", tx1_check["date"].min(), "to", tx1_check["date"].max())

# Check duplicate item-date records
duplicate_count = (
    tx1_check
    .group_by(["item_id", "date"])
    .len()
    .filter(pl.col("len") > 1)
    .height
)

print("\nDuplicate item-date records:", duplicate_count)

# Check negative values
for column in [
    "sales", "ema_28", "rolling_mean_7",
    "rolling_mean_28", "rolling_mean_56",
    "rolling_std_7", "rolling_std_28", "zero_sales_run"
]:
    count = tx1_check.filter(pl.col(column) < 0).height
    print(f"Negative {column}: {count}")

# Check invalid price features
for column in ["price_momentum", "discount_ratio"]:
    count = tx1_check.filter(
        pl.col(column).is_not_null() & ~pl.col(column).is_finite()
    ).height
    print(f"Invalid {column}: {count}")

print("\nFile exists:", TX1_FEATURE_PATH.exists())
print("TX_1 VALIDATION COMPLETED")

TX_1 FEATURE VALIDATION
Shape: (5832737, 30)
Stores: ['TX_1']
Unique products: 3049
Date range: 2011-01-29 to 2016-04-24

Duplicate item-date records: 0
Negative sales: 0
Negative ema_28: 0
Negative rolling_mean_7: 0
Negative rolling_mean_28: 0
Negative rolling_mean_56: 0
Negative rolling_std_7: 0
Negative rolling_std_28: 0
Negative zero_sales_run: 0
Invalid price_momentum: 0
Invalid discount_ratio: 0

File exists: True
TX_1 VALIDATION COMPLETED


In [35]:
# ============================================================
# CELL 33: GENERATE ADVANCED FEATURES FOR TX_2
# Purpose:
# Apply the corrected feature engineering function to TX_2
# and save its 30-column feature dataset.
# ============================================================

import gc

# Release the previous validation dataframe
del tx1_check
gc.collect()

STORE_TO_PROCESS = "TX_2"

# Load only TX_2 from the main panel
store_panel = (
    pl.scan_parquet(PANEL_PATH)
    .filter(pl.col("store_id") == STORE_TO_PROCESS)
    .sort(["item_id", "date"])
    .collect()
)

print("Loaded store:", STORE_TO_PROCESS)
print("Panel shape:", store_panel.shape)

# Generate advanced features
store_features = create_advanced_features(store_panel)

print("\nGenerated feature shape:", store_features.shape)

# Save the feature dataset
STORE_FEATURE_PATH = (
    PROCESSED_DIR / f"{STORE_TO_PROCESS.lower()}_advanced_features.parquet"
)

store_features.write_parquet(STORE_FEATURE_PATH)

print("\nSaved:", STORE_FEATURE_PATH)
print(
    "File size (MB):",
    round(STORE_FEATURE_PATH.stat().st_size / (1024 ** 2), 2)
)

print(f"\n{STORE_TO_PROCESS} FEATURE GENERATION COMPLETED")

Loaded store: TX_2
Panel shape: (5832737, 7)

Generated feature shape: (5832737, 30)

Saved: D:\Retail-Demand\data\processed\tx_2_advanced_features.parquet
File size (MB): 71.51

TX_2 FEATURE GENERATION COMPLETED


In [36]:
# ============================================================
# CELL 34: VALIDATE TX_2 ADVANCED FEATURES
# Purpose:
# Verify data integrity and check for invalid feature values.
# ============================================================

TX2_FEATURE_PATH = PROCESSED_DIR / "tx_2_advanced_features.parquet"

tx2_check = pl.read_parquet(TX2_FEATURE_PATH)

print("TX_2 FEATURE VALIDATION")
print("=" * 50)

print("Shape:", tx2_check.shape)
print("Stores:", tx2_check["store_id"].unique().to_list())
print("Unique products:", tx2_check["item_id"].n_unique())
print("Date range:", tx2_check["date"].min(), "to", tx2_check["date"].max())

# Check duplicate item-date records
duplicate_count = (
    tx2_check
    .group_by(["item_id", "date"])
    .len()
    .filter(pl.col("len") > 1)
    .height
)

print("\nDuplicate item-date records:", duplicate_count)

# Check negative values
for column in [
    "sales", "ema_28", "rolling_mean_7",
    "rolling_mean_28", "rolling_mean_56",
    "rolling_std_7", "rolling_std_28", "zero_sales_run"
]:
    count = tx2_check.filter(pl.col(column) < 0).height
    print(f"Negative {column}: {count}")

# Check invalid price features
for column in ["price_momentum", "discount_ratio"]:
    count = tx2_check.filter(
        pl.col(column).is_not_null() & ~pl.col(column).is_finite()
    ).height
    print(f"Invalid {column}: {count}")

print("\nFile exists:", TX2_FEATURE_PATH.exists())
print("TX_2 VALIDATION COMPLETED")

TX_2 FEATURE VALIDATION
Shape: (5832737, 30)
Stores: ['TX_2']
Unique products: 3049
Date range: 2011-01-29 to 2016-04-24

Duplicate item-date records: 0
Negative sales: 0
Negative ema_28: 0
Negative rolling_mean_7: 0
Negative rolling_mean_28: 0
Negative rolling_mean_56: 0
Negative rolling_std_7: 0
Negative rolling_std_28: 0
Negative zero_sales_run: 0
Invalid price_momentum: 0
Invalid discount_ratio: 0

File exists: True
TX_2 VALIDATION COMPLETED


In [37]:
# ============================================================
# CELL 35: GENERATE ADVANCED FEATURES FOR TX_3
# Purpose:
# Generate the 30 engineered features for TX_3 and save them.
# ============================================================

import gc

# Release previous validation dataframe
del tx2_check
gc.collect()

STORE_TO_PROCESS = "TX_3"

# Load TX_3 from the main panel
store_panel = (
    pl.scan_parquet(PANEL_PATH)
    .filter(pl.col("store_id") == STORE_TO_PROCESS)
    .sort(["item_id", "date"])
    .collect()
)

print("Loaded store:", STORE_TO_PROCESS)
print("Panel shape:", store_panel.shape)

# Generate advanced features
store_features = create_advanced_features(store_panel)

print("\nGenerated feature shape:", store_features.shape)

# Save the feature dataset
STORE_FEATURE_PATH = (
    PROCESSED_DIR / f"{STORE_TO_PROCESS.lower()}_advanced_features.parquet"
)

store_features.write_parquet(STORE_FEATURE_PATH)

print("\nSaved:", STORE_FEATURE_PATH)
print(
    "File size (MB):",
    round(STORE_FEATURE_PATH.stat().st_size / (1024 ** 2), 2)
)

print(f"\n{STORE_TO_PROCESS} FEATURE GENERATION COMPLETED")

Loaded store: TX_3
Panel shape: (5832737, 7)

Generated feature shape: (5832737, 30)

Saved: D:\Retail-Demand\data\processed\tx_3_advanced_features.parquet
File size (MB): 67.93

TX_3 FEATURE GENERATION COMPLETED


In [38]:
# ============================================================
# CELL 36: VALIDATE TX_3 ADVANCED FEATURES
# Purpose:
# Check data integrity, duplicate records, negative values,
# and invalid price features.
# ============================================================

TX3_FEATURE_PATH = PROCESSED_DIR / "tx_3_advanced_features.parquet"

tx3_check = pl.read_parquet(TX3_FEATURE_PATH)

print("TX_3 FEATURE VALIDATION")
print("=" * 50)

print("Shape:", tx3_check.shape)
print("Stores:", tx3_check["store_id"].unique().to_list())
print("Unique products:", tx3_check["item_id"].n_unique())
print("Date range:", tx3_check["date"].min(), "to", tx3_check["date"].max())

# Check duplicate item-date records
duplicate_count = (
    tx3_check
    .group_by(["item_id", "date"])
    .len()
    .filter(pl.col("len") > 1)
    .height
)

print("\nDuplicate item-date records:", duplicate_count)

# Check negative values
for column in [
    "sales", "ema_28", "rolling_mean_7",
    "rolling_mean_28", "rolling_mean_56",
    "rolling_std_7", "rolling_std_28", "zero_sales_run"
]:
    count = tx3_check.filter(pl.col(column) < 0).height
    print(f"Negative {column}: {count}")

# Check invalid price features
for column in ["price_momentum", "discount_ratio"]:
    count = tx3_check.filter(
        pl.col(column).is_not_null() & ~pl.col(column).is_finite()
    ).height
    print(f"Invalid {column}: {count}")

print("\nFile exists:", TX3_FEATURE_PATH.exists())
print("TX_3 VALIDATION COMPLETED")

TX_3 FEATURE VALIDATION
Shape: (5832737, 30)
Stores: ['TX_3']
Unique products: 3049
Date range: 2011-01-29 to 2016-04-24

Duplicate item-date records: 0
Negative sales: 0
Negative ema_28: 0
Negative rolling_mean_7: 0
Negative rolling_mean_28: 0
Negative rolling_mean_56: 0
Negative rolling_std_7: 0
Negative rolling_std_28: 0
Negative zero_sales_run: 0
Invalid price_momentum: 0
Invalid discount_ratio: 0

File exists: True
TX_3 VALIDATION COMPLETED


In [39]:
# ============================================================
# CELL 37: GENERATE ADVANCED FEATURES FOR WI_1
# Purpose:
# Generate the same 30 engineered features for WI_1
# and save the resulting dataset.
# ============================================================

import gc

# Release previous validation dataframe
del tx3_check
gc.collect()

STORE_TO_PROCESS = "WI_1"

# Load WI_1 from the main panel
store_panel = (
    pl.scan_parquet(PANEL_PATH)
    .filter(pl.col("store_id") == STORE_TO_PROCESS)
    .sort(["item_id", "date"])
    .collect()
)

print("Loaded store:", STORE_TO_PROCESS)
print("Panel shape:", store_panel.shape)

# Generate advanced features
store_features = create_advanced_features(store_panel)

print("\nGenerated feature shape:", store_features.shape)

# Save the feature dataset
STORE_FEATURE_PATH = (
    PROCESSED_DIR / f"{STORE_TO_PROCESS.lower()}_advanced_features.parquet"
)

store_features.write_parquet(STORE_FEATURE_PATH)

print("\nSaved:", STORE_FEATURE_PATH)
print(
    "File size (MB):",
    round(STORE_FEATURE_PATH.stat().st_size / (1024 ** 2), 2)
)

print(f"\n{STORE_TO_PROCESS} FEATURE GENERATION COMPLETED")

Loaded store: WI_1
Panel shape: (5832737, 7)

Generated feature shape: (5832737, 30)

Saved: D:\Retail-Demand\data\processed\wi_1_advanced_features.parquet
File size (MB): 65.8

WI_1 FEATURE GENERATION COMPLETED


In [40]:
# ============================================================
# CELL 38: VALIDATE WI_1 ADVANCED FEATURES
# Purpose:
# Verify dataset integrity and check for invalid values.
# ============================================================

WI1_FEATURE_PATH = PROCESSED_DIR / "wi_1_advanced_features.parquet"

wi1_check = pl.read_parquet(WI1_FEATURE_PATH)

print("WI_1 FEATURE VALIDATION")
print("=" * 50)

print("Shape:", wi1_check.shape)
print("Stores:", wi1_check["store_id"].unique().to_list())
print("Unique products:", wi1_check["item_id"].n_unique())
print("Date range:", wi1_check["date"].min(), "to", wi1_check["date"].max())

# Check duplicate item-date records
duplicate_count = (
    wi1_check
    .group_by(["item_id", "date"])
    .len()
    .filter(pl.col("len") > 1)
    .height
)

print("\nDuplicate item-date records:", duplicate_count)

# Check negative values
for column in [
    "sales", "ema_28", "rolling_mean_7",
    "rolling_mean_28", "rolling_mean_56",
    "rolling_std_7", "rolling_std_28", "zero_sales_run"
]:
    count = wi1_check.filter(pl.col(column) < 0).height
    print(f"Negative {column}: {count}")

# Check invalid price features
for column in ["price_momentum", "discount_ratio"]:
    count = wi1_check.filter(
        pl.col(column).is_not_null() & ~pl.col(column).is_finite()
    ).height
    print(f"Invalid {column}: {count}")

print("\nFile exists:", WI1_FEATURE_PATH.exists())
print("WI_1 VALIDATION COMPLETED")

WI_1 FEATURE VALIDATION
Shape: (5832737, 30)
Stores: ['WI_1']
Unique products: 3049
Date range: 2011-01-29 to 2016-04-24

Duplicate item-date records: 0
Negative sales: 0
Negative ema_28: 0
Negative rolling_mean_7: 0
Negative rolling_mean_28: 0
Negative rolling_mean_56: 0
Negative rolling_std_7: 0
Negative rolling_std_28: 0
Negative zero_sales_run: 0
Invalid price_momentum: 0
Invalid discount_ratio: 0

File exists: True
WI_1 VALIDATION COMPLETED


In [41]:
# ============================================================
# CELL 39: GENERATE ADVANCED FEATURES FOR WI_2
# Purpose:
# Generate the same 30 engineered features for WI_2
# and save the resulting dataset.
# ============================================================

import gc

# Release previous validation dataframe
del wi1_check
gc.collect()

STORE_TO_PROCESS = "WI_2"

# Load only WI_2 from the main panel
store_panel = (
    pl.scan_parquet(PANEL_PATH)
    .filter(pl.col("store_id") == STORE_TO_PROCESS)
    .sort(["item_id", "date"])
    .collect()
)

print("Loaded store:", STORE_TO_PROCESS)
print("Panel shape:", store_panel.shape)

# Generate advanced features
store_features = create_advanced_features(store_panel)

print("\nGenerated feature shape:", store_features.shape)

# Save the feature dataset
STORE_FEATURE_PATH = (
    PROCESSED_DIR / f"{STORE_TO_PROCESS.lower()}_advanced_features.parquet"
)

store_features.write_parquet(STORE_FEATURE_PATH)

print("\nSaved:", STORE_FEATURE_PATH)
print(
    "File size (MB):",
    round(STORE_FEATURE_PATH.stat().st_size / (1024 ** 2), 2)
)

print(f"\n{STORE_TO_PROCESS} FEATURE GENERATION COMPLETED")

Loaded store: WI_2
Panel shape: (5832737, 7)

Generated feature shape: (5832737, 30)

Saved: D:\Retail-Demand\data\processed\wi_2_advanced_features.parquet
File size (MB): 66.94

WI_2 FEATURE GENERATION COMPLETED


In [42]:
# ============================================================
# CELL 40: VALIDATE WI_2 ADVANCED FEATURES
# Purpose:
# Verify dataset integrity and check for invalid values.
# ============================================================

WI2_FEATURE_PATH = PROCESSED_DIR / "wi_2_advanced_features.parquet"

wi2_check = pl.read_parquet(WI2_FEATURE_PATH)

print("WI_2 FEATURE VALIDATION")
print("=" * 50)

print("Shape:", wi2_check.shape)
print("Stores:", wi2_check["store_id"].unique().to_list())
print("Unique products:", wi2_check["item_id"].n_unique())
print("Date range:", wi2_check["date"].min(), "to", wi2_check["date"].max())

# Check duplicate item-date records
duplicate_count = (
    wi2_check
    .group_by(["item_id", "date"])
    .len()
    .filter(pl.col("len") > 1)
    .height
)

print("\nDuplicate item-date records:", duplicate_count)

# Check negative values
for column in [
    "sales", "ema_28", "rolling_mean_7",
    "rolling_mean_28", "rolling_mean_56",
    "rolling_std_7", "rolling_std_28", "zero_sales_run"
]:
    count = wi2_check.filter(pl.col(column) < 0).height
    print(f"Negative {column}: {count}")

# Check invalid price features
for column in ["price_momentum", "discount_ratio"]:
    count = wi2_check.filter(
        pl.col(column).is_not_null() & ~pl.col(column).is_finite()
    ).height
    print(f"Invalid {column}: {count}")

print("\nFile exists:", WI2_FEATURE_PATH.exists())
print("WI_2 VALIDATION COMPLETED")

WI_2 FEATURE VALIDATION
Shape: (5832737, 30)
Stores: ['WI_2']
Unique products: 3049
Date range: 2011-01-29 to 2016-04-24

Duplicate item-date records: 0
Negative sales: 0
Negative ema_28: 0
Negative rolling_mean_7: 0
Negative rolling_mean_28: 0
Negative rolling_mean_56: 0
Negative rolling_std_7: 0
Negative rolling_std_28: 0
Negative zero_sales_run: 0
Invalid price_momentum: 0
Invalid discount_ratio: 0

File exists: True
WI_2 VALIDATION COMPLETED


In [43]:
# ============================================================
# CELL 41: GENERATE ADVANCED FEATURES FOR WI_3
# Purpose:
# Generate all 30 engineered features for WI_3 and save them.
# ============================================================

import gc

# Release previous validation dataframe
del wi2_check
gc.collect()

STORE_TO_PROCESS = "WI_3"

# Load only WI_3 from the main panel
store_panel = (
    pl.scan_parquet(PANEL_PATH)
    .filter(pl.col("store_id") == STORE_TO_PROCESS)
    .sort(["item_id", "date"])
    .collect()
)

print("Loaded store:", STORE_TO_PROCESS)
print("Panel shape:", store_panel.shape)

# Generate advanced features
store_features = create_advanced_features(store_panel)

print("\nGenerated feature shape:", store_features.shape)

# Save the feature dataset
STORE_FEATURE_PATH = (
    PROCESSED_DIR / f"{STORE_TO_PROCESS.lower()}_advanced_features.parquet"
)

store_features.write_parquet(STORE_FEATURE_PATH)

print("\nSaved:", STORE_FEATURE_PATH)
print(
    "File size (MB):",
    round(STORE_FEATURE_PATH.stat().st_size / (1024 ** 2), 2)
)

print(f"\n{STORE_TO_PROCESS} FEATURE GENERATION COMPLETED")

Loaded store: WI_3
Panel shape: (5832737, 7)

Generated feature shape: (5832737, 30)

Saved: D:\Retail-Demand\data\processed\wi_3_advanced_features.parquet
File size (MB): 68.26

WI_3 FEATURE GENERATION COMPLETED


In [44]:
# ============================================================
# CELL 42: VALIDATE WI_3 ADVANCED FEATURES
# Purpose:
# Verify the final store's dataset integrity and feature values.
# ============================================================

WI3_FEATURE_PATH = PROCESSED_DIR / "wi_3_advanced_features.parquet"

wi3_check = pl.read_parquet(WI3_FEATURE_PATH)

print("WI_3 FEATURE VALIDATION")
print("=" * 50)

print("Shape:", wi3_check.shape)
print("Stores:", wi3_check["store_id"].unique().to_list())
print("Unique products:", wi3_check["item_id"].n_unique())
print("Date range:", wi3_check["date"].min(), "to", wi3_check["date"].max())

# Check duplicate item-date records
duplicate_count = (
    wi3_check
    .group_by(["item_id", "date"])
    .len()
    .filter(pl.col("len") > 1)
    .height
)

print("\nDuplicate item-date records:", duplicate_count)

# Check negative values
for column in [
    "sales", "ema_28", "rolling_mean_7",
    "rolling_mean_28", "rolling_mean_56",
    "rolling_std_7", "rolling_std_28", "zero_sales_run"
]:
    count = wi3_check.filter(pl.col(column) < 0).height
    print(f"Negative {column}: {count}")

# Check invalid price features
for column in ["price_momentum", "discount_ratio"]:
    count = wi3_check.filter(
        pl.col(column).is_not_null() & ~pl.col(column).is_finite()
    ).height
    print(f"Invalid {column}: {count}")

print("\nFile exists:", WI3_FEATURE_PATH.exists())
print("WI_3 VALIDATION COMPLETED")

WI_3 FEATURE VALIDATION
Shape: (5832737, 30)
Stores: ['WI_3']
Unique products: 3049
Date range: 2011-01-29 to 2016-04-24

Duplicate item-date records: 0
Negative sales: 0
Negative ema_28: 0
Negative rolling_mean_7: 0
Negative rolling_mean_28: 0
Negative rolling_mean_56: 0
Negative rolling_std_7: 0
Negative rolling_std_28: 0
Negative zero_sales_run: 0
Invalid price_momentum: 0
Invalid discount_ratio: 0

File exists: True
WI_3 VALIDATION COMPLETED


In [45]:
# ============================================================
# CELL 43: FINAL VERIFICATION OF ALL 10 STORES
# Purpose:
# Confirm all feature files exist, have the expected row and
# column counts, and contain the correct store identifier.
# ============================================================

import gc

# Release the previous validation dataframe
del wi3_check
gc.collect()

EXPECTED_STORES = [
    "CA_1", "CA_2", "CA_3", "CA_4",
    "TX_1", "TX_2", "TX_3",
    "WI_1", "WI_2", "WI_3"
]

EXPECTED_ROWS = 5_832_737
EXPECTED_COLUMNS = 30

verification_results = []

for store in EXPECTED_STORES:

    file_path = PROCESSED_DIR / f"{store.lower()}_advanced_features.parquet"

    if not file_path.exists():
        verification_results.append({
            "store": store,
            "rows": 0,
            "columns": 0,
            "actual_store": "MISSING",
            "status": "FAILED - FILE MISSING"
        })
        continue

    # Read Parquet schema without loading the entire dataset
    schema = pl.read_parquet_schema(file_path)
    column_count = len(schema)

    # Check row count and store identity using lazy scan
    summary = (
        pl.scan_parquet(file_path)
        .select(
            pl.len().alias("rows"),
            pl.col("store_id").n_unique().alias("unique_stores"),
            pl.col("store_id").first().alias("actual_store")
        )
        .collect()
    )

    row_count = summary["rows"][0]
    unique_stores = summary["unique_stores"][0]
    actual_store = summary["actual_store"][0]

    passed = (
        row_count == EXPECTED_ROWS
        and column_count == EXPECTED_COLUMNS
        and unique_stores == 1
        and actual_store == store
    )

    verification_results.append({
        "store": store,
        "rows": row_count,
        "columns": column_count,
        "actual_store": actual_store,
        "status": "PASSED" if passed else "FAILED"
    })

# Display verification report
verification_df = pl.DataFrame(verification_results)

print("FINAL MULTI-STORE FEATURE VERIFICATION")
print("=" * 70)
print(verification_df)

passed_count = verification_df.filter(
    pl.col("status") == "PASSED"
).height

failed_count = len(EXPECTED_STORES) - passed_count

print("\nTotal stores:", len(EXPECTED_STORES))
print("Passed:", passed_count)
print("Failed:", failed_count)

if failed_count == 0:
    print("\nALL 10 STORE FEATURE FILES VERIFIED SUCCESSFULLY")
else:
    print("\nSOME FILES FAILED VERIFICATION - CHECK THE REPORT")

FINAL MULTI-STORE FEATURE VERIFICATION
shape: (10, 5)
┌───────┬─────────┬─────────┬──────────────┬───────────────────────┐
│ store ┆ rows    ┆ columns ┆ actual_store ┆ status                │
│ ---   ┆ ---     ┆ ---     ┆ ---          ┆ ---                   │
│ str   ┆ i64     ┆ i64     ┆ str          ┆ str                   │
╞═══════╪═════════╪═════════╪══════════════╪═══════════════════════╡
│ CA_1  ┆ 0       ┆ 0       ┆ MISSING      ┆ FAILED - FILE MISSING │
│ CA_2  ┆ 0       ┆ 0       ┆ MISSING      ┆ FAILED - FILE MISSING │
│ CA_3  ┆ 5832737 ┆ 30      ┆ CA_3         ┆ PASSED                │
│ CA_4  ┆ 5832737 ┆ 30      ┆ CA_4         ┆ PASSED                │
│ TX_1  ┆ 5832737 ┆ 30      ┆ TX_1         ┆ PASSED                │
│ TX_2  ┆ 5832737 ┆ 30      ┆ TX_2         ┆ PASSED                │
│ TX_3  ┆ 5832737 ┆ 30      ┆ TX_3         ┆ PASSED                │
│ WI_1  ┆ 5832737 ┆ 30      ┆ WI_1         ┆ PASSED                │
│ WI_2  ┆ 5832737 ┆ 30      ┆ WI_2         ┆ PASS

In [46]:
# ============================================================
# CELL 44: INSPECT ALL ADVANCED FEATURE FILES
# Purpose:
# Identify the actual filenames in the processed directory
# and locate the CA_1 and CA_2 datasets.
# ============================================================

print("ADVANCED FEATURE FILES")
print("=" * 60)

feature_files = sorted(
    PROCESSED_DIR.glob("*advanced_features.parquet")
)

for file in feature_files:
    print(file.name, "|", round(file.stat().st_size / (1024 ** 2), 2), "MB")

print("\nTotal feature files found:", len(feature_files))

ADVANCED FEATURE FILES
ca1_advanced_features.parquet | 72.41 MB
ca2_advanced_features.parquet | 64.65 MB
ca_3_advanced_features.parquet | 76.32 MB
ca_4_advanced_features.parquet | 64.0 MB
tx_1_advanced_features.parquet | 68.17 MB
tx_2_advanced_features.parquet | 71.51 MB
tx_3_advanced_features.parquet | 67.93 MB
wi_1_advanced_features.parquet | 65.8 MB
wi_2_advanced_features.parquet | 66.94 MB
wi_3_advanced_features.parquet | 68.26 MB

Total feature files found: 10


In [47]:
# ============================================================
# CELL 45: CORRECTED FINAL VERIFICATION OF ALL 10 STORES
# Purpose:
# Verify all feature files using their actual filenames.
# Check row count, column count, and store identity.
# ============================================================

EXPECTED_STORES = [
    "CA_1", "CA_2", "CA_3", "CA_4",
    "TX_1", "TX_2", "TX_3",
    "WI_1", "WI_2", "WI_3"
]

# Actual filenames, preserving the original CA_1 and CA_2 names
FILE_MAPPING = {
    "CA_1": "ca1_advanced_features.parquet",
    "CA_2": "ca2_advanced_features.parquet",
    "CA_3": "ca_3_advanced_features.parquet",
    "CA_4": "ca_4_advanced_features.parquet",
    "TX_1": "tx_1_advanced_features.parquet",
    "TX_2": "tx_2_advanced_features.parquet",
    "TX_3": "tx_3_advanced_features.parquet",
    "WI_1": "wi_1_advanced_features.parquet",
    "WI_2": "wi_2_advanced_features.parquet",
    "WI_3": "wi_3_advanced_features.parquet"
}

EXPECTED_ROWS = 5_832_737
EXPECTED_COLUMNS = 30

verification_results = []

for store in EXPECTED_STORES:

    file_path = PROCESSED_DIR / FILE_MAPPING[store]

    if not file_path.exists():
        verification_results.append({
            "store": store,
            "rows": 0,
            "columns": 0,
            "actual_store": "MISSING",
            "status": "FAILED"
        })
        continue

    schema = pl.read_parquet_schema(file_path)

    summary = (
        pl.scan_parquet(file_path)
        .select(
            pl.len().alias("rows"),
            pl.col("store_id").n_unique().alias("unique_stores"),
            pl.col("store_id").first().alias("actual_store")
        )
        .collect()
    )

    row_count = summary["rows"][0]
    column_count = len(schema)
    unique_stores = summary["unique_stores"][0]
    actual_store = summary["actual_store"][0]

    passed = (
        row_count == EXPECTED_ROWS
        and column_count == EXPECTED_COLUMNS
        and unique_stores == 1
        and actual_store == store
    )

    verification_results.append({
        "store": store,
        "rows": row_count,
        "columns": column_count,
        "actual_store": actual_store,
        "status": "PASSED" if passed else "FAILED"
    })

verification_df = pl.DataFrame(verification_results)

print("CORRECTED FINAL FEATURE VERIFICATION")
print("=" * 65)
print(verification_df)

passed_count = verification_df.filter(
    pl.col("status") == "PASSED"
).height

failed_count = len(EXPECTED_STORES) - passed_count

print("\nTotal stores:", len(EXPECTED_STORES))
print("Passed:", passed_count)
print("Failed:", failed_count)

if failed_count == 0:
    print("\nALL 10 STORE FEATURE DATASETS VERIFIED SUCCESSFULLY!")
else:
    print("\nVERIFICATION FAILED - CHECK THE RESULTS")

CORRECTED FINAL FEATURE VERIFICATION
shape: (10, 5)
┌───────┬─────────┬─────────┬──────────────┬────────┐
│ store ┆ rows    ┆ columns ┆ actual_store ┆ status │
│ ---   ┆ ---     ┆ ---     ┆ ---          ┆ ---    │
│ str   ┆ i64     ┆ i64     ┆ str          ┆ str    │
╞═══════╪═════════╪═════════╪══════════════╪════════╡
│ CA_1  ┆ 5832737 ┆ 30      ┆ CA_1         ┆ PASSED │
│ CA_2  ┆ 5832737 ┆ 30      ┆ CA_2         ┆ PASSED │
│ CA_3  ┆ 5832737 ┆ 30      ┆ CA_3         ┆ PASSED │
│ CA_4  ┆ 5832737 ┆ 30      ┆ CA_4         ┆ PASSED │
│ TX_1  ┆ 5832737 ┆ 30      ┆ TX_1         ┆ PASSED │
│ TX_2  ┆ 5832737 ┆ 30      ┆ TX_2         ┆ PASSED │
│ TX_3  ┆ 5832737 ┆ 30      ┆ TX_3         ┆ PASSED │
│ WI_1  ┆ 5832737 ┆ 30      ┆ WI_1         ┆ PASSED │
│ WI_2  ┆ 5832737 ┆ 30      ┆ WI_2         ┆ PASSED │
│ WI_3  ┆ 5832737 ┆ 30      ┆ WI_3         ┆ PASSED │
└───────┴─────────┴─────────┴──────────────┴────────┘

Total stores: 10
Passed: 10
Failed: 0

ALL 10 STORE FEATURE DATASETS VERIFIED SUCCE

In [48]:
# ============================================================
# CELL 46: INSPECT NOTEBOOK 2 TRAINING IMPLEMENTATION
# Purpose:
# Extract existing LightGBM training and prediction cells
# so we can adapt them for all 10 stores and 3 folds.
# ============================================================

import json
from pathlib import Path

NOTEBOOK_2_PATH = NOTEBOOK_DIR / "02_advanced_feature_engineering.ipynb"

with open(NOTEBOOK_2_PATH, "r", encoding="utf-8") as file:
    notebook_data = json.load(file)

keywords = [
    "lgb.train",
    "LGBMRegressor",
    "lightgbm",
    "tweedie",
    "objective",
    "best_iteration",
    "predict("
]

print("LIGHTGBM TRAINING CELLS FROM NOTEBOOK 2")
print("=" * 65)

for cell_number, cell in enumerate(notebook_data["cells"]):

    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    if any(keyword.lower() in source.lower() for keyword in keywords):
        print(f"\n{'=' * 65}")
        print(f"CELL NUMBER: {cell_number}")
        print("=" * 65)
        print(source)

print("\nNOTEBOOK 2 INSPECTION COMPLETED")

LIGHTGBM TRAINING CELLS FROM NOTEBOOK 2

CELL NUMBER: 12

# ============================================================
# THURSDAY - CELL 12: TRAIN ADVANCED LIGHTGBM L2 MODEL
# Purpose:
# 1. Train LightGBM using squared-error regression.
# 2. Use internal validation for early stopping.
# 3. Evaluate predictions on the untouched outer validation set.
# 4. Save the trained model and prediction results.
# ============================================================

import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

# STEP 1: Configure the LightGBM L2 model.
l2_model = lgb.LGBMRegressor(
    objective="regression_l2",
    n_estimators=700,
    learning_rate=0.05,
    num_leaves=31,
    max_bin=127,
    min_child_samples=100,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    n_jobs=4,
    random_state=42,
    verbosity=-1
)

print("LIGHTGBM L2 TRAINING STARTED")
print("--------------------------------")
prin

In [49]:
# ============================================================
# CELL 47: INSPECT TRAINING DATA PREPARATION
# Purpose:
# Extract the existing feature selection, fold definitions,
# chronological splits, and categorical preprocessing logic.
# ============================================================

import json

NOTEBOOK_2_PATH = NOTEBOOK_DIR / "02_advanced_feature_engineering.ipynb"

with open(NOTEBOOK_2_PATH, "r", encoding="utf-8") as file:
    notebook_data = json.load(file)

keywords = [
    "feature_columns",
    "categorical_features",
    "numeric_features",
    "fold_definitions",
    "X_train",
    "X_early_stop",
    "X_outer",
    "outer_validation_start",
    "train_end",
    "early_stop"
]

print("TRAINING PREPARATION CODE")
print("=" * 70)

for cell_number, cell in enumerate(notebook_data["cells"]):

    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    if any(keyword.lower() in source.lower() for keyword in keywords):

        print(f"\n{'=' * 70}")
        print(f"CELL NUMBER: {cell_number}")
        print("=" * 70)
        print(source)

print("\nTRAINING PREPARATION INSPECTION COMPLETED")

TRAINING PREPARATION CODE

CELL NUMBER: 9

# ============================================================
# THURSDAY - CELL 9: DEFINE ROLLING-ORIGIN FOLDS
# Purpose:
# Define the three 28-day validation periods and create
# chronological training and validation boundaries.
# ============================================================

fold_definitions = {
    1: {
        "train_end": pd.Timestamp("2016-01-31"),
        "validation_start": pd.Timestamp("2016-02-01"),
        "validation_end": pd.Timestamp("2016-02-28")
    },
    2: {
        "train_end": pd.Timestamp("2016-02-28"),
        "validation_start": pd.Timestamp("2016-02-29"),
        "validation_end": pd.Timestamp("2016-03-27")
    },
    3: {
        "train_end": pd.Timestamp("2016-03-27"),
        "validation_start": pd.Timestamp("2016-03-28"),
        "validation_end": pd.Timestamp("2016-04-24")
    }
}

# The final 28 days of each training window are used
# as internal validation for early stopping.
INNER_VALIDATION_DA

In [50]:
# ============================================================
# CELL 48: CHECK TRAINING RESOURCES AND FOLD 1 DATA
# Purpose:
# 1. Check available system memory.
# 2. Inspect the CA_1 feature dataset schema.
# 3. Verify chronological partition row counts.
# 4. Confirm that the required model features exist.
# ============================================================

import psutil
import polars as pl
import pandas as pd

# Check available memory
memory = psutil.virtual_memory()

print("SYSTEM RESOURCE CHECK")
print("=" * 55)
print("Total RAM (GB):", round(memory.total / (1024**3), 2))
print("Available RAM (GB):", round(memory.available / (1024**3), 2))
print("RAM usage (%):", memory.percent)

# Locate CA_1 feature dataset
CA1_FEATURE_PATH = PROCESSED_DIR / "ca1_advanced_features.parquet"

print("\nCA_1 FEATURE DATASET")
print("=" * 55)
print("File exists:", CA1_FEATURE_PATH.exists())

# Inspect schema without loading the full dataset
ca1_schema = pl.read_parquet_schema(CA1_FEATURE_PATH)

print("Number of columns:", len(ca1_schema))
print("Columns:", list(ca1_schema.keys()))

# Define model features from Notebook 2
feature_columns = [
    "item_id", "event_name_1", "day_of_week", "month", "snap_flag",
    "lag_28", "lag_35", "lag_42", "lag_56", "lag_364",
    "rolling_mean_7", "rolling_mean_28", "rolling_mean_56",
    "rolling_std_7", "rolling_std_28",
    "day_of_week_sin", "day_of_week_cos",
    "month_sin", "month_cos",
    "ema_28", "zero_sales_run",
    "price_lag_28", "price_lag_56",
    "price_momentum", "historical_max_price_56", "discount_ratio"
]

missing_features = [
    col for col in feature_columns
    if col not in ca1_schema
]

print("\nRequired model features:", len(feature_columns))
print("Missing features:", missing_features)

# Define Fold 1 chronological partitions
fold1_train_end = pd.Timestamp("2016-01-31")
fold1_validation_start = pd.Timestamp("2016-02-01")
fold1_validation_end = pd.Timestamp("2016-02-28")

inner_validation_start = (
    fold1_train_end - pd.Timedelta(days=27)
)

inner_train_end = inner_validation_start - pd.Timedelta(days=1)

# Scan dataset without loading everything into memory
ca1_scan = pl.scan_parquet(CA1_FEATURE_PATH)

def count_date_rows(start_date, end_date):
    return (
        ca1_scan
        .filter(
            (pl.col("date") >= start_date) &
            (pl.col("date") <= end_date)
        )
        .select(pl.len())
        .collect()
        .item()
    )

training_rows = count_date_rows(
    ca1_schema_date_start := pd.Timestamp("2011-01-29"),
    inner_train_end
)

early_stop_rows = count_date_rows(
    inner_validation_start,
    fold1_train_end
)

outer_validation_rows = count_date_rows(
    fold1_validation_start,
    fold1_validation_end
)

print("\nFOLD 1 PARTITION VERIFICATION")
print("=" * 55)
print("Internal training ends:", inner_train_end.date())
print("Internal validation:", inner_validation_start.date(), "to", fold1_train_end.date())
print("Outer validation:", fold1_validation_start.date(), "to", fold1_validation_end.date())

print("\nTraining rows:", training_rows)
print("Early stopping rows:", early_stop_rows)
print("Outer validation rows:", outer_validation_rows)

print("\nMissing features:", len(missing_features))

print("\nRESOURCE AND DATA CHECK COMPLETED")

SYSTEM RESOURCE CHECK
Total RAM (GB): 15.82
Available RAM (GB): 5.47
RAM usage (%): 65.4

CA_1 FEATURE DATASET
File exists: True
Number of columns: 30
Columns: ['item_id', 'store_id', 'date', 'sales', 'sell_price', 'event_name_1', 'snap_flag', 'lag_28', 'lag_35', 'lag_42', 'lag_56', 'lag_364', 'day_of_week', 'month', 'rolling_mean_7', 'rolling_mean_28', 'rolling_mean_56', 'rolling_std_7', 'rolling_std_28', 'day_of_week_sin', 'day_of_week_cos', 'month_sin', 'month_cos', 'ema_28', 'zero_sales_run', 'price_lag_28', 'price_lag_56', 'price_momentum', 'historical_max_price_56', 'discount_ratio']

Required model features: 26
Missing features: []

FOLD 1 PARTITION VERIFICATION
Internal training ends: 2016-01-03
Internal validation: 2016-01-04 to 2016-01-31
Outer validation: 2016-02-01 to 2016-02-28

Training rows: 5491249
Early stopping rows: 85372
Outer validation rows: 85372

Missing features: 0

RESOURCE AND DATA CHECK COMPLETED


In [51]:
# ============================================================
# CELL 49: PREPARE CA_1 FOLD 1 MODEL DATA
# Purpose:
# 1. Load only the required columns.
# 2. Create chronological training/validation partitions.
# 3. Apply consistent categorical encodings.
# 4. Reduce memory usage using float32.
# ============================================================

import gc
import numpy as np
import pandas as pd
import polars as pl

categorical_features = [
    "item_id",
    "event_name_1",
    "day_of_week",
    "month"
]

numeric_features = [
    col for col in feature_columns
    if col not in categorical_features
]

# Read only the columns needed for model training
required_columns = ["date", "sales"] + feature_columns

ca1_data = (
    pl.scan_parquet(CA1_FEATURE_PATH)
    .select(required_columns)
    .collect()
    .to_pandas()
)

ca1_data["date"] = pd.to_datetime(ca1_data["date"])

print("CA_1 DATA LOADED:", ca1_data.shape)

# Define chronological boundaries
inner_validation_start = pd.Timestamp("2016-01-04")
inner_train_end = pd.Timestamp("2016-01-03")

outer_validation_start = pd.Timestamp("2016-02-01")
outer_validation_end = pd.Timestamp("2016-02-28")

# Create the three partitions
train_data = ca1_data[
    ca1_data["date"] <= inner_train_end
].copy()

early_stop_data = ca1_data[
    (ca1_data["date"] >= inner_validation_start) &
    (ca1_data["date"] <= pd.Timestamp("2016-01-31"))
].copy()

outer_test_data = ca1_data[
    (ca1_data["date"] >= outer_validation_start) &
    (ca1_data["date"] <= outer_validation_end)
].copy()

# Separate features and target
X_train = train_data[feature_columns].copy()
y_train = train_data["sales"].to_numpy(dtype=np.float32)

X_early_stop = early_stop_data[feature_columns].copy()
y_early_stop = early_stop_data["sales"].to_numpy(dtype=np.float32)

X_outer = outer_test_data[feature_columns].copy()
y_outer = outer_test_data["sales"].to_numpy(dtype=np.float32)

# Apply category levels from training data to all partitions
category_types = {}

for col in categorical_features:
    categories = pd.Index(X_train[col].dropna().unique())

    category_types[col] = pd.CategoricalDtype(
        categories=categories
    )

    for dataset in [X_train, X_early_stop, X_outer]:
        dataset[col] = dataset[col].astype(category_types[col])

# Convert numerical features to float32
for col in numeric_features:
    for dataset in [X_train, X_early_stop, X_outer]:
        dataset[col] = pd.to_numeric(
            dataset[col], errors="coerce"
        ).astype(np.float32)

# Verify partitions
print("\nPARTITION SHAPES")
print("Training:", X_train.shape)
print("Early stopping:", X_early_stop.shape)
print("Outer validation:", X_outer.shape)

print("\nTARGET SHAPES")
print("Training:", y_train.shape)
print("Early stopping:", y_early_stop.shape)
print("Outer validation:", y_outer.shape)

print("\nMISSING VALUES")
print("Training:", X_train.isna().sum().sum())
print("Early stopping:", X_early_stop.isna().sum().sum())
print("Outer validation:", X_outer.isna().sum().sum())

print("\nDATA TYPES")
print(X_train.dtypes.value_counts())

# Assertions
assert len(X_train) == 5_491_249
assert len(X_early_stop) == 85_372
assert len(X_outer) == 85_372
assert len(feature_columns) == 26

print("\nFOLD 1 DATA PREPARATION VERIFIED!")

CA_1 DATA LOADED: (5832737, 28)

PARTITION SHAPES
Training: (5491249, 26)
Early stopping: (85372, 26)
Outer validation: (85372, 26)

TARGET SHAPES
Training: (5491249,)
Early stopping: (85372,)
Outer validation: (85372,)

MISSING VALUES
Training: 14200239
Early stopping: 79838
Outer validation: 70579

DATA TYPES
float32     22
category     1
category     1
category     1
category     1
Name: count, dtype: int64

FOLD 1 DATA PREPARATION VERIFIED!


In [52]:
# ============================================================
# CELL 50: MISSING VALUE AND DATA QUALITY DIAGNOSTICS
# Purpose:
# 1. Identify which features contain missing values.
# 2. Check whether target sales values are missing or invalid.
# 3. Check numerical features for infinite values.
# 4. Decide whether the data is safe for LightGBM training.
# ============================================================

datasets = {
    "Training": (X_train, y_train),
    "Early Stopping": (X_early_stop, y_early_stop),
    "Outer Validation": (X_outer, y_outer)
}

for dataset_name, (X, y) in datasets.items():

    print(f"\n{'=' * 55}")
    print(dataset_name.upper())
    print("=" * 55)

    # Missing values by feature
    missing_counts = X.isna().sum()
    missing_counts = missing_counts[missing_counts > 0]

    print("\nFEATURES WITH MISSING VALUES:")
    if len(missing_counts) > 0:
        print(missing_counts.sort_values(ascending=False))
    else:
        print("No missing feature values.")

    # Target checks
    print("\nTARGET CHECK:")
    print("Missing sales:", pd.isna(y).sum())
    print("Negative sales:", (y < 0).sum())
    print("Infinite sales:", np.isinf(y).sum())

    # Numerical infinity checks
    infinite_counts = {}

    for col in numeric_features:
        count = np.isinf(X[col].to_numpy(dtype=np.float32)).sum()
        if count > 0:
            infinite_counts[col] = int(count)

    print("\nINFINITE NUMERICAL FEATURE VALUES:")
    print(infinite_counts if infinite_counts else "None")

print("\nDATA QUALITY DIAGNOSTICS COMPLETED.")


TRAINING

FEATURES WITH MISSING VALUES:
event_name_1               5052193
discount_ratio             1382532
historical_max_price_56    1382532
price_momentum             1300328
price_lag_56               1300328
price_lag_28               1215068
lag_364                    1109836
rolling_mean_56             253067
lag_56                      170744
rolling_mean_28             167695
rolling_std_28              167695
lag_42                      128058
lag_35                      106715
rolling_std_7               103666
rolling_mean_7              103666
lag_28                       85372
zero_sales_run               85372
ema_28                       85372
dtype: int64

TARGET CHECK:
Missing sales: 0
Negative sales: 0
Infinite sales: 0

INFINITE NUMERICAL FEATURE VALUES:
None

EARLY STOPPING

FEATURES WITH MISSING VALUES:
event_name_1               79274
discount_ratio               123
historical_max_price_56      123
price_momentum               112
price_lag_56                

In [53]:
# ============================================================
# CELL 51: TRAIN CA_1 FOLD 1 - LIGHTGBM L2
# Purpose:
# 1. Release duplicate DataFrames to reduce memory usage.
# 2. Train LightGBM using the internal early-stopping period.
# 3. Evaluate predictions on the untouched outer validation set.
# 4. Save the trained model and predictions.
# ============================================================

import gc
import psutil
import lightgbm as lgb
import numpy as np
import pandas as pd
import polars as pl

# Preserve validation metadata before releasing source DataFrames
outer_metadata = outer_test_data[["item_id", "date", "sales"]].copy()

# Release duplicate DataFrames
del ca1_data, train_data, early_stop_data, outer_test_data
gc.collect()

print("Available RAM before training:",
      round(psutil.virtual_memory().available / (1024**3), 2), "GB")

# Define LightGBM L2 model
l2_model = lgb.LGBMRegressor(
    objective="regression_l2",
    n_estimators=700,
    learning_rate=0.05,
    num_leaves=31,
    max_bin=127,
    min_child_samples=100,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    n_jobs=4,
    random_state=42,
    verbosity=-1
)

# Train with internal early stopping
print("\nSTARTING LIGHTGBM L2 TRAINING...")

l2_model.fit(
    X_train,
    y_train,
    eval_set=[(X_early_stop, y_early_stop)],
    eval_metric="rmse",
    categorical_feature=categorical_features,
    callbacks=[
        lgb.early_stopping(50),
        lgb.log_evaluation(50)
    ]
)

print("\nTRAINING COMPLETED")
print("Best iteration:", l2_model.best_iteration_)

# Predict on untouched outer validation data
l2_predictions = np.maximum(
    l2_model.predict(
        X_outer,
        num_iteration=l2_model.best_iteration_
    ),
    0
)

actual = y_outer

# Calculate evaluation metrics
mae = np.mean(np.abs(actual - l2_predictions))
rmse = np.sqrt(np.mean((actual - l2_predictions) ** 2))

wape = (
    np.sum(np.abs(actual - l2_predictions)) /
    np.sum(np.abs(actual))
) * 100

bias = (
    np.sum(l2_predictions - actual) /
    np.sum(actual)
) * 100

print("\nCA_1 FOLD 1 - LIGHTGBM L2 RESULTS")
print("=" * 45)
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"WAPE: {wape:.4f}%")
print(f"Bias: {bias:.4f}%")

# Create prediction output
l2_prediction_output = outer_metadata.copy()
l2_prediction_output = l2_prediction_output.rename(
    columns={"sales": "actual", "date": "forecast_date"}
)

l2_prediction_output["prediction"] = l2_predictions
l2_prediction_output["fold"] = 1
l2_prediction_output["model"] = "LightGBM_L2"
l2_prediction_output["horizon"] = (
    l2_prediction_output.groupby("item_id").cumcount() + 1
)

# Save model
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)

l2_model_path = MODEL_DIR / "lightgbm_l2_advanced_ca1_fold1.txt"
l2_model.booster_.save_model(str(l2_model_path))

# Save predictions
l2_prediction_path = (
    PREDICTION_DIR / "lightgbm_l2_advanced_ca1_fold1.parquet"
)

pl.from_pandas(l2_prediction_output).write_parquet(
    l2_prediction_path
)

print("\nSAVED OUTPUTS")
print("Model:", l2_model_path)
print("Predictions:", l2_prediction_path)
print("Prediction rows:", len(l2_prediction_output))

print("\nAVAILABLE RAM AFTER TRAINING:",
      round(psutil.virtual_memory().available / (1024**3), 2), "GB")

Available RAM before training: 5.18 GB

STARTING LIGHTGBM L2 TRAINING...


D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[50]	valid_0's rmse: 2.15549	valid_0's l2: 4.64615
[100]	valid_0's rmse: 2.14085	valid_0's l2: 4.58324
[150]	valid_0's rmse: 2.14271	valid_0's l2: 4.5912
Early stopping, best iteration is:
[100]	valid_0's rmse: 2.14085	valid_0's l2: 4.58324

TRAINING COMPLETED
Best iteration: 100

CA_1 FOLD 1 - LIGHTGBM L2 RESULTS
MAE:  1.1100
RMSE: 2.2052
WAPE: 77.1289%
Bias: -2.6981%

SAVED OUTPUTS
Model: D:\Retail-Demand\models\lightgbm_l2_advanced_ca1_fold1.txt
Predictions: D:\Retail-Demand\predictions\lightgbm_l2_advanced_ca1_fold1.parquet
Prediction rows: 85372

AVAILABLE RAM AFTER TRAINING: 7.6 GB


In [54]:
# ============================================================
# CELL 52: TRAIN CA_1 FOLD 1 - LIGHTGBM TWEEDIE
# Purpose:
# 1. Train a Tweedie regression model on the same Fold 1 data.
# 2. Use the same early-stopping period for a fair comparison.
# 3. Evaluate on the untouched outer validation period.
# 4. Save the model and predictions.
# ============================================================

tweedie_model = lgb.LGBMRegressor(
    objective="tweedie",
    tweedie_variance_power=1.2,
    n_estimators=700,
    learning_rate=0.05,
    num_leaves=31,
    max_bin=127,
    min_child_samples=100,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    n_jobs=4,
    random_state=42,
    verbosity=-1
)

print("STARTING LIGHTGBM TWEEDIE TRAINING...")

tweedie_model.fit(
    X_train,
    y_train,
    eval_set=[(X_early_stop, y_early_stop)],
    eval_metric="rmse",
    categorical_feature=categorical_features,
    callbacks=[
        lgb.early_stopping(50),
        lgb.log_evaluation(50)
    ]
)

print("\nTRAINING COMPLETED")
print("Best iteration:", tweedie_model.best_iteration_)

# Predict on the same untouched outer validation data
tweedie_predictions = np.maximum(
    tweedie_model.predict(
        X_outer,
        num_iteration=tweedie_model.best_iteration_
    ),
    0
)

# Calculate evaluation metrics
tweedie_mae = np.mean(np.abs(y_outer - tweedie_predictions))
tweedie_rmse = np.sqrt(
    np.mean((y_outer - tweedie_predictions) ** 2)
)

tweedie_wape = (
    np.sum(np.abs(y_outer - tweedie_predictions)) /
    np.sum(np.abs(y_outer))
) * 100

tweedie_bias = (
    np.sum(tweedie_predictions - y_outer) /
    np.sum(y_outer)
) * 100

print("\nCA_1 FOLD 1 - LIGHTGBM TWEEDIE RESULTS")
print("=" * 45)
print(f"MAE:  {tweedie_mae:.4f}")
print(f"RMSE: {tweedie_rmse:.4f}")
print(f"WAPE: {tweedie_wape:.4f}%")
print(f"Bias: {tweedie_bias:.4f}%")

# Create prediction output
tweedie_prediction_output = outer_metadata.copy()

tweedie_prediction_output = tweedie_prediction_output.rename(
    columns={"sales": "actual", "date": "forecast_date"}
)

tweedie_prediction_output["prediction"] = tweedie_predictions
tweedie_prediction_output["fold"] = 1
tweedie_prediction_output["model"] = "LightGBM_Tweedie"
tweedie_prediction_output["horizon"] = (
    tweedie_prediction_output.groupby("item_id").cumcount() + 1
)

# Save model and predictions
tweedie_model_path = (
    MODEL_DIR / "lightgbm_tweedie_advanced_ca1_fold1.txt"
)

tweedie_model.booster_.save_model(str(tweedie_model_path))

tweedie_prediction_path = (
    PREDICTION_DIR / "lightgbm_tweedie_advanced_ca1_fold1.parquet"
)

pl.from_pandas(tweedie_prediction_output).write_parquet(
    tweedie_prediction_path
)

print("\nSAVED OUTPUTS")
print("Model:", tweedie_model_path)
print("Predictions:", tweedie_prediction_path)
print("Prediction rows:", len(tweedie_prediction_output))

print("\nAVAILABLE RAM:",
      round(psutil.virtual_memory().available / (1024**3), 2), "GB")

STARTING LIGHTGBM TWEEDIE TRAINING...


D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[50]	valid_0's rmse: 2.17411	valid_0's tweedie: 7.22942
[100]	valid_0's rmse: 2.14178	valid_0's tweedie: 7.20929
[150]	valid_0's rmse: 2.14163	valid_0's tweedie: 7.20504
Early stopping, best iteration is:
[122]	valid_0's rmse: 2.14086	valid_0's tweedie: 7.20671

TRAINING COMPLETED
Best iteration: 122

CA_1 FOLD 1 - LIGHTGBM TWEEDIE RESULTS
MAE:  1.0999
RMSE: 2.1974
WAPE: 76.4282%
Bias: -3.1261%

SAVED OUTPUTS
Model: D:\Retail-Demand\models\lightgbm_tweedie_advanced_ca1_fold1.txt
Predictions: D:\Retail-Demand\predictions\lightgbm_tweedie_advanced_ca1_fold1.parquet
Prediction rows: 85372

AVAILABLE RAM: 7.3 GB


In [55]:
# ============================================================
# CELL 53: INSPECT BASELINE PREDICTION FILES
# Purpose:
# 1. Find existing Seasonal Naive and Croston SBA predictions.
# 2. Inspect their filenames, schemas and row counts.
# 3. Confirm they contain the CA_1 Fold 1 predictions.
# ============================================================

print("BASELINE PREDICTION FILES")
print("=" * 55)

baseline_files = sorted(PREDICTION_DIR.glob("*.parquet"))

for file_path in baseline_files:
    filename = file_path.name.lower()

    if "seasonal" in filename or "croston" in filename:
        print(f"\nFile: {file_path.name}")

        baseline_df = pl.read_parquet(file_path)

        print("Shape:", baseline_df.shape)
        print("Columns:", baseline_df.columns)
        print("Preview:")
        print(baseline_df.head(3))

print("\nBASELINE FILE INSPECTION COMPLETED.")

BASELINE PREDICTION FILES

File: croston_sba_fold_1.parquet
Shape: (853720, 6)
Columns: ['item_id', 'store_id', 'forecast_date', 'horizon', 'forecast', 'actual']
Preview:
shape: (3, 6)
┌─────────────────┬──────────┬───────────────┬─────────┬──────────┬────────┐
│ item_id         ┆ store_id ┆ forecast_date ┆ horizon ┆ forecast ┆ actual │
│ ---             ┆ ---      ┆ ---           ┆ ---     ┆ ---      ┆ ---    │
│ str             ┆ str      ┆ date          ┆ i64     ┆ f64      ┆ i64    │
╞═════════════════╪══════════╪═══════════════╪═════════╪══════════╪════════╡
│ HOUSEHOLD_2_053 ┆ WI_3     ┆ 2016-02-03    ┆ 3       ┆ 0.199172 ┆ 0      │
│ HOUSEHOLD_2_058 ┆ WI_3     ┆ 2016-02-03    ┆ 3       ┆ 0.200055 ┆ 0      │
│ HOUSEHOLD_2_065 ┆ WI_3     ┆ 2016-02-03    ┆ 3       ┆ 0.07486  ┆ 0      │
└─────────────────┴──────────┴───────────────┴─────────┴──────────┴────────┘

File: croston_sba_fold_2.parquet
Shape: (853720, 6)
Columns: ['item_id', 'store_id', 'forecast_date', 'horizon', 'forecas

In [56]:
# ============================================================
# CELL 54: CA_1 FOLD 1 - FOUR MODEL COMPARISON
# Purpose:
# 1. Filter Seasonal Naive and Croston SBA to CA_1.
# 2. Align all four models using item and forecast date.
# 3. Verify actual values match.
# 4. Calculate MAE, RMSE, WAPE and Bias.
# 5. Save the comparison table.
# ============================================================

def calculate_metrics(actual, prediction):
    actual = np.asarray(actual, dtype=np.float64)
    prediction = np.asarray(prediction, dtype=np.float64)

    return {
        "MAE": np.mean(np.abs(actual - prediction)),
        "RMSE": np.sqrt(np.mean((actual - prediction) ** 2)),
        "WAPE (%)": (
            np.sum(np.abs(actual - prediction)) /
            np.sum(np.abs(actual))
        ) * 100,
        "Bias (%)": (
            np.sum(prediction - actual) /
            np.sum(actual)
        ) * 100
    }


# Load and filter baseline predictions
seasonal_path = PREDICTION_DIR / "seasonal_naive_fold_1.parquet"
croston_path = PREDICTION_DIR / "croston_sba_fold_1.parquet"

seasonal_ca1 = (
    pl.read_parquet(seasonal_path)
    .filter(pl.col("store_id") == "CA_1")
    .to_pandas()
)

croston_ca1 = (
    pl.read_parquet(croston_path)
    .filter(pl.col("store_id") == "CA_1")
    .to_pandas()
)

# Normalize baseline column names
seasonal_ca1 = seasonal_ca1.rename(
    columns={"forecast": "prediction"}
)

croston_ca1 = croston_ca1.rename(
    columns={"forecast": "prediction"}
)

# Prepare LightGBM outputs
l2_ca1 = l2_prediction_output.copy()
tweedie_ca1 = tweedie_prediction_output.copy()

# Normalize date types
for df in [seasonal_ca1, croston_ca1, l2_ca1, tweedie_ca1]:
    df["forecast_date"] = pd.to_datetime(df["forecast_date"])

# Verify row counts
print("ROW COUNT VERIFICATION")
print("Seasonal Naive:", len(seasonal_ca1))
print("Croston SBA:", len(croston_ca1))
print("LightGBM L2:", len(l2_ca1))
print("LightGBM Tweedie:", len(tweedie_ca1))

assert all(
    len(df) == 85372
    for df in [seasonal_ca1, croston_ca1, l2_ca1, tweedie_ca1]
), "Unexpected row count detected!"

# Verify actual sales values align by item and date
key_columns = ["item_id", "forecast_date"]

reference = seasonal_ca1[
    key_columns + ["actual"]
].sort_values(key_columns).reset_index(drop=True)

for name, df in [
    ("Croston SBA", croston_ca1),
    ("LightGBM L2", l2_ca1),
    ("LightGBM Tweedie", tweedie_ca1)
]:
    check = df[
        key_columns + ["actual"]
    ].sort_values(key_columns).reset_index(drop=True)

    assert reference[key_columns].equals(check[key_columns]), (
        f"{name}: item/date alignment mismatch"
    )

    assert np.array_equal(
        reference["actual"].to_numpy(),
        check["actual"].to_numpy()
    ), f"{name}: actual sales mismatch"

print("\nALL FOUR MODELS HAVE MATCHING OBSERVATIONS.")

# Calculate comparison metrics
model_predictions = {
    "Seasonal Naive": seasonal_ca1["prediction"],
    "Croston SBA": croston_ca1["prediction"],
    "LightGBM L2": l2_ca1["prediction"],
    "LightGBM Tweedie": tweedie_ca1["prediction"]
}

comparison_rows = []

actual_values = reference["actual"].to_numpy()

for model_name, predictions in model_predictions.items():
    metrics = calculate_metrics(actual_values, predictions)

    comparison_rows.append({
        "Model": model_name,
        **metrics
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df = comparison_df.sort_values(
    "WAPE (%)"
).reset_index(drop=True)

print("\nCA_1 FOLD 1 MODEL COMPARISON")
print("=" * 75)
print(comparison_df.round(4).to_string(index=False))

# Save comparison
comparison_path = (
    RESULTS_DIR / "ca1_fold1_four_model_comparison.csv"
)

comparison_df.to_csv(comparison_path, index=False)

print("\nComparison saved:", comparison_path)
print("BEST MODEL BY WAPE:", comparison_df.iloc[0]["Model"])

ROW COUNT VERIFICATION
Seasonal Naive: 85372
Croston SBA: 85372
LightGBM L2: 85372
LightGBM Tweedie: 85372

ALL FOUR MODELS HAVE MATCHING OBSERVATIONS.

CA_1 FOLD 1 MODEL COMPARISON
           Model    MAE   RMSE  WAPE (%)  Bias (%)
LightGBM Tweedie 1.0999 2.1974   76.4282   -3.1261
     LightGBM L2 1.1100 2.2052   77.1289   -2.6981
     Croston SBA 2.0681 4.2675  143.7066   -0.7856
  Seasonal Naive 2.1600 4.6648  150.0936   -6.9104

Comparison saved: D:\Retail-Demand\results\ca1_fold1_four_model_comparison.csv
BEST MODEL BY WAPE: LightGBM Tweedie


In [57]:
# ============================================================
# CELL 55: PREPARE CA_1 FOLD 2
# Purpose:
# 1. Release Fold 1 training data from memory.
# 2. Load the CA_1 feature data for Fold 2.
# 3. Create independent training, early-stopping and test sets.
# 4. Verify the chronological split and row counts.
# ============================================================

import gc
import psutil
import pandas as pd
import numpy as np
import polars as pl

# Release Fold 1 data and models no longer needed in memory
del X_train, X_early_stop, X_outer
del y_train, y_early_stop, y_outer
del l2_model, tweedie_model
del l2_predictions, tweedie_predictions
del outer_metadata
del l2_prediction_output, tweedie_prediction_output
del seasonal_ca1, croston_ca1, l2_ca1, tweedie_ca1
del reference, comparison_df, model_predictions, comparison_rows

gc.collect()

print("AVAILABLE RAM:", round(
    psutil.virtual_memory().available / (1024**3), 2
), "GB")

# Fold 2 boundaries
fold2_train_end = pd.Timestamp("2016-02-28")
fold2_inner_validation_start = pd.Timestamp("2016-02-01")
fold2_inner_train_end = pd.Timestamp("2016-01-31")

fold2_outer_validation_start = pd.Timestamp("2016-02-29")
fold2_outer_validation_end = pd.Timestamp("2016-03-27")

# Load only the required columns
required_columns = ["date", "sales"] + feature_columns

fold2_data = (
    pl.scan_parquet(CA1_FEATURE_PATH)
    .select(required_columns)
    .collect()
    .to_pandas()
)

fold2_data["date"] = pd.to_datetime(fold2_data["date"])

# Create chronological partitions
train_data = fold2_data[
    fold2_data["date"] <= fold2_inner_train_end
].copy()

early_stop_data = fold2_data[
    (fold2_data["date"] >= fold2_inner_validation_start) &
    (fold2_data["date"] <= fold2_train_end)
].copy()

outer_test_data = fold2_data[
    (fold2_data["date"] >= fold2_outer_validation_start) &
    (fold2_data["date"] <= fold2_outer_validation_end)
].copy()

# Separate features and target
X_train = train_data[feature_columns].copy()
y_train = train_data["sales"].to_numpy(dtype=np.float32)

X_early_stop = early_stop_data[feature_columns].copy()
y_early_stop = early_stop_data["sales"].to_numpy(dtype=np.float32)

X_outer = outer_test_data[feature_columns].copy()
y_outer = outer_test_data["sales"].to_numpy(dtype=np.float32)

# Apply consistent categorical encodings
categorical_features = [
    "item_id", "event_name_1", "day_of_week", "month"
]

numeric_features = [
    col for col in feature_columns
    if col not in categorical_features
]

category_types = {}

for col in categorical_features:
    categories = pd.Index(X_train[col].dropna().unique())
    category_types[col] = pd.CategoricalDtype(categories=categories)

    for dataset in [X_train, X_early_stop, X_outer]:
        dataset[col] = dataset[col].astype(category_types[col])

# Convert numerical features to float32
for col in numeric_features:
    for dataset in [X_train, X_early_stop, X_outer]:
        dataset[col] = pd.to_numeric(
            dataset[col], errors="coerce"
        ).astype(np.float32)

# Preserve outer validation metadata
outer_metadata = outer_test_data[
    ["item_id", "date", "sales"]
].copy()

# Verify
print("\nFOLD 2 PARTITION VERIFICATION")
print("=" * 50)
print("Training:", X_train.shape)
print("Early stopping:", X_early_stop.shape)
print("Outer validation:", X_outer.shape)

print("\nDATE RANGES")
print("Training ends:", train_data["date"].max().date())
print("Early stopping:", early_stop_data["date"].min().date(),
      "to", early_stop_data["date"].max().date())
print("Outer validation:", outer_test_data["date"].min().date(),
      "to", outer_test_data["date"].max().date())

assert len(X_early_stop) == 85372
assert len(X_outer) == 85372
assert len(X_train) == 5576610

print("\nFOLD 2 DATA PREPARATION VERIFIED!")
print("AVAILABLE RAM:", round(
    psutil.virtual_memory().available / (1024**3), 2
), "GB")

AVAILABLE RAM: 7.93 GB

FOLD 2 PARTITION VERIFICATION
Training: (5576621, 26)
Early stopping: (85372, 26)
Outer validation: (85372, 26)

DATE RANGES
Training ends: 2016-01-31
Early stopping: 2016-02-01 to 2016-02-28
Outer validation: 2016-02-29 to 2016-03-27


AssertionError: 

In [58]:
# ============================================================
# CELL 56: VERIFY FOLD 2 ROW COUNT
# Purpose:
# 1. Recheck the actual row count directly from the feature file.
# 2. Compare it with the prepared training partition.
# 3. Confirm the correct count before model training.
# ============================================================

direct_train_count = (
    pl.scan_parquet(CA1_FEATURE_PATH)
    .filter(pl.col("date") <= pd.Timestamp("2016-01-31"))
    .select(pl.len())
    .collect()
    .item()
)

print("DIRECT TRAINING ROW COUNT:", direct_train_count)
print("PREPARED TRAINING ROW COUNT:", len(X_train))
print("DIFFERENCE:", len(X_train) - direct_train_count)

print("\nEARLY STOPPING ROWS:", len(X_early_stop))
print("OUTER VALIDATION ROWS:", len(X_outer))

assert direct_train_count == len(X_train)
assert len(X_early_stop) == 85372
assert len(X_outer) == 85372

print("\nFOLD 2 PARTITIONS VERIFIED!")

DIRECT TRAINING ROW COUNT: 5576621
PREPARED TRAINING ROW COUNT: 5576621
DIFFERENCE: 0

EARLY STOPPING ROWS: 85372
OUTER VALIDATION ROWS: 85372

FOLD 2 PARTITIONS VERIFIED!


In [59]:
# ============================================================
# CELL 57: TRAIN CA_1 FOLD 2 - LIGHTGBM L2
# ============================================================

l2_model = lgb.LGBMRegressor(
    objective="regression_l2",
    n_estimators=700,
    learning_rate=0.05,
    num_leaves=31,
    max_bin=127,
    min_child_samples=100,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    n_jobs=4,
    random_state=42,
    verbosity=-1
)

print("STARTING CA_1 FOLD 2 LIGHTGBM L2 TRAINING...")

l2_model.fit(
    X_train,
    y_train,
    eval_set=[(X_early_stop, y_early_stop)],
    eval_metric="rmse",
    categorical_feature=categorical_features,
    callbacks=[
        lgb.early_stopping(50),
        lgb.log_evaluation(50)
    ]
)

print("\nBEST ITERATION:", l2_model.best_iteration_)

# Predict on Fold 2 outer validation
l2_predictions = np.maximum(
    l2_model.predict(
        X_outer,
        num_iteration=l2_model.best_iteration_
    ),
    0
)

# Calculate metrics
fold2_l2_metrics = calculate_metrics(y_outer, l2_predictions)

print("\nCA_1 FOLD 2 - LIGHTGBM L2 RESULTS")
for metric, value in fold2_l2_metrics.items():
    print(f"{metric}: {value:.4f}")

# Prepare prediction output
l2_prediction_output = outer_metadata.copy()

l2_prediction_output = l2_prediction_output.rename(
    columns={"sales": "actual", "date": "forecast_date"}
)

l2_prediction_output["prediction"] = l2_predictions
l2_prediction_output["fold"] = 2
l2_prediction_output["model"] = "LightGBM_L2"
l2_prediction_output["horizon"] = (
    l2_prediction_output.groupby("item_id").cumcount() + 1
)

# Save model and predictions
l2_model_path = MODEL_DIR / "lightgbm_l2_advanced_ca1_fold2.txt"
l2_model.booster_.save_model(str(l2_model_path))

l2_prediction_path = (
    PREDICTION_DIR / "lightgbm_l2_advanced_ca1_fold2.parquet"
)

pl.from_pandas(l2_prediction_output).write_parquet(
    l2_prediction_path
)

print("\nMODEL SAVED:", l2_model_path)
print("PREDICTIONS SAVED:", l2_prediction_path)
print("PREDICTION ROWS:", len(l2_prediction_output))
print("AVAILABLE RAM:",
      round(psutil.virtual_memory().available / (1024**3), 2), "GB")

STARTING CA_1 FOLD 2 LIGHTGBM L2 TRAINING...


D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[50]	valid_0's rmse: 2.229	valid_0's l2: 4.96846
[100]	valid_0's rmse: 2.20629	valid_0's l2: 4.8677
[150]	valid_0's rmse: 2.21381	valid_0's l2: 4.90094
Early stopping, best iteration is:
[109]	valid_0's rmse: 2.20584	valid_0's l2: 4.86573

BEST ITERATION: 109

CA_1 FOLD 2 - LIGHTGBM L2 RESULTS
MAE: 1.1265
RMSE: 2.2121
WAPE (%): 76.7848
Bias (%): -2.6274

MODEL SAVED: D:\Retail-Demand\models\lightgbm_l2_advanced_ca1_fold2.txt
PREDICTIONS SAVED: D:\Retail-Demand\predictions\lightgbm_l2_advanced_ca1_fold2.parquet
PREDICTION ROWS: 85372
AVAILABLE RAM: 5.48 GB


In [60]:
# ============================================================
# CELL 58: TRAIN CA_1 FOLD 2 - LIGHTGBM TWEEDIE
# ============================================================

tweedie_model = lgb.LGBMRegressor(
    objective="tweedie",
    tweedie_variance_power=1.2,
    n_estimators=700,
    learning_rate=0.05,
    num_leaves=31,
    max_bin=127,
    min_child_samples=100,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    n_jobs=4,
    random_state=42,
    verbosity=-1
)

print("STARTING CA_1 FOLD 2 LIGHTGBM TWEEDIE TRAINING...")

tweedie_model.fit(
    X_train,
    y_train,
    eval_set=[(X_early_stop, y_early_stop)],
    eval_metric="rmse",
    categorical_feature=categorical_features,
    callbacks=[
        lgb.early_stopping(50),
        lgb.log_evaluation(50)
    ]
)

print("\nBEST ITERATION:", tweedie_model.best_iteration_)

# Predict on Fold 2 outer validation
tweedie_predictions = np.maximum(
    tweedie_model.predict(
        X_outer,
        num_iteration=tweedie_model.best_iteration_
    ),
    0
)

# Calculate metrics
fold2_tweedie_metrics = calculate_metrics(
    y_outer, tweedie_predictions
)

print("\nCA_1 FOLD 2 - LIGHTGBM TWEEDIE RESULTS")
for metric, value in fold2_tweedie_metrics.items():
    print(f"{metric}: {value:.4f}")

# Prepare prediction output
tweedie_prediction_output = outer_metadata.copy()

tweedie_prediction_output = tweedie_prediction_output.rename(
    columns={"sales": "actual", "date": "forecast_date"}
)

tweedie_prediction_output["prediction"] = tweedie_predictions
tweedie_prediction_output["fold"] = 2
tweedie_prediction_output["model"] = "LightGBM_Tweedie"
tweedie_prediction_output["horizon"] = (
    tweedie_prediction_output.groupby("item_id").cumcount() + 1
)

# Save model and predictions
tweedie_model_path = (
    MODEL_DIR / "lightgbm_tweedie_advanced_ca1_fold2.txt"
)

tweedie_model.booster_.save_model(str(tweedie_model_path))

tweedie_prediction_path = (
    PREDICTION_DIR / "lightgbm_tweedie_advanced_ca1_fold2.parquet"
)

pl.from_pandas(tweedie_prediction_output).write_parquet(
    tweedie_prediction_path
)

print("\nMODEL SAVED:", tweedie_model_path)
print("PREDICTIONS SAVED:", tweedie_prediction_path)
print("PREDICTION ROWS:", len(tweedie_prediction_output))
print("AVAILABLE RAM:",
      round(psutil.virtual_memory().available / (1024**3), 2), "GB")

STARTING CA_1 FOLD 2 LIGHTGBM TWEEDIE TRAINING...


D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[50]	valid_0's rmse: 2.259	valid_0's tweedie: 7.44285
[100]	valid_0's rmse: 2.20228	valid_0's tweedie: 7.42337
[150]	valid_0's rmse: 2.19273	valid_0's tweedie: 7.41808
[200]	valid_0's rmse: 2.1891	valid_0's tweedie: 7.41474
[250]	valid_0's rmse: 2.18919	valid_0's tweedie: 7.4129
Early stopping, best iteration is:
[206]	valid_0's rmse: 2.1885	valid_0's tweedie: 7.41423

BEST ITERATION: 206

CA_1 FOLD 2 - LIGHTGBM TWEEDIE RESULTS
MAE: 1.1103
RMSE: 2.1918
WAPE (%): 75.6835
Bias (%): -2.0513

MODEL SAVED: D:\Retail-Demand\models\lightgbm_tweedie_advanced_ca1_fold2.txt
PREDICTIONS SAVED: D:\Retail-Demand\predictions\lightgbm_tweedie_advanced_ca1_fold2.parquet
PREDICTION ROWS: 85372
AVAILABLE RAM: 5.44 GB


In [61]:
# ============================================================
# CELL 59: CA_1 FOLD 2 - FOUR MODEL COMPARISON
# ============================================================

# Load Fold 2 baseline predictions and filter CA_1
seasonal_ca1 = (
    pl.read_parquet(PREDICTION_DIR / "seasonal_naive_fold_2.parquet")
    .filter(pl.col("store_id") == "CA_1")
    .to_pandas()
)

croston_ca1 = (
    pl.read_parquet(PREDICTION_DIR / "croston_sba_fold_2.parquet")
    .filter(pl.col("store_id") == "CA_1")
    .to_pandas()
)

seasonal_ca1 = seasonal_ca1.rename(columns={"forecast": "prediction"})
croston_ca1 = croston_ca1.rename(columns={"forecast": "prediction"})

l2_ca1 = l2_prediction_output.copy()
tweedie_ca1 = tweedie_prediction_output.copy()

# Standardize dates
for df in [seasonal_ca1, croston_ca1, l2_ca1, tweedie_ca1]:
    df["forecast_date"] = pd.to_datetime(df["forecast_date"])

# Verify row counts
print("ROW COUNT VERIFICATION")
for name, df in [
    ("Seasonal Naive", seasonal_ca1),
    ("Croston SBA", croston_ca1),
    ("LightGBM L2", l2_ca1),
    ("LightGBM Tweedie", tweedie_ca1)
]:
    print(f"{name}: {len(df)}")

assert all(
    len(df) == 85372
    for df in [seasonal_ca1, croston_ca1, l2_ca1, tweedie_ca1]
)

# Verify identical item/date keys and actual sales
key_columns = ["item_id", "forecast_date"]

reference = (
    seasonal_ca1[key_columns + ["actual"]]
    .sort_values(key_columns)
    .reset_index(drop=True)
)

for name, df in [
    ("Croston SBA", croston_ca1),
    ("LightGBM L2", l2_ca1),
    ("LightGBM Tweedie", tweedie_ca1)
]:
    check = (
        df[key_columns + ["actual"]]
        .sort_values(key_columns)
        .reset_index(drop=True)
    )

    assert reference[key_columns].equals(check[key_columns])
    assert np.array_equal(
        reference["actual"].to_numpy(),
        check["actual"].to_numpy()
    )

print("\nALL FOUR MODELS HAVE MATCHING OBSERVATIONS.")

# Calculate metrics
model_predictions = {
    "Seasonal Naive": seasonal_ca1["prediction"],
    "Croston SBA": croston_ca1["prediction"],
    "LightGBM L2": l2_ca1["prediction"],
    "LightGBM Tweedie": tweedie_ca1["prediction"]
}

actual_values = reference["actual"].to_numpy()

comparison_rows = []

for model_name, predictions in model_predictions.items():
    metrics = calculate_metrics(actual_values, predictions)

    comparison_rows.append({
        "Fold": 2,
        "Store": "CA_1",
        "Model": model_name,
        **metrics
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df = comparison_df.sort_values(
    "WAPE (%)"
).reset_index(drop=True)

print("\nCA_1 FOLD 2 MODEL COMPARISON")
print("=" * 75)
print(comparison_df.round(4).to_string(index=False))

# Save results
comparison_path = RESULTS_DIR / "ca1_fold2_four_model_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)

print("\nCOMPARISON SAVED:", comparison_path)
print("BEST MODEL BY WAPE:", comparison_df.iloc[0]["Model"])

ROW COUNT VERIFICATION
Seasonal Naive: 85372
Croston SBA: 85372
LightGBM L2: 85372
LightGBM Tweedie: 85372

ALL FOUR MODELS HAVE MATCHING OBSERVATIONS.

CA_1 FOLD 2 MODEL COMPARISON
 Fold Store            Model    MAE   RMSE  WAPE (%)  Bias (%)
    2  CA_1 LightGBM Tweedie 1.1103 2.1918   75.6835   -2.0513
    2  CA_1      LightGBM L2 1.1265 2.2121   76.7848   -2.6274
    2  CA_1      Croston SBA 2.1076 4.4251  143.6583    1.3629
    2  CA_1   Seasonal Naive 2.1981 4.8005  149.8315   -4.5175

COMPARISON SAVED: D:\Retail-Demand\results\ca1_fold2_four_model_comparison.csv
BEST MODEL BY WAPE: LightGBM Tweedie


In [62]:
# ============================================================
# CELL 60: PREPARE CA_1 FOLD 3
# Purpose:
# 1. Release Fold 2 data from memory.
# 2. Prepare chronological Fold 3 partitions.
# 3. Apply consistent categorical and numerical data types.
# 4. Verify partition sizes and dates.
# ============================================================

import gc
import psutil
import pandas as pd
import numpy as np
import polars as pl

# Release Fold 2 data and models
del X_train, X_early_stop, X_outer
del y_train, y_early_stop, y_outer
del l2_model, tweedie_model
del l2_predictions, tweedie_predictions
del outer_metadata
del l2_prediction_output, tweedie_prediction_output
del seasonal_ca1, croston_ca1, l2_ca1, tweedie_ca1
del reference, comparison_df, model_predictions, comparison_rows

gc.collect()

print("AVAILABLE RAM:", round(
    psutil.virtual_memory().available / (1024**3), 2
), "GB")

# Fold 3 date boundaries
fold3_inner_train_end = pd.Timestamp("2016-02-28")
fold3_inner_validation_start = pd.Timestamp("2016-02-29")
fold3_outer_validation_start = pd.Timestamp("2016-03-28")
fold3_outer_validation_end = pd.Timestamp("2016-04-24")

# Load required features
required_columns = ["date", "sales"] + feature_columns

fold3_data = (
    pl.scan_parquet(CA1_FEATURE_PATH)
    .select(required_columns)
    .collect()
    .to_pandas()
)

fold3_data["date"] = pd.to_datetime(fold3_data["date"])

# Create partitions
train_data = fold3_data[
    fold3_data["date"] <= fold3_inner_train_end
].copy()

early_stop_data = fold3_data[
    (fold3_data["date"] >= fold3_inner_validation_start) &
    (fold3_data["date"] <= pd.Timestamp("2016-03-27"))
].copy()

outer_test_data = fold3_data[
    (fold3_data["date"] >= fold3_outer_validation_start) &
    (fold3_data["date"] <= fold3_outer_validation_end)
].copy()

# Separate features and target
X_train = train_data[feature_columns].copy()
y_train = train_data["sales"].to_numpy(dtype=np.float32)

X_early_stop = early_stop_data[feature_columns].copy()
y_early_stop = early_stop_data["sales"].to_numpy(dtype=np.float32)

X_outer = outer_test_data[feature_columns].copy()
y_outer = outer_test_data["sales"].to_numpy(dtype=np.float32)

# Categorical and numerical features
categorical_features = [
    "item_id", "event_name_1", "day_of_week", "month"
]

numeric_features = [
    col for col in feature_columns
    if col not in categorical_features
]

# Apply consistent categorical levels
category_types = {}

for col in categorical_features:
    categories = pd.Index(X_train[col].dropna().unique())
    category_types[col] = pd.CategoricalDtype(categories=categories)

    for dataset in [X_train, X_early_stop, X_outer]:
        dataset[col] = dataset[col].astype(category_types[col])

# Convert numerical features to float32
for col in numeric_features:
    for dataset in [X_train, X_early_stop, X_outer]:
        dataset[col] = pd.to_numeric(
            dataset[col], errors="coerce"
        ).astype(np.float32)

# Preserve outer validation metadata
outer_metadata = outer_test_data[
    ["item_id", "date", "sales"]
].copy()

# Verify
print("\nFOLD 3 PARTITION VERIFICATION")
print("=" * 50)
print("Training:", X_train.shape)
print("Early stopping:", X_early_stop.shape)
print("Outer validation:", X_outer.shape)

print("\nDATE RANGES")
print("Training ends:", train_data["date"].max().date())
print(
    "Early stopping:",
    early_stop_data["date"].min().date(),
    "to",
    early_stop_data["date"].max().date()
)
print(
    "Outer validation:",
    outer_test_data["date"].min().date(),
    "to",
    outer_test_data["date"].max().date()
)

assert len(X_early_stop) == 85372
assert len(X_outer) == 85372
assert len(X_train) == 5661930

print("\nFOLD 3 DATA PREPARATION VERIFIED!")
print("AVAILABLE RAM:", round(
    psutil.virtual_memory().available / (1024**3), 2
), "GB")

AVAILABLE RAM: 5.79 GB


ArrowMemoryError: malloc of size 45296000 failed

In [63]:
# ============================================================
# CELL 61: MEMORY CLEANUP AFTER FOLD 3 ERROR
# Purpose: Release partially loaded datasets and temporary objects.
# ============================================================

import gc
import psutil

for variable in [
    "fold3_data",
    "fold2_data",
    "train_data",
    "early_stop_data",
    "outer_test_data"
]:
    if variable in globals():
        del globals()[variable]

gc.collect()

memory = psutil.virtual_memory()

print("MEMORY CLEANUP COMPLETED")
print("Available RAM:",
      round(memory.available / (1024**3), 2), "GB")
print("RAM usage:", memory.percent, "%")

MEMORY CLEANUP COMPLETED
Available RAM: 4.25 GB
RAM usage: 73.1 %


In [64]:
# ============================================================
# CELL 62: MEMORY-EFFICIENT CA_1 FOLD 3 PREPARATION
# Purpose:
# 1. Filter partitions directly using Polars.
# 2. Convert only the required partitions to Pandas.
# 3. Apply consistent feature data types.
# ============================================================

# Fold 3 date boundaries
train_end = pd.Timestamp("2016-02-28")
early_start = pd.Timestamp("2016-02-29")
early_end = pd.Timestamp("2016-03-27")
outer_start = pd.Timestamp("2016-03-28")
outer_end = pd.Timestamp("2016-04-24")

required_columns = ["date", "sales"] + feature_columns

# Lazy scan of the feature file
fold3_scan = pl.scan_parquet(CA1_FEATURE_PATH)

# Load only the training partition
train_data = (
    fold3_scan
    .filter(pl.col("date") <= pl.lit(train_end.date()))
    .select(required_columns)
    .collect()
    .to_pandas()
)

gc.collect()

# Load only the early stopping partition
early_stop_data = (
    fold3_scan
    .filter(
        (pl.col("date") >= pl.lit(early_start.date())) &
        (pl.col("date") <= pl.lit(early_end.date()))
    )
    .select(required_columns)
    .collect()
    .to_pandas()
)

gc.collect()

# Load only the outer validation partition
outer_test_data = (
    fold3_scan
    .filter(
        (pl.col("date") >= pl.lit(outer_start.date())) &
        (pl.col("date") <= pl.lit(outer_end.date()))
    )
    .select(required_columns)
    .collect()
    .to_pandas()
)

# Convert dates
for df in [train_data, early_stop_data, outer_test_data]:
    df["date"] = pd.to_datetime(df["date"])

# Separate features and targets
X_train = train_data[feature_columns].copy()
y_train = train_data["sales"].to_numpy(dtype=np.float32)

X_early_stop = early_stop_data[feature_columns].copy()
y_early_stop = early_stop_data["sales"].to_numpy(dtype=np.float32)

X_outer = outer_test_data[feature_columns].copy()
y_outer = outer_test_data["sales"].to_numpy(dtype=np.float32)

# Categorical and numerical features
categorical_features = [
    "item_id", "event_name_1", "day_of_week", "month"
]

numeric_features = [
    col for col in feature_columns
    if col not in categorical_features
]

# Apply consistent categorical levels
category_types = {}

for col in categorical_features:
    categories = pd.Index(X_train[col].dropna().unique())
    category_types[col] = pd.CategoricalDtype(categories=categories)

    for dataset in [X_train, X_early_stop, X_outer]:
        dataset[col] = dataset[col].astype(category_types[col])

# Convert numeric features to float32
for col in numeric_features:
    for dataset in [X_train, X_early_stop, X_outer]:
        dataset[col] = pd.to_numeric(
            dataset[col], errors="coerce"
        ).astype(np.float32)

# Preserve outer validation metadata
outer_metadata = outer_test_data[
    ["item_id", "date", "sales"]
].copy()

# Verification
print("FOLD 3 PARTITION VERIFICATION")
print("=" * 50)

print("Training:", X_train.shape)
print("Early stopping:", X_early_stop.shape)
print("Outer validation:", X_outer.shape)

print("\nDATE RANGES")
print("Training ends:", train_data["date"].max().date())
print(
    "Early stopping:",
    early_stop_data["date"].min().date(),
    "to",
    early_stop_data["date"].max().date()
)
print(
    "Outer validation:",
    outer_test_data["date"].min().date(),
    "to",
    outer_test_data["date"].max().date()
)

print("\nAVAILABLE RAM:",
      round(psutil.virtual_memory().available / (1024**3), 2), "GB")

assert len(X_early_stop) == 85372
assert len(X_outer) == 85372

print("\nFOLD 3 DATA PREPARATION COMPLETED!")

FOLD 3 PARTITION VERIFICATION
Training: (5661993, 26)
Early stopping: (85372, 26)
Outer validation: (85372, 26)

DATE RANGES
Training ends: 2016-02-28
Early stopping: 2016-02-29 to 2016-03-27
Outer validation: 2016-03-28 to 2016-04-24

AVAILABLE RAM: 3.92 GB

FOLD 3 DATA PREPARATION COMPLETED!


In [65]:
# ============================================================
# CELL 63: RELEASE TEMPORARY FOLD 3 DATAFRAMES
# Purpose: Free memory before LightGBM training.
# ============================================================

import gc
import psutil

del train_data
del early_stop_data
del outer_test_data

gc.collect()

memory = psutil.virtual_memory()

print("TEMPORARY DATAFRAMES RELEASED")
print("Available RAM:",
      round(memory.available / (1024**3), 2), "GB")
print("RAM usage:", memory.percent, "%")

print("\nMODEL DATA SHAPES")
print("Training:", X_train.shape)
print("Early stopping:", X_early_stop.shape)
print("Outer validation:", X_outer.shape)

TEMPORARY DATAFRAMES RELEASED
Available RAM: 4.01 GB
RAM usage: 74.6 %

MODEL DATA SHAPES
Training: (5661993, 26)
Early stopping: (85372, 26)
Outer validation: (85372, 26)


In [66]:
# ============================================================
# CELL 64: TRAIN CA_1 FOLD 3 - MEMORY-CONSCIOUS LIGHTGBM L2
# ============================================================

import gc
import lightgbm as lgb
import numpy as np
import psutil

l2_model = lgb.LGBMRegressor(
    objective="regression_l2",
    n_estimators=700,
    learning_rate=0.05,
    num_leaves=31,
    max_bin=63,
    min_child_samples=100,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    n_jobs=2,
    random_state=42,
    verbosity=-1
)

print("AVAILABLE RAM BEFORE TRAINING:",
      round(psutil.virtual_memory().available / (1024**3), 2), "GB")

print("\nSTARTING CA_1 FOLD 3 LIGHTGBM L2 TRAINING...")

l2_model.fit(
    X_train,
    y_train,
    eval_set=[(X_early_stop, y_early_stop)],
    eval_metric="rmse",
    categorical_feature=categorical_features,
    callbacks=[
        lgb.early_stopping(50),
        lgb.log_evaluation(50)
    ]
)

print("\nBEST ITERATION:", l2_model.best_iteration_)

# Generate outer validation predictions
l2_predictions = np.maximum(
    l2_model.predict(
        X_outer,
        num_iteration=l2_model.best_iteration_
    ),
    0
)

# Calculate metrics
fold3_l2_metrics = calculate_metrics(y_outer, l2_predictions)

print("\nCA_1 FOLD 3 - LIGHTGBM L2 RESULTS")
for metric, value in fold3_l2_metrics.items():
    print(f"{metric}: {value:.4f}")

# Prepare prediction output
l2_prediction_output = outer_metadata.copy()

l2_prediction_output = l2_prediction_output.rename(
    columns={"sales": "actual", "date": "forecast_date"}
)

l2_prediction_output["prediction"] = l2_predictions
l2_prediction_output["fold"] = 3
l2_prediction_output["model"] = "LightGBM_L2"
l2_prediction_output["horizon"] = (
    l2_prediction_output.groupby("item_id").cumcount() + 1
)

# Save model and predictions
l2_model_path = MODEL_DIR / "lightgbm_l2_advanced_ca1_fold3.txt"
l2_model.booster_.save_model(str(l2_model_path))

l2_prediction_path = (
    PREDICTION_DIR / "lightgbm_l2_advanced_ca1_fold3.parquet"
)

pl.from_pandas(l2_prediction_output).write_parquet(
    l2_prediction_path
)

print("\nMODEL SAVED:", l2_model_path)
print("PREDICTIONS SAVED:", l2_prediction_path)
print("PREDICTION ROWS:", len(l2_prediction_output))

print("\nAVAILABLE RAM AFTER TRAINING:",
      round(psutil.virtual_memory().available / (1024**3), 2), "GB")

AVAILABLE RAM BEFORE TRAINING: 3.87 GB

STARTING CA_1 FOLD 3 LIGHTGBM L2 TRAINING...


D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[50]	valid_0's rmse: 2.22458	valid_0's l2: 4.94874
[100]	valid_0's rmse: 2.20791	valid_0's l2: 4.87486
[150]	valid_0's rmse: 2.20852	valid_0's l2: 4.87756
Early stopping, best iteration is:
[122]	valid_0's rmse: 2.20577	valid_0's l2: 4.86543

BEST ITERATION: 122

CA_1 FOLD 3 - LIGHTGBM L2 RESULTS
MAE: 1.1278
RMSE: 2.1621
WAPE (%): 75.2469
Bias (%): -1.3057

MODEL SAVED: D:\Retail-Demand\models\lightgbm_l2_advanced_ca1_fold3.txt
PREDICTIONS SAVED: D:\Retail-Demand\predictions\lightgbm_l2_advanced_ca1_fold3.parquet
PREDICTION ROWS: 85372

AVAILABLE RAM AFTER TRAINING: 5.48 GB


In [67]:
# ============================================================
# CELL 65: TRAIN CA_1 FOLD 3 - LIGHTGBM TWEEDIE
# ============================================================

tweedie_model = lgb.LGBMRegressor(
    objective="tweedie",
    tweedie_variance_power=1.2,
    n_estimators=700,
    learning_rate=0.05,
    num_leaves=31,
    max_bin=63,
    min_child_samples=100,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    n_jobs=2,
    random_state=42,
    verbosity=-1
)

print("AVAILABLE RAM BEFORE TRAINING:",
      round(psutil.virtual_memory().available / (1024**3), 2), "GB")

print("\nSTARTING CA_1 FOLD 3 LIGHTGBM TWEEDIE TRAINING...")

tweedie_model.fit(
    X_train,
    y_train,
    eval_set=[(X_early_stop, y_early_stop)],
    eval_metric="rmse",
    categorical_feature=categorical_features,
    callbacks=[
        lgb.early_stopping(50),
        lgb.log_evaluation(50)
    ]
)

print("\nBEST ITERATION:", tweedie_model.best_iteration_)

# Generate outer validation predictions
tweedie_predictions = np.maximum(
    tweedie_model.predict(
        X_outer,
        num_iteration=tweedie_model.best_iteration_
    ),
    0
)

# Calculate metrics
fold3_tweedie_metrics = calculate_metrics(
    y_outer, tweedie_predictions
)

print("\nCA_1 FOLD 3 - LIGHTGBM TWEEDIE RESULTS")
for metric, value in fold3_tweedie_metrics.items():
    print(f"{metric}: {value:.4f}")

# Prepare prediction output
tweedie_prediction_output = outer_metadata.copy()

tweedie_prediction_output = tweedie_prediction_output.rename(
    columns={"sales": "actual", "date": "forecast_date"}
)

tweedie_prediction_output["prediction"] = tweedie_predictions
tweedie_prediction_output["fold"] = 3
tweedie_prediction_output["model"] = "LightGBM_Tweedie"
tweedie_prediction_output["horizon"] = (
    tweedie_prediction_output.groupby("item_id").cumcount() + 1
)

# Save model and predictions
tweedie_model_path = (
    MODEL_DIR / "lightgbm_tweedie_advanced_ca1_fold3.txt"
)

tweedie_model.booster_.save_model(str(tweedie_model_path))

tweedie_prediction_path = (
    PREDICTION_DIR / "lightgbm_tweedie_advanced_ca1_fold3.parquet"
)

pl.from_pandas(tweedie_prediction_output).write_parquet(
    tweedie_prediction_path
)

print("\nMODEL SAVED:", tweedie_model_path)
print("PREDICTIONS SAVED:", tweedie_prediction_path)
print("PREDICTION ROWS:", len(tweedie_prediction_output))

print("\nAVAILABLE RAM AFTER TRAINING:",
      round(psutil.virtual_memory().available / (1024**3), 2), "GB")

AVAILABLE RAM BEFORE TRAINING: 5.0 GB

STARTING CA_1 FOLD 3 LIGHTGBM TWEEDIE TRAINING...


D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[50]	valid_0's rmse: 2.21849	valid_0's tweedie: 7.59856
[100]	valid_0's rmse: 2.18488	valid_0's tweedie: 7.58234
[150]	valid_0's rmse: 2.18479	valid_0's tweedie: 7.58018
Early stopping, best iteration is:
[135]	valid_0's rmse: 2.18385	valid_0's tweedie: 7.57996

BEST ITERATION: 135

CA_1 FOLD 3 - LIGHTGBM TWEEDIE RESULTS
MAE: 1.1207
RMSE: 2.1816
WAPE (%): 74.7761
Bias (%): -0.7945

MODEL SAVED: D:\Retail-Demand\models\lightgbm_tweedie_advanced_ca1_fold3.txt
PREDICTIONS SAVED: D:\Retail-Demand\predictions\lightgbm_tweedie_advanced_ca1_fold3.parquet
PREDICTION ROWS: 85372

AVAILABLE RAM AFTER TRAINING: 5.35 GB


In [68]:
# ============================================================
# CELL 66: CA_1 FOLD 3 - FOUR MODEL COMPARISON
# ============================================================

# Load Fold 3 baseline predictions for CA_1
seasonal_ca1 = (
    pl.read_parquet(PREDICTION_DIR / "seasonal_naive_fold_3.parquet")
    .filter(pl.col("store_id") == "CA_1")
    .to_pandas()
)

croston_ca1 = (
    pl.read_parquet(PREDICTION_DIR / "croston_sba_fold_3.parquet")
    .filter(pl.col("store_id") == "CA_1")
    .to_pandas()
)

seasonal_ca1 = seasonal_ca1.rename(columns={"forecast": "prediction"})
croston_ca1 = croston_ca1.rename(columns={"forecast": "prediction"})

l2_ca1 = l2_prediction_output.copy()
tweedie_ca1 = tweedie_prediction_output.copy()

for df in [seasonal_ca1, croston_ca1, l2_ca1, tweedie_ca1]:
    df["forecast_date"] = pd.to_datetime(df["forecast_date"])

# Verify row counts
print("ROW COUNT VERIFICATION")

for name, df in [
    ("Seasonal Naive", seasonal_ca1),
    ("Croston SBA", croston_ca1),
    ("LightGBM L2", l2_ca1),
    ("LightGBM Tweedie", tweedie_ca1)
]:
    print(f"{name}: {len(df)}")

assert all(
    len(df) == 85372
    for df in [seasonal_ca1, croston_ca1, l2_ca1, tweedie_ca1]
)

# Verify identical item/date keys and actual values
key_columns = ["item_id", "forecast_date"]

reference = (
    seasonal_ca1[key_columns + ["actual"]]
    .sort_values(key_columns)
    .reset_index(drop=True)
)

for name, df in [
    ("Croston SBA", croston_ca1),
    ("LightGBM L2", l2_ca1),
    ("LightGBM Tweedie", tweedie_ca1)
]:
    check = (
        df[key_columns + ["actual"]]
        .sort_values(key_columns)
        .reset_index(drop=True)
    )

    assert reference[key_columns].equals(check[key_columns])
    assert np.array_equal(
        reference["actual"].to_numpy(),
        check["actual"].to_numpy()
    )

print("\nALL FOUR MODELS HAVE MATCHING OBSERVATIONS.")

# Calculate metrics
model_predictions = {
    "Seasonal Naive": seasonal_ca1["prediction"],
    "Croston SBA": croston_ca1["prediction"],
    "LightGBM L2": l2_ca1["prediction"],
    "LightGBM Tweedie": tweedie_ca1["prediction"]
}

actual_values = reference["actual"].to_numpy()

comparison_rows = []

for model_name, predictions in model_predictions.items():
    metrics = calculate_metrics(actual_values, predictions)

    comparison_rows.append({
        "Fold": 3,
        "Store": "CA_1",
        "Model": model_name,
        **metrics
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df = comparison_df.sort_values(
    "WAPE (%)"
).reset_index(drop=True)

print("\nCA_1 FOLD 3 MODEL COMPARISON")
print("=" * 75)
print(comparison_df.round(4).to_string(index=False))

# Save results
comparison_path = RESULTS_DIR / "ca1_fold3_four_model_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)

print("\nCOMPARISON SAVED:", comparison_path)
print("BEST MODEL BY WAPE:", comparison_df.iloc[0]["Model"])

ROW COUNT VERIFICATION
Seasonal Naive: 85372
Croston SBA: 85372
LightGBM L2: 85372
LightGBM Tweedie: 85372

ALL FOUR MODELS HAVE MATCHING OBSERVATIONS.

CA_1 FOLD 3 MODEL COMPARISON
 Fold Store            Model    MAE   RMSE  WAPE (%)  Bias (%)
    3  CA_1 LightGBM Tweedie 1.1207 2.1816   74.7761   -0.7945
    3  CA_1      LightGBM L2 1.1278 2.1621   75.2469   -1.3057
    3  CA_1      Croston SBA 2.1431 4.4499  142.9914    0.3217
    3  CA_1   Seasonal Naive 2.2854 4.9627  152.4825   -1.6217

COMPARISON SAVED: D:\Retail-Demand\results\ca1_fold3_four_model_comparison.csv
BEST MODEL BY WAPE: LightGBM Tweedie


In [69]:
# ============================================================
# CELL 67: CONSOLIDATE CA_1 THREE-FOLD RESULTS
# Purpose:
# 1. Combine the three fold comparison files.
# 2. Calculate average metrics for each model.
# 3. Identify the best model by average WAPE.
# 4. Save consolidated results.
# ============================================================

import pandas as pd
from pathlib import Path

fold_files = [
    RESULTS_DIR / "ca1_fold1_four_model_comparison.csv",
    RESULTS_DIR / "ca1_fold2_four_model_comparison.csv",
    RESULTS_DIR / "ca1_fold3_four_model_comparison.csv"
]

for path in fold_files:
    assert path.exists(), f"Missing comparison file: {path}"

ca1_all_folds = pd.concat(
    [pd.read_csv(path) for path in fold_files],
    ignore_index=True
)

print("FOLD RESULTS")
print(ca1_all_folds.round(4).to_string(index=False))

# Calculate average metrics across folds
ca1_summary = (
    ca1_all_folds
    .groupby(["Store", "Model"], as_index=False)
    [["MAE", "RMSE", "WAPE (%)", "Bias (%)"]]
    .mean()
    .sort_values("WAPE (%)")
    .reset_index(drop=True)
)

print("\nCA_1 THREE-FOLD AVERAGE")
print("=" * 65)
print(ca1_summary.round(4).to_string(index=False))

# Save consolidated outputs
fold_output_path = RESULTS_DIR / "ca1_all_fold_comparisons.csv"
summary_output_path = RESULTS_DIR / "ca1_three_fold_model_summary.csv"

ca1_all_folds.to_csv(fold_output_path, index=False)
ca1_summary.to_csv(summary_output_path, index=False)

print("\nSAVED:")
print(fold_output_path)
print(summary_output_path)

print("\nBEST MODEL BY AVERAGE WAPE:")
print(ca1_summary.iloc[0]["Model"])

FOLD RESULTS
           Model    MAE   RMSE  WAPE (%)  Bias (%)  Fold Store
LightGBM Tweedie 1.0999 2.1974   76.4282   -3.1261   NaN   NaN
     LightGBM L2 1.1100 2.2052   77.1289   -2.6981   NaN   NaN
     Croston SBA 2.0681 4.2675  143.7066   -0.7856   NaN   NaN
  Seasonal Naive 2.1600 4.6648  150.0936   -6.9104   NaN   NaN
LightGBM Tweedie 1.1103 2.1918   75.6835   -2.0513   2.0  CA_1
     LightGBM L2 1.1265 2.2121   76.7848   -2.6274   2.0  CA_1
     Croston SBA 2.1076 4.4251  143.6583    1.3629   2.0  CA_1
  Seasonal Naive 2.1981 4.8005  149.8315   -4.5175   2.0  CA_1
LightGBM Tweedie 1.1207 2.1816   74.7761   -0.7945   3.0  CA_1
     LightGBM L2 1.1278 2.1621   75.2469   -1.3057   3.0  CA_1
     Croston SBA 2.1431 4.4499  142.9914    0.3217   3.0  CA_1
  Seasonal Naive 2.2854 4.9627  152.4825   -1.6217   3.0  CA_1

CA_1 THREE-FOLD AVERAGE
Store            Model    MAE   RMSE  WAPE (%)  Bias (%)
 CA_1 LightGBM Tweedie 1.1155 2.1867   75.2298   -1.4229
 CA_1      LightGBM L2 1.1271

In [70]:
# ============================================================
# CELL 68: FIX CA_1 FOLD 1 METADATA
# ============================================================

fold1_path = RESULTS_DIR / "ca1_fold1_four_model_comparison.csv"

fold1_df = pd.read_csv(fold1_path)

fold1_df["Fold"] = 1
fold1_df["Store"] = "CA_1"

fold1_df = fold1_df[
    ["Fold", "Store", "Model", "MAE", "RMSE", "WAPE (%)", "Bias (%)"]
]

fold1_df.to_csv(fold1_path, index=False)

# Recombine all three folds
fold_files = [
    fold1_path,
    RESULTS_DIR / "ca1_fold2_four_model_comparison.csv",
    RESULTS_DIR / "ca1_fold3_four_model_comparison.csv"
]

ca1_all_folds = pd.concat(
    [pd.read_csv(path) for path in fold_files],
    ignore_index=True
)

ca1_summary = (
    ca1_all_folds
    .groupby(["Store", "Model"], as_index=False)
    [["MAE", "RMSE", "WAPE (%)", "Bias (%)"]]
    .mean()
    .sort_values("WAPE (%)")
    .reset_index(drop=True)
)

ca1_all_folds.to_csv(
    RESULTS_DIR / "ca1_all_fold_comparisons.csv",
    index=False
)

ca1_summary.to_csv(
    RESULTS_DIR / "ca1_three_fold_model_summary.csv",
    index=False
)

print("METADATA VERIFICATION")
print(ca1_all_folds[["Fold", "Store", "Model"]].to_string(index=False))

print("\nFINAL CA_1 SUMMARY")
print(ca1_summary.round(4).to_string(index=False))

assert ca1_all_folds["Fold"].notna().all()
assert ca1_all_folds["Store"].notna().all()

print("\nCA_1 CONSOLIDATED RESULTS VERIFIED!")

METADATA VERIFICATION
 Fold Store            Model
    1  CA_1 LightGBM Tweedie
    1  CA_1      LightGBM L2
    1  CA_1      Croston SBA
    1  CA_1   Seasonal Naive
    2  CA_1 LightGBM Tweedie
    2  CA_1      LightGBM L2
    2  CA_1      Croston SBA
    2  CA_1   Seasonal Naive
    3  CA_1 LightGBM Tweedie
    3  CA_1      LightGBM L2
    3  CA_1      Croston SBA
    3  CA_1   Seasonal Naive

FINAL CA_1 SUMMARY
Store            Model    MAE   RMSE  WAPE (%)  Bias (%)
 CA_1 LightGBM Tweedie 1.1103 2.1903   75.6293   -1.9906
 CA_1      LightGBM L2 1.1214 2.1931   76.3869   -2.2104
 CA_1      Croston SBA 2.1063 4.3809  143.4521    0.2997
 CA_1   Seasonal Naive 2.2145 4.8093  150.8025   -4.3499

CA_1 CONSOLIDATED RESULTS VERIFIED!


In [73]:

# STEP 1: Configure feature file paths for the remaining nine stores
# Purpose: Map each store to its previously generated advanced feature dataset.

from pathlib import Path

FEATURE_DIR = Path("data/processed")
MODEL_DIR = Path("models")
PREDICTION_DIR = Path("predictions")
RESULTS_DIR = Path("results")

FEATURE_FILES = {
    "CA_2": FEATURE_DIR / "ca2_advanced_features.parquet",
    "CA_3": FEATURE_DIR / "ca_3_advanced_features.parquet",
    "CA_4": FEATURE_DIR / "ca_4_advanced_features.parquet",
    "TX_1": FEATURE_DIR / "tx_1_advanced_features.parquet",
    "TX_2": FEATURE_DIR / "tx_2_advanced_features.parquet",
    "TX_3": FEATURE_DIR / "tx_3_advanced_features.parquet",
    "WI_1": FEATURE_DIR / "wi_1_advanced_features.parquet",
    "WI_2": FEATURE_DIR / "wi_2_advanced_features.parquet",
    "WI_3": FEATURE_DIR / "wi_3_advanced_features.parquet"
}

# Create output directories if they do not already exist
for directory in [MODEL_DIR, PREDICTION_DIR, RESULTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("MULTI-STORE TRAINING CONFIGURATION")
print("Stores configured:", len(FEATURE_FILES))

for store, path in FEATURE_FILES.items():
    print(f"{store}: {path}")


MULTI-STORE TRAINING CONFIGURATION
Stores configured: 9
CA_2: data\processed\ca2_advanced_features.parquet
CA_3: data\processed\ca_3_advanced_features.parquet
CA_4: data\processed\ca_4_advanced_features.parquet
TX_1: data\processed\tx_1_advanced_features.parquet
TX_2: data\processed\tx_2_advanced_features.parquet
TX_3: data\processed\tx_3_advanced_features.parquet
WI_1: data\processed\wi_1_advanced_features.parquet
WI_2: data\processed\wi_2_advanced_features.parquet
WI_3: data\processed\wi_3_advanced_features.parquet


In [74]:

# STEP 2: Configure the multi-store forecasting experiments
# Purpose:
# 1. Define the 26 model input features.
# 2. Define categorical features.
# 3. Configure the three rolling-origin folds.
# 4. Set consistent memory-conscious LightGBM parameters.

feature_columns = [
    "item_id", "event_name_1", "day_of_week", "month", "snap_flag",
    "lag_28", "lag_35", "lag_42", "lag_56", "lag_364",
    "rolling_mean_7", "rolling_mean_28", "rolling_mean_56",
    "rolling_std_7", "rolling_std_28",
    "day_of_week_sin", "day_of_week_cos",
    "month_sin", "month_cos",
    "ema_28", "zero_sales_run",
    "price_lag_28", "price_lag_56",
    "price_momentum", "historical_max_price_56",
    "discount_ratio"
]

categorical_features = [
    "item_id", "event_name_1", "day_of_week", "month"
]

numeric_features = [
    col for col in feature_columns
    if col not in categorical_features
]

# Rolling-origin split dates
FOLD_CONFIG = {
    1: {
        "train_end": "2016-01-03",
        "early_start": "2016-01-04",
        "early_end": "2016-01-31",
        "valid_start": "2016-02-01",
        "valid_end": "2016-02-28"
    },
    2: {
        "train_end": "2016-01-31",
        "early_start": "2016-02-01",
        "early_end": "2016-02-28",
        "valid_start": "2016-02-29",
        "valid_end": "2016-03-27"
    },
    3: {
        "train_end": "2016-02-28",
        "early_start": "2016-02-29",
        "early_end": "2016-03-27",
        "valid_start": "2016-03-28",
        "valid_end": "2016-04-24"
    }
}

# Consistent memory-conscious model settings
COMMON_PARAMS = {
    "n_estimators": 700,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_bin": 63,
    "min_child_samples": 100,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "n_jobs": 2,
    "random_state": 42,
    "verbosity": -1
}

print("FEATURE CONFIGURATION")
print("Total features:", len(feature_columns))
print("Categorical features:", len(categorical_features))
print("Numeric features:", len(numeric_features))

print("\nROLLING-ORIGIN FOLDS")
for fold, config in FOLD_CONFIG.items():
    print(f"Fold {fold}:")
    print("  Training ends:", config["train_end"])
    print("  Early stopping:", config["early_start"], "to", config["early_end"])
    print("  Validation:", config["valid_start"], "to", config["valid_end"])

print("\nMODEL CONFIGURATION")
print(COMMON_PARAMS)


FEATURE CONFIGURATION
Total features: 26
Categorical features: 4
Numeric features: 22

ROLLING-ORIGIN FOLDS
Fold 1:
  Training ends: 2016-01-03
  Early stopping: 2016-01-04 to 2016-01-31
  Validation: 2016-02-01 to 2016-02-28
Fold 2:
  Training ends: 2016-01-31
  Early stopping: 2016-02-01 to 2016-02-28
  Validation: 2016-02-29 to 2016-03-27
Fold 3:
  Training ends: 2016-02-28
  Early stopping: 2016-02-29 to 2016-03-27
  Validation: 2016-03-28 to 2016-04-24

MODEL CONFIGURATION
{'n_estimators': 700, 'learning_rate': 0.05, 'num_leaves': 31, 'max_bin': 63, 'min_child_samples': 100, 'subsample': 0.8, 'subsample_freq': 1, 'colsample_bytree': 0.8, 'reg_lambda': 1.0, 'n_jobs': 2, 'random_state': 42, 'verbosity': -1}


In [75]:

# STEP 3: Verify CA_2 Fold 1 data partitions
# Purpose: Load the CA_2 feature dataset using Polars and verify
# training, early-stopping, and validation row counts before model training.

import polars as pl

store = "CA_2"
fold = 1

feature_path = FEATURE_FILES[store]
config = FOLD_CONFIG[fold]

# Scan the Parquet file without loading the entire dataset into memory
store_data = pl.scan_parquet(feature_path)

# Convert date boundaries to Polars date expressions
train_end = pl.lit(config["train_end"]).str.to_date()
early_start = pl.lit(config["early_start"]).str.to_date()
early_end = pl.lit(config["early_end"]).str.to_date()
valid_start = pl.lit(config["valid_start"]).str.to_date()
valid_end = pl.lit(config["valid_end"]).str.to_date()

# Count rows in each chronological partition
partition_counts = store_data.select([
    (pl.col("date") <= train_end).sum().alias("Training"),
    (
        (pl.col("date") >= early_start) &
        (pl.col("date") <= early_end)
    ).sum().alias("Early_Stopping"),
    (
        (pl.col("date") >= valid_start) &
        (pl.col("date") <= valid_end)
    ).sum().alias("Validation")
]).collect()

print(f"{store} - FOLD {fold} PARTITION VERIFICATION")
print(partition_counts)

print("\nDATE RANGE")
print(
    store_data.select([
        pl.col("date").min().alias("First Date"),
        pl.col("date").max().alias("Last Date")
    ]).collect()
)


FileNotFoundError: The system cannot find the path specified. (os error 3): data/processed/ca2_advanced_features.parquet

In [76]:

# STEP 3A: Diagnose the feature file path
# Purpose: Check the notebook working directory and locate
# the existing CA_2 advanced feature dataset.

from pathlib import Path

print("CURRENT WORKING DIRECTORY:")
print(Path.cwd())

print("\nEXPECTED PROJECT DIRECTORY:")
print(Path("D:/Retail-Demand").resolve())

print("\nCHECKING POSSIBLE FILE LOCATIONS:")

possible_paths = [
    Path("D:/Retail-Demand/data/processed/ca2_advanced_features.parquet"),
    Path("D:/Retail-Demand/data/processed/ca_2_advanced_features.parquet"),
    Path("data/processed/ca2_advanced_features.parquet"),
    Path("data/processed/ca_2_advanced_features.parquet")
]

for path in possible_paths:
    print(f"{path} -> {'FOUND' if path.exists() else 'NOT FOUND'}")


CURRENT WORKING DIRECTORY:
D:\Retail-Demand\notebooks

EXPECTED PROJECT DIRECTORY:
D:\Retail-Demand

CHECKING POSSIBLE FILE LOCATIONS:
D:\Retail-Demand\data\processed\ca2_advanced_features.parquet -> FOUND
D:\Retail-Demand\data\processed\ca_2_advanced_features.parquet -> NOT FOUND
data\processed\ca2_advanced_features.parquet -> NOT FOUND
data\processed\ca_2_advanced_features.parquet -> NOT FOUND


In [77]:

# STEP 3B: Correct project paths
# Purpose: Set the project root explicitly so all notebooks
# can access data, models, predictions, and results correctly.

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

FEATURE_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
PREDICTION_DIR = PROJECT_ROOT / "predictions"
RESULTS_DIR = PROJECT_ROOT / "results"

FEATURE_FILES = {
    "CA_2": FEATURE_DIR / "ca2_advanced_features.parquet",
    "CA_3": FEATURE_DIR / "ca_3_advanced_features.parquet",
    "CA_4": FEATURE_DIR / "ca_4_advanced_features.parquet",
    "TX_1": FEATURE_DIR / "tx_1_advanced_features.parquet",
    "TX_2": FEATURE_DIR / "tx_2_advanced_features.parquet",
    "TX_3": FEATURE_DIR / "tx_3_advanced_features.parquet",
    "WI_1": FEATURE_DIR / "wi_1_advanced_features.parquet",
    "WI_2": FEATURE_DIR / "wi_2_advanced_features.parquet",
    "WI_3": FEATURE_DIR / "wi_3_advanced_features.parquet"
}

for directory in [MODEL_DIR, PREDICTION_DIR, RESULTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT ROOT:", PROJECT_ROOT)
print("\nFEATURE FILE CHECK:")

for store, path in FEATURE_FILES.items():
    print(f"{store}: {'FOUND' if path.exists() else 'MISSING'}")


PROJECT ROOT: D:\Retail-Demand

FEATURE FILE CHECK:
CA_2: FOUND
CA_3: FOUND
CA_4: FOUND
TX_1: FOUND
TX_2: FOUND
TX_3: FOUND
WI_1: FOUND
WI_2: FOUND
WI_3: FOUND


In [78]:

# STEP 4: Verify CA_2 Fold 1 chronological data partitions
# Purpose: Confirm the training, early-stopping, and validation
# periods before loading data into memory for model training.

import polars as pl

store = "CA_2"
fold = 1

feature_path = FEATURE_FILES[store]
config = FOLD_CONFIG[fold]

# Scan the existing feature file using its verified absolute path
store_data = pl.scan_parquet(feature_path)

# Define the three chronological partitions
partition_counts = store_data.select([
    (pl.col("date") <= pl.lit(config["train_end"]).str.to_date())
        .sum().alias("Training"),

    (
        (pl.col("date") >= pl.lit(config["early_start"]).str.to_date()) &
        (pl.col("date") <= pl.lit(config["early_end"]).str.to_date())
    ).sum().alias("Early_Stopping"),

    (
        (pl.col("date") >= pl.lit(config["valid_start"]).str.to_date()) &
        (pl.col("date") <= pl.lit(config["valid_end"]).str.to_date())
    ).sum().alias("Validation")
]).collect()

print(f"{store} - FOLD {fold} PARTITION VERIFICATION")
print(partition_counts)

print("\nDATASET DATE RANGE:")
print(
    store_data.select([
        pl.col("date").min().alias("First Date"),
        pl.col("date").max().alias("Last Date")
    ]).collect()
)


CA_2 - FOLD 1 PARTITION VERIFICATION
shape: (1, 3)
┌──────────┬────────────────┬────────────┐
│ Training ┆ Early_Stopping ┆ Validation │
│ ---      ┆ ---            ┆ ---        │
│ u32      ┆ u32            ┆ u32        │
╞══════════╪════════════════╪════════════╡
│ 5491249  ┆ 85372          ┆ 85372      │
└──────────┴────────────────┴────────────┘

DATASET DATE RANGE:
shape: (1, 2)
┌────────────┬────────────┐
│ First Date ┆ Last Date  │
│ ---        ┆ ---        │
│ date       ┆ date       │
╞════════════╪════════════╡
│ 2011-01-29 ┆ 2016-04-24 │
└────────────┴────────────┘


In [79]:

# STEP 5: Check available system memory
# Purpose: Ensure sufficient RAM is available before loading
# the CA_2 Fold 1 training and validation partitions.

import psutil

memory = psutil.virtual_memory()

print("SYSTEM MEMORY STATUS")
print(f"Total RAM:     {memory.total / (1024**3):.2f} GB")
print(f"Available RAM: {memory.available / (1024**3):.2f} GB")
print(f"Used RAM:      {memory.percent:.1f}%")

print("\nCA_2 FOLD 1 DATA")
print("Training rows: 5,491,249")
print("Early-stop rows: 85,372")
print("Validation rows: 85,372")

if memory.available / (1024**3) >= 4:
    print("\nSTATUS: Sufficient memory to proceed cautiously.")
else:
    print("\nSTATUS: Low available memory. We should free memory first.")


SYSTEM MEMORY STATUS
Total RAM:     15.82 GB
Available RAM: 6.61 GB
Used RAM:      58.2%

CA_2 FOLD 1 DATA
Training rows: 5,491,249
Early-stop rows: 85,372
Validation rows: 85,372

STATUS: Sufficient memory to proceed cautiously.


In [81]:

# STEP 6: Load CA_2 Fold 1 training partition
# Purpose: Load the required features safely.
# Convert categorical columns to String in Polars, then to
# Pandas category after conversion.

import gc
import pandas as pd
import polars as pl
import psutil

store = "CA_2"
fold = 1
config = FOLD_CONFIG[fold]

required_columns = feature_columns + ["sales"]

# Build expressions with compatible data types
select_expressions = []

for col in required_columns:
    if col in categorical_features:
        select_expressions.append(pl.col(col).cast(pl.String))
    elif col in numeric_features or col == "sales":
        select_expressions.append(pl.col(col).cast(pl.Float32))
    else:
        select_expressions.append(pl.col(col))

# Filter in Polars before collecting
train_pl = (
    pl.scan_parquet(FEATURE_FILES[store])
    .filter(
        pl.col("date") <= pl.lit(config["train_end"]).str.to_date()
    )
    .select(select_expressions)
    .collect()
)

print("POLARS TRAINING DATA LOADED")
print("Shape:", train_pl.shape)

# Convert to Pandas
train_pd = train_pl.to_pandas()
del train_pl
gc.collect()

# Convert categorical columns to Pandas category
for col in categorical_features:
    train_pd[col] = train_pd[col].astype("category")

# Separate features and target
X_train = train_pd[feature_columns].copy()
y_train = train_pd["sales"].copy()

del train_pd
gc.collect()

print("\nPANDAS TRAINING DATA")
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("\nCATEGORICAL DATA TYPES")
print(X_train[categorical_features].dtypes)

memory = psutil.virtual_memory()
print(f"\nAvailable RAM: {memory.available / (1024**3):.2f} GB")


POLARS TRAINING DATA LOADED
Shape: (5491249, 27)

PANDAS TRAINING DATA
X_train shape: (5491249, 26)
y_train shape: (5491249,)

CATEGORICAL DATA TYPES
item_id         category
event_name_1    category
day_of_week     category
month           category
dtype: object

Available RAM: 4.82 GB


In [82]:

# STEP 7: Load CA_2 Fold 1 early-stopping and validation data
# Purpose:
# Load the two 28-day partitions and align categorical levels
# with the training dataset.

# Read only the dates needed for early stopping and validation
eval_pl = (
    pl.scan_parquet(FEATURE_FILES[store])
    .filter(
        (pl.col("date") >= pl.lit(config["early_start"]).str.to_date()) &
        (pl.col("date") <= pl.lit(config["valid_end"]).str.to_date())
    )
    .select(
        [pl.col(c).cast(pl.String) for c in categorical_features]
        + [pl.col(c).cast(pl.Float32) for c in numeric_features]
        + [
            pl.col("sales").cast(pl.Float32),
            pl.col("date")
        ]
    )
    .collect()
)

print("EVALUATION DATA LOADED:", eval_pl.shape)

eval_pd = eval_pl.to_pandas()
del eval_pl
gc.collect()

# Align categorical levels with the training dataset
for col in categorical_features:
    categories = X_train[col].cat.categories
    eval_pd[col] = pd.Categorical(
        eval_pd[col],
        categories=categories
    )

# Separate early-stopping and outer-validation periods
early_mask = (
    eval_pd["date"] >= pd.Timestamp(config["early_start"])
) & (
    eval_pd["date"] <= pd.Timestamp(config["early_end"])
)

valid_mask = (
    eval_pd["date"] >= pd.Timestamp(config["valid_start"])
) & (
    eval_pd["date"] <= pd.Timestamp(config["valid_end"])
)

X_early_stop = eval_pd.loc[early_mask, feature_columns].copy()
y_early_stop = eval_pd.loc[early_mask, "sales"].copy()

X_valid = eval_pd.loc[valid_mask, feature_columns].copy()
y_valid = eval_pd.loc[valid_mask, "sales"].copy()

del eval_pd, early_mask, valid_mask
gc.collect()

print("\nPARTITION SHAPES")
print("Training:", X_train.shape, y_train.shape)
print("Early stopping:", X_early_stop.shape, y_early_stop.shape)
print("Validation:", X_valid.shape, y_valid.shape)

print("\nTARGET CHECK")
print("Negative training targets:", (y_train < 0).sum())
print("Negative validation targets:", (y_valid < 0).sum())

memory = psutil.virtual_memory()
print(f"\nAvailable RAM: {memory.available / (1024**3):.2f} GB")


EVALUATION DATA LOADED: (170744, 28)

PARTITION SHAPES
Training: (5491249, 26) (5491249,)
Early stopping: (85372, 26) (85372,)
Validation: (85372, 26) (85372,)

TARGET CHECK
Negative training targets: 0
Negative validation targets: 0

Available RAM: 7.54 GB


In [83]:

# STEP 8: Train LightGBM L2 for CA_2 Fold 1
# Purpose:
# Train a regression model using squared-error loss,
# select the best iteration using early stopping,
# evaluate on the outer validation partition, and save the model.

import lightgbm as lgb
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

l2_model = lgb.LGBMRegressor(
    objective="regression_l2",
    **COMMON_PARAMS
)

l2_model.fit(
    X_train,
    y_train,
    eval_set=[(X_early_stop, y_early_stop)],
    eval_metric="rmse",
    categorical_feature=categorical_features,
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=50)
    ]
)

# Predict on the untouched outer validation partition
ca2_fold1_l2_pred = np.maximum(
    l2_model.predict(X_valid),
    0
)

# Calculate evaluation metrics
actual = y_valid.to_numpy()
predicted = ca2_fold1_l2_pred

mae = mean_absolute_error(actual, predicted)
rmse = np.sqrt(mean_squared_error(actual, predicted))
wape = np.sum(np.abs(actual - predicted)) / np.sum(actual) * 100
bias = np.sum(predicted - actual) / np.sum(actual) * 100

print("\nCA_2 FOLD 1 - LIGHTGBM L2 RESULTS")
print("Best iteration:", l2_model.best_iteration_)
print(f"MAE:       {mae:.4f}")
print(f"RMSE:      {rmse:.4f}")
print(f"WAPE:      {wape:.4f}%")
print(f"Bias:      {bias:.4f}%")

# Save trained model
l2_model_path = MODEL_DIR / "lightgbm_l2_advanced_ca_2_fold1.txt"
l2_model.booster_.save_model(str(l2_model_path))

print("\nMODEL SAVED:", l2_model_path)


D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[50]	valid_0's rmse: 2.05596	valid_0's l2: 4.22697
[100]	valid_0's rmse: 2.00495	valid_0's l2: 4.01981
[150]	valid_0's rmse: 1.98807	valid_0's l2: 3.95244
[200]	valid_0's rmse: 1.98028	valid_0's l2: 3.92152
[250]	valid_0's rmse: 1.97802	valid_0's l2: 3.91255
[300]	valid_0's rmse: 1.97561	valid_0's l2: 3.90302
[350]	valid_0's rmse: 1.97367	valid_0's l2: 3.89538
[400]	valid_0's rmse: 1.97206	valid_0's l2: 3.88902
[450]	valid_0's rmse: 1.96888	valid_0's l2: 3.87648
[500]	valid_0's rmse: 1.96772	valid_0's l2: 3.87191
Early stopping, best iteration is:
[485]	valid_0's rmse: 1.96747	valid_0's l2: 3.87092

CA_2 FOLD 1 - LIGHTGBM L2 RESULTS
Best iteration: 485
MAE:       1.0857
RMSE:      2.1274
WAPE:      78.7475%
Bias:      -8.9192%

MODEL SAVED: D:\Retail-Demand\models\lightgbm_l2_advanced_ca_2_fold1.txt


In [84]:

# STEP 9: Train LightGBM Tweedie for CA_2 Fold 1
# Purpose:
# Train using Tweedie loss, apply early stopping,
# evaluate on the outer validation partition, and save the model.

tweedie_model = lgb.LGBMRegressor(
    objective="tweedie",
    tweedie_variance_power=1.2,
    **COMMON_PARAMS
)

tweedie_model.fit(
    X_train,
    y_train,
    eval_set=[(X_early_stop, y_early_stop)],
    eval_metric="rmse",
    categorical_feature=categorical_features,
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=50)
    ]
)

# Predict on the untouched outer validation partition
ca2_fold1_tweedie_pred = np.maximum(
    tweedie_model.predict(X_valid),
    0
)

# Calculate evaluation metrics
actual = y_valid.to_numpy()
predicted = ca2_fold1_tweedie_pred

tweedie_mae = mean_absolute_error(actual, predicted)
tweedie_rmse = np.sqrt(mean_squared_error(actual, predicted))
tweedie_wape = (
    np.sum(np.abs(actual - predicted)) / np.sum(actual) * 100
)
tweedie_bias = (
    np.sum(predicted - actual) / np.sum(actual) * 100
)

print("\nCA_2 FOLD 1 - LIGHTGBM TWEEDIE RESULTS")
print("Best iteration:", tweedie_model.best_iteration_)
print(f"MAE:       {tweedie_mae:.4f}")
print(f"RMSE:      {tweedie_rmse:.4f}")
print(f"WAPE:      {tweedie_wape:.4f}%")
print(f"Bias:      {tweedie_bias:.4f}%")

# Save trained model
tweedie_model_path = (
    MODEL_DIR / "lightgbm_tweedie_advanced_ca_2_fold1.txt"
)
tweedie_model.booster_.save_model(str(tweedie_model_path))

print("\nMODEL SAVED:", tweedie_model_path)


D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[50]	valid_0's rmse: 2.10235	valid_0's tweedie: 7.17603
[100]	valid_0's rmse: 2.01156	valid_0's tweedie: 7.15917
[150]	valid_0's rmse: 1.97452	valid_0's tweedie: 7.15729
[200]	valid_0's rmse: 1.9521	valid_0's tweedie: 7.15637
[250]	valid_0's rmse: 1.93353	valid_0's tweedie: 7.15578
[300]	valid_0's rmse: 1.93323	valid_0's tweedie: 7.15613
Early stopping, best iteration is:
[284]	valid_0's rmse: 1.93111	valid_0's tweedie: 7.15529

CA_2 FOLD 1 - LIGHTGBM TWEEDIE RESULTS
Best iteration: 284
MAE:       1.0590
RMSE:      2.0271
WAPE:      76.8108%
Bias:      -9.2443%

MODEL SAVED: D:\Retail-Demand\models\lightgbm_tweedie_advanced_ca_2_fold1.txt


In [85]:

# STEP 10: Save CA_2 Fold 1 LightGBM predictions
# Purpose: Preserve actual sales, forecast dates, and predictions
# for both models so they can be compared with the baselines.

# Reload validation metadata in the same chronological order
validation_metadata = (
    pl.scan_parquet(FEATURE_FILES["CA_2"])
    .filter(
        (pl.col("date") >= pl.lit(config["valid_start"]).str.to_date()) &
        (pl.col("date") <= pl.lit(config["valid_end"]).str.to_date())
    )
    .select(["item_id", "date", "sales"])
    .collect()
)

metadata_pd = validation_metadata.to_pandas()

assert len(metadata_pd) == len(ca2_fold1_l2_pred) == len(ca2_fold1_tweedie_pred)
assert np.array_equal(metadata_pd["sales"].to_numpy(), y_valid.to_numpy())

# Build prediction output
for model_name, predictions in [
    ("l2", ca2_fold1_l2_pred),
    ("tweedie", ca2_fold1_tweedie_pred)
]:
    output = metadata_pd.copy()
    output["store_id"] = "CA_2"
    output["actual"] = output.pop("sales")
    output["prediction"] = predictions
    output["fold"] = 1
    output["model"] = model_name
    output["horizon"] = (
        output["date"] - pd.Timestamp(config["valid_start"])
    ).dt.days + 1

    output_path = (
        PREDICTION_DIR /
        f"lightgbm_{model_name}_advanced_ca_2_fold1.parquet"
    )

    output.to_parquet(output_path, index=False)
    print("SAVED:", output_path, "| Rows:", len(output))

print("\nCA_2 FOLD 1 PREDICTIONS SAVED AND VERIFIED")


SAVED: D:\Retail-Demand\predictions\lightgbm_l2_advanced_ca_2_fold1.parquet | Rows: 85372
SAVED: D:\Retail-Demand\predictions\lightgbm_tweedie_advanced_ca_2_fold1.parquet | Rows: 85372

CA_2 FOLD 1 PREDICTIONS SAVED AND VERIFIED


In [86]:

# STEP 11: Inspect existing CA_2 baseline prediction files
# Purpose: Identify the exact schemas of Seasonal Naive and
# Croston SBA files before merging them with LightGBM predictions.

baseline_files = {
    "Seasonal Naive": PREDICTION_DIR / "seasonal_naive_fold_1.parquet",
    "Croston SBA": PREDICTION_DIR / "croston_sba_fold_1.parquet"
}

for model_name, path in baseline_files.items():
    print(f"\n{model_name}")
    print("File:", path)
    print("Exists:", path.exists())

    if path.exists():
        baseline_df = pl.read_parquet_schema(path)
        print("Columns:", list(baseline_df.keys()))
        print("Schema:", baseline_df)
    else:
        print("WARNING: Baseline file not found.")



Seasonal Naive
File: D:\Retail-Demand\predictions\seasonal_naive_fold_1.parquet
Exists: True
Columns: ['item_id', 'store_id', 'forecast_date', 'horizon', 'forecast', 'actual']
Schema: Schema({'item_id': String, 'store_id': String, 'forecast_date': Date, 'horizon': Int64, 'forecast': Int64, 'actual': Int64})

Croston SBA
File: D:\Retail-Demand\predictions\croston_sba_fold_1.parquet
Exists: True
Columns: ['item_id', 'store_id', 'forecast_date', 'horizon', 'forecast', 'actual']
Schema: Schema({'item_id': String, 'store_id': String, 'forecast_date': Date, 'horizon': Int64, 'forecast': Float64, 'actual': Int64})


In [87]:

# STEP 11: Compare all four forecasting models for CA_2 Fold 1
# Purpose:
# Compare Seasonal Naive, Croston SBA, LightGBM L2,
# and LightGBM Tweedie using identical validation observations.

import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

store = "CA_2"
fold = 1

# Load and filter baseline predictions for CA_2
seasonal_df = (
    pl.read_parquet(baseline_files["Seasonal Naive"])
    .filter(pl.col("store_id") == store)
    .to_pandas()
)

croston_df = (
    pl.read_parquet(baseline_files["Croston SBA"])
    .filter(pl.col("store_id") == store)
    .to_pandas()
)

# Load saved LightGBM predictions
l2_df = pd.read_parquet(
    PREDICTION_DIR / "lightgbm_l2_advanced_ca_2_fold1.parquet"
)

tweedie_df = pd.read_parquet(
    PREDICTION_DIR / "lightgbm_tweedie_advanced_ca_2_fold1.parquet"
)

# Normalize the prediction columns and keys
def normalize_predictions(df, prediction_column):
    result = df.copy()

    if "date" in result.columns:
        result = result.rename(columns={"date": "forecast_date"})

    return result[
        ["item_id", "store_id", "forecast_date", "actual", prediction_column]
    ].rename(columns={prediction_column: "prediction"})

model_predictions = {
    "Seasonal Naive": normalize_predictions(seasonal_df, "forecast"),
    "Croston SBA": normalize_predictions(croston_df, "forecast"),
    "LightGBM L2": normalize_predictions(l2_df, "prediction"),
    "LightGBM Tweedie": normalize_predictions(tweedie_df, "prediction")
}

# Validate observation counts and actual-value alignment
comparison_rows = []

reference = model_predictions["Seasonal Naive"].sort_values(
    ["item_id", "forecast_date"]
).reset_index(drop=True)

for model_name, df in model_predictions.items():
    df = df.sort_values(["item_id", "forecast_date"]).reset_index(drop=True)

    assert len(df) == 85372, f"{model_name}: unexpected row count"
    assert df[["item_id", "store_id", "forecast_date"]].equals(
        reference[["item_id", "store_id", "forecast_date"]]
    ), f"{model_name}: observation keys do not match"

    assert np.array_equal(
        df["actual"].to_numpy(),
        reference["actual"].to_numpy()
    ), f"{model_name}: actual values do not match"

    actual = df["actual"].to_numpy(dtype=float)
    predicted = df["prediction"].to_numpy(dtype=float)

    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    wape = np.abs(actual - predicted).sum() / actual.sum() * 100
    bias = (predicted - actual).sum() / actual.sum() * 100

    comparison_rows.append({
        "Fold": fold,
        "Store": store,
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "WAPE (%)": wape,
        "Bias (%)": bias
    })

comparison_df = (
    pd.DataFrame(comparison_rows)
    .sort_values("WAPE (%)")
    .reset_index(drop=True)
)

print("CA_2 FOLD 1 - FOUR MODEL COMPARISON")
print(comparison_df.to_string(index=False))

# Save comparison results
comparison_path = RESULTS_DIR / "ca_2_fold1_four_model_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)

print("\nAll four models: 85,372 matching observations")
print("COMPARISON SAVED:", comparison_path)
print("\nBEST MODEL:", comparison_df.iloc[0]["Model"])


CA_2 FOLD 1 - FOUR MODEL COMPARISON
 Fold Store            Model      MAE     RMSE  WAPE (%)  Bias (%)
    1  CA_2 LightGBM Tweedie 1.058978 2.027062 76.810793 -9.244271
    1  CA_2      LightGBM L2 1.085679 2.127390 78.747522 -8.919181
    1  CA_2      Croston SBA 1.141579 2.125852 82.802092 -0.223287
    1  CA_2   Seasonal Naive 1.266774 2.537557 91.882822 -2.097688

All four models: 85,372 matching observations
COMPARISON SAVED: D:\Retail-Demand\results\ca_2_fold1_four_model_comparison.csv

BEST MODEL: LightGBM Tweedie


In [88]:

# STEP 12: Clean up Fold 1 memory and verify CA_2 Fold 2
# Purpose: Release large objects from Fold 1 and check the
# chronological partitions for Fold 2 before training.

import gc
import psutil
import polars as pl

# Release Fold 1 objects that are no longer needed
objects_to_release = [
    "X_train", "y_train",
    "X_early_stop", "y_early_stop",
    "X_valid", "y_valid",
    "l2_model", "tweedie_model",
    "ca2_fold1_l2_pred", "ca2_fold1_tweedie_pred",
    "metadata_pd", "validation_metadata",
    "seasonal_df", "croston_df", "l2_df", "tweedie_df",
    "comparison_df", "model_predictions", "reference",
    "train_pd", "eval_pd"
]

for object_name in objects_to_release:
    globals().pop(object_name, None)

gc.collect()

# Configure CA_2 Fold 2
store = "CA_2"
fold = 2
config = FOLD_CONFIG[fold]

# Count chronological partitions directly from the Parquet file
fold2_counts = (
    pl.scan_parquet(FEATURE_FILES[store])
    .select([
        (pl.col("date") <= pl.lit(config["train_end"]).str.to_date())
        .sum().alias("Training"),

        (
            (pl.col("date") >= pl.lit(config["early_start"]).str.to_date()) &
            (pl.col("date") <= pl.lit(config["early_end"]).str.to_date())
        ).sum().alias("Early_Stopping"),

        (
            (pl.col("date") >= pl.lit(config["valid_start"]).str.to_date()) &
            (pl.col("date") <= pl.lit(config["valid_end"]).str.to_date())
        ).sum().alias("Validation")
    ])
    .collect()
)

print("CA_2 - FOLD 2 PARTITION VERIFICATION")
print(fold2_counts)

memory = psutil.virtual_memory()
print(f"\nAvailable RAM: {memory.available / (1024**3):.2f} GB")


CA_2 - FOLD 2 PARTITION VERIFICATION
shape: (1, 3)
┌──────────┬────────────────┬────────────┐
│ Training ┆ Early_Stopping ┆ Validation │
│ ---      ┆ ---            ┆ ---        │
│ u32      ┆ u32            ┆ u32        │
╞══════════╪════════════════╪════════════╡
│ 5576621  ┆ 85372          ┆ 85372      │
└──────────┴────────────────┴────────────┘

Available RAM: 8.43 GB


In [89]:

# STEP 13: Load CA_2 Fold 2 datasets
# Purpose: Load the three chronological partitions using Polars
# first, then convert to Pandas with aligned categorical features.

import gc
import pandas as pd
import polars as pl
import psutil

store = "CA_2"
fold = 2
config = FOLD_CONFIG[fold]

required_columns = feature_columns + ["sales"]

select_expressions = []

for col in required_columns:
    if col in categorical_features:
        select_expressions.append(pl.col(col).cast(pl.String))
    else:
        select_expressions.append(pl.col(col).cast(pl.Float32))

# Load training partition
train_pl = (
    pl.scan_parquet(FEATURE_FILES[store])
    .filter(
        pl.col("date") <= pl.lit(config["train_end"]).str.to_date()
    )
    .select(select_expressions)
    .collect()
)

train_pd = train_pl.to_pandas()
del train_pl
gc.collect()

for col in categorical_features:
    train_pd[col] = train_pd[col].astype("category")

X_train = train_pd[feature_columns].copy()
y_train = train_pd["sales"].copy()

del train_pd
gc.collect()

print("TRAINING DATA:", X_train.shape, y_train.shape)

# Load early-stopping and validation partitions
eval_pl = (
    pl.scan_parquet(FEATURE_FILES[store])
    .filter(
        (pl.col("date") >= pl.lit(config["early_start"]).str.to_date()) &
        (pl.col("date") <= pl.lit(config["valid_end"]).str.to_date())
    )
    .select(
        [pl.col(c).cast(pl.String) for c in categorical_features]
        + [pl.col(c).cast(pl.Float32) for c in numeric_features]
        + [
            pl.col("sales").cast(pl.Float32),
            pl.col("date")
        ]
    )
    .collect()
)

eval_pd = eval_pl.to_pandas()
del eval_pl
gc.collect()

# Match evaluation categorical levels to training levels
for col in categorical_features:
    eval_pd[col] = pd.Categorical(
        eval_pd[col],
        categories=X_train[col].cat.categories
    )

early_mask = (
    (eval_pd["date"] >= pd.Timestamp(config["early_start"])) &
    (eval_pd["date"] <= pd.Timestamp(config["early_end"]))
)

valid_mask = (
    (eval_pd["date"] >= pd.Timestamp(config["valid_start"])) &
    (eval_pd["date"] <= pd.Timestamp(config["valid_end"]))
)

X_early_stop = eval_pd.loc[early_mask, feature_columns].copy()
y_early_stop = eval_pd.loc[early_mask, "sales"].copy()

X_valid = eval_pd.loc[valid_mask, feature_columns].copy()
y_valid = eval_pd.loc[valid_mask, "sales"].copy()

del eval_pd, early_mask, valid_mask
gc.collect()

print("\nCA_2 FOLD 2 PARTITION SHAPES")
print("Training:", X_train.shape, y_train.shape)
print("Early stopping:", X_early_stop.shape, y_early_stop.shape)
print("Validation:", X_valid.shape, y_valid.shape)

print("\nNegative targets:")
print("Training:", (y_train < 0).sum())
print("Validation:", (y_valid < 0).sum())

memory = psutil.virtual_memory()
print(f"\nAvailable RAM: {memory.available / (1024**3):.2f} GB")


TRAINING DATA: (5576621, 26) (5576621,)

CA_2 FOLD 2 PARTITION SHAPES
Training: (5576621, 26) (5576621,)
Early stopping: (85372, 26) (85372,)
Validation: (85372, 26) (85372,)

Negative targets:
Training: 0
Validation: 0

Available RAM: 7.76 GB


In [90]:

# STEP 14: Train LightGBM L2 for CA_2 Fold 2
# Purpose: Train with squared-error loss, use the early-stopping
# partition to select the best iteration, and evaluate on Fold 2 validation.

l2_model = lgb.LGBMRegressor(
    objective="regression_l2",
    **COMMON_PARAMS
)

l2_model.fit(
    X_train,
    y_train,
    eval_set=[(X_early_stop, y_early_stop)],
    eval_metric="rmse",
    categorical_feature=categorical_features,
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=50)
    ]
)

ca2_fold2_l2_pred = np.maximum(l2_model.predict(X_valid), 0)

actual = y_valid.to_numpy()
predicted = ca2_fold2_l2_pred

mae = mean_absolute_error(actual, predicted)
rmse = np.sqrt(mean_squared_error(actual, predicted))
wape = np.abs(actual - predicted).sum() / actual.sum() * 100
bias = (predicted - actual).sum() / actual.sum() * 100

print("\nCA_2 FOLD 2 - LIGHTGBM L2 RESULTS")
print("Best iteration:", l2_model.best_iteration_)
print(f"MAE:       {mae:.4f}")
print(f"RMSE:      {rmse:.4f}")
print(f"WAPE:      {wape:.4f}%")
print(f"Bias:      {bias:.4f}%")

l2_model_path = MODEL_DIR / "lightgbm_l2_advanced_ca_2_fold2.txt"
l2_model.booster_.save_model(str(l2_model_path))

print("\nMODEL SAVED:", l2_model_path)


D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[50]	valid_0's rmse: 2.09737	valid_0's l2: 4.39896
[100]	valid_0's rmse: 2.06429	valid_0's l2: 4.26128
[150]	valid_0's rmse: 2.06766	valid_0's l2: 4.27521
Early stopping, best iteration is:
[112]	valid_0's rmse: 2.06191	valid_0's l2: 4.25149

CA_2 FOLD 2 - LIGHTGBM L2 RESULTS
Best iteration: 112
MAE:       1.1012
RMSE:      2.1058
WAPE:      81.5390%
Bias:      -2.6522%

MODEL SAVED: D:\Retail-Demand\models\lightgbm_l2_advanced_ca_2_fold2.txt


In [91]:

# STEP 15: Train LightGBM Tweedie for CA_2 Fold 2
# Purpose: Train with Tweedie loss and compare its performance
# against L2 using the same Fold 2 validation partition.

tweedie_model = lgb.LGBMRegressor(
    objective="tweedie",
    tweedie_variance_power=1.2,
    **COMMON_PARAMS
)

tweedie_model.fit(
    X_train,
    y_train,
    eval_set=[(X_early_stop, y_early_stop)],
    eval_metric="rmse",
    categorical_feature=categorical_features,
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=50)
    ]
)

ca2_fold2_tweedie_pred = np.maximum(
    tweedie_model.predict(X_valid), 0
)

actual = y_valid.to_numpy()
predicted = ca2_fold2_tweedie_pred

mae = mean_absolute_error(actual, predicted)
rmse = np.sqrt(mean_squared_error(actual, predicted))
wape = np.abs(actual - predicted).sum() / actual.sum() * 100
bias = (predicted - actual).sum() / actual.sum() * 100

print("\nCA_2 FOLD 2 - LIGHTGBM TWEEDIE RESULTS")
print("Best iteration:", tweedie_model.best_iteration_)
print(f"MAE:       {mae:.4f}")
print(f"RMSE:      {rmse:.4f}")
print(f"WAPE:      {wape:.4f}%")
print(f"Bias:      {bias:.4f}%")

tweedie_model_path = (
    MODEL_DIR / "lightgbm_tweedie_advanced_ca_2_fold2.txt"
)
tweedie_model.booster_.save_model(str(tweedie_model_path))

print("\nMODEL SAVED:", tweedie_model_path)


D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[50]	valid_0's rmse: 2.12755	valid_0's tweedie: 7.33036
[100]	valid_0's rmse: 2.05806	valid_0's tweedie: 7.32313
Early stopping, best iteration is:
[81]	valid_0's rmse: 2.07058	valid_0's tweedie: 7.32285

CA_2 FOLD 2 - LIGHTGBM TWEEDIE RESULTS
Best iteration: 81
MAE:       1.0972
RMSE:      2.1019
WAPE:      81.2403%
Bias:      -3.2020%

MODEL SAVED: D:\Retail-Demand\models\lightgbm_tweedie_advanced_ca_2_fold2.txt


In [92]:

# STEP 16: Save CA_2 Fold 2 LightGBM predictions
# Purpose: Save actual sales, forecast dates, and predictions
# for L2 and Tweedie so they can be compared with both baselines.

validation_metadata = (
    pl.scan_parquet(FEATURE_FILES["CA_2"])
    .filter(
        (pl.col("date") >= pl.lit(config["valid_start"]).str.to_date()) &
        (pl.col("date") <= pl.lit(config["valid_end"]).str.to_date())
    )
    .select(["item_id", "date", "sales"])
    .collect()
)

metadata_pd = validation_metadata.to_pandas()

assert len(metadata_pd) == len(ca2_fold2_l2_pred) == len(ca2_fold2_tweedie_pred)
assert np.array_equal(metadata_pd["sales"].to_numpy(), y_valid.to_numpy())

for model_name, predictions in [
    ("l2", ca2_fold2_l2_pred),
    ("tweedie", ca2_fold2_tweedie_pred)
]:
    output = metadata_pd.copy()
    output["store_id"] = "CA_2"
    output["actual"] = output.pop("sales")
    output["prediction"] = predictions
    output["fold"] = 2
    output["model"] = model_name
    output["horizon"] = (
        output["date"] - pd.Timestamp(config["valid_start"])
    ).dt.days + 1

    output_path = (
        PREDICTION_DIR /
        f"lightgbm_{model_name}_advanced_ca_2_fold2.parquet"
    )

    output.to_parquet(output_path, index=False)
    print("SAVED:", output_path, "| Rows:", len(output))

print("\nCA_2 FOLD 2 PREDICTIONS SAVED AND VERIFIED")


SAVED: D:\Retail-Demand\predictions\lightgbm_l2_advanced_ca_2_fold2.parquet | Rows: 85372
SAVED: D:\Retail-Demand\predictions\lightgbm_tweedie_advanced_ca_2_fold2.parquet | Rows: 85372

CA_2 FOLD 2 PREDICTIONS SAVED AND VERIFIED


In [93]:

# STEP 17: Compare all four models for CA_2 Fold 2
# Purpose: Evaluate both LightGBM models against Seasonal Naive
# and Croston SBA on the same 85,372 validation observations.

import pandas as pd
import numpy as np
import polars as pl
from sklearn.metrics import mean_absolute_error, mean_squared_error

store = "CA_2"
fold = 2

# Load the Fold 2 baseline files
seasonal_df = (
    pl.read_parquet(PREDICTION_DIR / "seasonal_naive_fold_2.parquet")
    .filter(pl.col("store_id") == store)
    .to_pandas()
)

croston_df = (
    pl.read_parquet(PREDICTION_DIR / "croston_sba_fold_2.parquet")
    .filter(pl.col("store_id") == store)
    .to_pandas()
)

# Load the saved LightGBM predictions
l2_df = pd.read_parquet(
    PREDICTION_DIR / "lightgbm_l2_advanced_ca_2_fold2.parquet"
)

tweedie_df = pd.read_parquet(
    PREDICTION_DIR / "lightgbm_tweedie_advanced_ca_2_fold2.parquet"
)

def normalize_predictions(df, prediction_column):
    result = df.copy()

    if "date" in result.columns:
        result = result.rename(columns={"date": "forecast_date"})

    return result[
        ["item_id", "store_id", "forecast_date", "actual", prediction_column]
    ].rename(columns={prediction_column: "prediction"})

model_predictions = {
    "Seasonal Naive": normalize_predictions(seasonal_df, "forecast"),
    "Croston SBA": normalize_predictions(croston_df, "forecast"),
    "LightGBM L2": normalize_predictions(l2_df, "prediction"),
    "LightGBM Tweedie": normalize_predictions(tweedie_df, "prediction")
}

# Verify identical observation keys and actual values
reference = (
    model_predictions["Seasonal Naive"]
    .sort_values(["item_id", "forecast_date"])
    .reset_index(drop=True)
)

comparison_rows = []

for model_name, df in model_predictions.items():
    df = (
        df.sort_values(["item_id", "forecast_date"])
        .reset_index(drop=True)
    )

    assert len(df) == 85372, f"{model_name}: unexpected row count"

    assert df[
        ["item_id", "store_id", "forecast_date"]
    ].equals(
        reference[["item_id", "store_id", "forecast_date"]]
    ), f"{model_name}: observation keys do not match"

    assert np.array_equal(
        df["actual"].to_numpy(),
        reference["actual"].to_numpy()
    ), f"{model_name}: actual values do not match"

    actual = df["actual"].to_numpy(dtype=float)
    predicted = df["prediction"].to_numpy(dtype=float)

    comparison_rows.append({
        "Fold": fold,
        "Store": store,
        "Model": model_name,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "WAPE (%)": np.abs(actual - predicted).sum() / actual.sum() * 100,
        "Bias (%)": (predicted - actual).sum() / actual.sum() * 100
    })

comparison_df = (
    pd.DataFrame(comparison_rows)
    .sort_values("WAPE (%)")
    .reset_index(drop=True)
)

print("CA_2 FOLD 2 - FOUR MODEL COMPARISON")
print(comparison_df.to_string(index=False))

comparison_path = RESULTS_DIR / "ca_2_fold2_four_model_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)

print("\nAll four models: 85,372 matching observations")
print("COMPARISON SAVED:", comparison_path)
print("\nBEST MODEL:", comparison_df.iloc[0]["Model"])


CA_2 FOLD 2 - FOUR MODEL COMPARISON
 Fold Store            Model      MAE     RMSE  WAPE (%)  Bias (%)
    2  CA_2 LightGBM Tweedie 1.097208 2.101914 81.240279 -3.202031
    2  CA_2      LightGBM L2 1.101243 2.105756 81.538998 -2.652231
    2  CA_2      Croston SBA 1.135918 2.140631 84.106432  4.488364
    2  CA_2   Seasonal Naive 1.286335 2.622716 95.243753  1.518634

All four models: 85,372 matching observations
COMPARISON SAVED: D:\Retail-Demand\results\ca_2_fold2_four_model_comparison.csv

BEST MODEL: LightGBM Tweedie


In [2]:

# RECOVERY CHECK: Verify saved CA_2 Fold 1 and Fold 2 outputs
# Purpose: Confirm completed work survived the kernel crash.

from pathlib import Path
import psutil

PROJECT_ROOT = Path.cwd().parent
MODEL_DIR = PROJECT_ROOT / "models"
PREDICTION_DIR = PROJECT_ROOT / "predictions"
RESULTS_DIR = PROJECT_ROOT / "results"

files_to_check = [
    MODEL_DIR / "lightgbm_l2_advanced_ca_2_fold1.txt",
    MODEL_DIR / "lightgbm_tweedie_advanced_ca_2_fold1.txt",
    MODEL_DIR / "lightgbm_l2_advanced_ca_2_fold2.txt",
    MODEL_DIR / "lightgbm_tweedie_advanced_ca_2_fold2.txt",
    PREDICTION_DIR / "lightgbm_l2_advanced_ca_2_fold1.parquet",
    PREDICTION_DIR / "lightgbm_tweedie_advanced_ca_2_fold1.parquet",
    PREDICTION_DIR / "lightgbm_l2_advanced_ca_2_fold2.parquet",
    PREDICTION_DIR / "lightgbm_tweedie_advanced_ca_2_fold2.parquet",
    RESULTS_DIR / "ca_2_fold1_four_model_comparison.csv",
    RESULTS_DIR / "ca_2_fold2_four_model_comparison.csv"
]

for path in files_to_check:
    print(f"{'OK' if path.exists() else 'MISSING'}: {path.name}")

memory = psutil.virtual_memory()
print(f"\nAvailable RAM: {memory.available / (1024**3):.2f} GB")


OK: lightgbm_l2_advanced_ca_2_fold1.txt
OK: lightgbm_tweedie_advanced_ca_2_fold1.txt
OK: lightgbm_l2_advanced_ca_2_fold2.txt
OK: lightgbm_tweedie_advanced_ca_2_fold2.txt
OK: lightgbm_l2_advanced_ca_2_fold1.parquet
OK: lightgbm_tweedie_advanced_ca_2_fold1.parquet
OK: lightgbm_l2_advanced_ca_2_fold2.parquet
OK: lightgbm_tweedie_advanced_ca_2_fold2.parquet
OK: ca_2_fold1_four_model_comparison.csv
OK: ca_2_fold2_four_model_comparison.csv

Available RAM: 8.92 GB


In [6]:

# STEP 20: Restore project configuration after Jupyter kernel restart
# Purpose: Restore imports, exact feature paths, model features,
# rolling-origin dates, and LightGBM settings.

import gc
import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb
import psutil

from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Project directories
PROJECT_ROOT = Path.cwd().parent

FEATURE_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
PREDICTION_DIR = PROJECT_ROOT / "predictions"
RESULTS_DIR = PROJECT_ROOT / "results"

# Exact existing feature filenames
FEATURE_FILES = {
    "CA_2": FEATURE_DIR / "ca2_advanced_features.parquet",
    "CA_3": FEATURE_DIR / "ca_3_advanced_features.parquet",
    "CA_4": FEATURE_DIR / "ca_4_advanced_features.parquet",
    "TX_1": FEATURE_DIR / "tx_1_advanced_features.parquet",
    "TX_2": FEATURE_DIR / "tx_2_advanced_features.parquet",
    "TX_3": FEATURE_DIR / "tx_3_advanced_features.parquet",
    "WI_1": FEATURE_DIR / "wi_1_advanced_features.parquet",
    "WI_2": FEATURE_DIR / "wi_2_advanced_features.parquet",
    "WI_3": FEATURE_DIR / "wi_3_advanced_features.parquet"
}

# Model features
feature_columns = [
    "item_id", "event_name_1", "day_of_week", "month", "snap_flag",
    "lag_28", "lag_35", "lag_42", "lag_56", "lag_364",
    "rolling_mean_7", "rolling_mean_28", "rolling_mean_56",
    "rolling_std_7", "rolling_std_28",
    "day_of_week_sin", "day_of_week_cos",
    "month_sin", "month_cos",
    "ema_28", "zero_sales_run",
    "price_lag_28", "price_lag_56",
    "price_momentum", "historical_max_price_56",
    "discount_ratio"
]

categorical_features = [
    "item_id", "event_name_1", "day_of_week", "month"
]

numeric_features = [
    col for col in feature_columns
    if col not in categorical_features
]

# Rolling-origin fold dates
FOLD_CONFIG = {
    1: {
        "train_end": "2016-01-03",
        "early_start": "2016-01-04",
        "early_end": "2016-01-31",
        "valid_start": "2016-02-01",
        "valid_end": "2016-02-28"
    },
    2: {
        "train_end": "2016-01-31",
        "early_start": "2016-02-01",
        "early_end": "2016-02-28",
        "valid_start": "2016-02-29",
        "valid_end": "2016-03-27"
    },
    3: {
        "train_end": "2016-02-28",
        "early_start": "2016-02-29",
        "early_end": "2016-03-27",
        "valid_start": "2016-03-28",
        "valid_end": "2016-04-24"
    }
}

# Consistent memory-conscious parameters for remaining stores
COMMON_PARAMS = {
    "n_estimators": 700,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_bin": 63,
    "min_child_samples": 100,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "n_jobs": 2,
    "random_state": 42,
    "verbosity": -1
}

print("NOTEBOOK CONFIGURATION RESTORED")
print("Project:", PROJECT_ROOT)
print("Features:", len(feature_columns))
print("Categorical:", len(categorical_features))
print("Numeric:", len(numeric_features))
print("Configured stores:", len(FEATURE_FILES))

print("\nFeature files:")
for store, path in FEATURE_FILES.items():
    print(f"{store}: {'FOUND' if path.exists() else 'MISSING'}")

print("\nAvailable RAM:",
      f"{psutil.virtual_memory().available / (1024**3):.2f} GB")


NOTEBOOK CONFIGURATION RESTORED
Project: D:\Retail-Demand
Features: 26
Categorical: 4
Numeric: 22
Configured stores: 9

Feature files:
CA_2: FOUND
CA_3: FOUND
CA_4: FOUND
TX_1: FOUND
TX_2: FOUND
TX_3: FOUND
WI_1: FOUND
WI_2: FOUND
WI_3: FOUND

Available RAM: 7.45 GB


In [4]:

# STEP 21: Load CA_2 Fold 3 training partition after recovery
# Purpose: Resume the interrupted Fold 3 workflow without
# repeating any completed model training.

store = "CA_2"
fold = 3
config = FOLD_CONFIG[fold]

required_columns = feature_columns + ["sales"]

select_expressions = []

for col in required_columns:
    if col in categorical_features:
        select_expressions.append(pl.col(col).cast(pl.String))
    else:
        select_expressions.append(pl.col(col).cast(pl.Float32))

# Filter the existing Parquet file before collecting
train_pl = (
    pl.scan_parquet(FEATURE_FILES[store])
    .filter(
        pl.col("date") <= pl.lit(config["train_end"]).str.to_date()
    )
    .select(select_expressions)
    .collect()
)

print("POLARS TRAINING DATA:", train_pl.shape)

train_pd = train_pl.to_pandas()
del train_pl
gc.collect()

for col in categorical_features:
    train_pd[col] = train_pd[col].astype("category")

X_train = train_pd[feature_columns].copy()
y_train = train_pd["sales"].copy()

del train_pd
gc.collect()

print("\nCA_2 FOLD 3 TRAINING DATA")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("\nCategorical types:")
print(X_train[categorical_features].dtypes)

memory = psutil.virtual_memory()
print(f"\nAvailable RAM: {memory.available / (1024**3):.2f} GB")


POLARS TRAINING DATA: (5661993, 27)

CA_2 FOLD 3 TRAINING DATA
X_train: (5661993, 26)
y_train: (5661993,)

Categorical types:
item_id         category
event_name_1    category
day_of_week     category
month           category
dtype: object

Available RAM: 5.92 GB


In [5]:

# STEP 22: Load CA_2 Fold 3 early-stopping and validation data
# Purpose: Load both 28-day evaluation partitions and align
# categorical levels with the training dataset.

eval_pl = (
    pl.scan_parquet(FEATURE_FILES[store])
    .filter(
        (pl.col("date") >= pl.lit(config["early_start"]).str.to_date()) &
        (pl.col("date") <= pl.lit(config["valid_end"]).str.to_date())
    )
    .select(
        [pl.col(c).cast(pl.String) for c in categorical_features]
        + [pl.col(c).cast(pl.Float32) for c in numeric_features]
        + [
            pl.col("sales").cast(pl.Float32),
            pl.col("date")
        ]
    )
    .collect()
)

print("EVALUATION DATA LOADED:", eval_pl.shape)

eval_pd = eval_pl.to_pandas()
del eval_pl
gc.collect()

# Align categorical levels with training
for col in categorical_features:
    eval_pd[col] = pd.Categorical(
        eval_pd[col],
        categories=X_train[col].cat.categories
    )

# Separate early-stopping and validation periods
early_mask = (
    (eval_pd["date"] >= pd.Timestamp(config["early_start"])) &
    (eval_pd["date"] <= pd.Timestamp(config["early_end"]))
)

valid_mask = (
    (eval_pd["date"] >= pd.Timestamp(config["valid_start"])) &
    (eval_pd["date"] <= pd.Timestamp(config["valid_end"]))
)

X_early_stop = eval_pd.loc[early_mask, feature_columns].copy()
y_early_stop = eval_pd.loc[early_mask, "sales"].copy()

X_valid = eval_pd.loc[valid_mask, feature_columns].copy()
y_valid = eval_pd.loc[valid_mask, "sales"].copy()

del eval_pd, early_mask, valid_mask
gc.collect()

print("\nCA_2 FOLD 3 PARTITION SHAPES")
print("Training:", X_train.shape, y_train.shape)
print("Early stopping:", X_early_stop.shape, y_early_stop.shape)
print("Validation:", X_valid.shape, y_valid.shape)

print("\nNEGATIVE TARGET CHECK")
print("Training:", (y_train < 0).sum())
print("Early stopping:", (y_early_stop < 0).sum())
print("Validation:", (y_valid < 0).sum())

memory = psutil.virtual_memory()
print(f"\nAvailable RAM: {memory.available / (1024**3):.2f} GB")


EVALUATION DATA LOADED: (170744, 28)

CA_2 FOLD 3 PARTITION SHAPES
Training: (5661993, 26) (5661993,)
Early stopping: (85372, 26) (85372,)
Validation: (85372, 26) (85372,)

NEGATIVE TARGET CHECK
Training: 0
Early stopping: 0
Validation: 0

Available RAM: 7.98 GB


In [6]:

# STEP 23: Train LightGBM L2 for CA_2 Fold 3
# Purpose: Train using squared-error loss, select the best
# iteration through early stopping, and evaluate on Fold 3 validation.

l2_model = lgb.LGBMRegressor(
    objective="regression_l2",
    **COMMON_PARAMS
)

l2_model.fit(
    X_train,
    y_train,
    eval_set=[(X_early_stop, y_early_stop)],
    eval_metric="rmse",
    categorical_feature=categorical_features,
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=50)
    ]
)

ca2_fold3_l2_pred = np.maximum(
    l2_model.predict(X_valid), 0
)

actual = y_valid.to_numpy()
predicted = ca2_fold3_l2_pred

mae = mean_absolute_error(actual, predicted)
rmse = np.sqrt(mean_squared_error(actual, predicted))
wape = np.abs(actual - predicted).sum() / actual.sum() * 100
bias = (predicted - actual).sum() / actual.sum() * 100

print("\nCA_2 FOLD 3 - LIGHTGBM L2 RESULTS")
print("Best iteration:", l2_model.best_iteration_)
print(f"MAE:       {mae:.4f}")
print(f"RMSE:      {rmse:.4f}")
print(f"WAPE:      {wape:.4f}%")
print(f"Bias:      {bias:.4f}%")

l2_model_path = MODEL_DIR / "lightgbm_l2_advanced_ca_2_fold3.txt"
l2_model.booster_.save_model(str(l2_model_path))

print("\nMODEL SAVED:", l2_model_path)


D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[50]	valid_0's rmse: 2.12322	valid_0's l2: 4.50806
[100]	valid_0's rmse: 2.10197	valid_0's l2: 4.41829
[150]	valid_0's rmse: 2.09986	valid_0's l2: 4.40942
Early stopping, best iteration is:
[142]	valid_0's rmse: 2.09965	valid_0's l2: 4.40854

CA_2 FOLD 3 - LIGHTGBM L2 RESULTS
Best iteration: 142
MAE:       1.1151
RMSE:      2.0663
WAPE:      76.1400%
Bias:      -10.1119%

MODEL SAVED: D:\Retail-Demand\models\lightgbm_l2_advanced_ca_2_fold3.txt


In [7]:

# STEP 24: Train LightGBM Tweedie for CA_2 Fold 3
# Purpose: Train using Tweedie loss, apply early stopping,
# evaluate on the outer validation partition, and save the model.

tweedie_model = lgb.LGBMRegressor(
    objective="tweedie",
    tweedie_variance_power=1.2,
    **COMMON_PARAMS
)

tweedie_model.fit(
    X_train,
    y_train,
    eval_set=[(X_early_stop, y_early_stop)],
    eval_metric="rmse",
    categorical_feature=categorical_features,
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=50)
    ]
)

ca2_fold3_tweedie_pred = np.maximum(
    tweedie_model.predict(X_valid), 0
)

actual = y_valid.to_numpy()
predicted = ca2_fold3_tweedie_pred

mae = mean_absolute_error(actual, predicted)
rmse = np.sqrt(mean_squared_error(actual, predicted))
wape = np.abs(actual - predicted).sum() / actual.sum() * 100
bias = (predicted - actual).sum() / actual.sum() * 100

print("\nCA_2 FOLD 3 - LIGHTGBM TWEEDIE RESULTS")
print("Best iteration:", tweedie_model.best_iteration_)
print(f"MAE:       {mae:.4f}")
print(f"RMSE:      {rmse:.4f}")
print(f"WAPE:      {wape:.4f}%")
print(f"Bias:      {bias:.4f}%")

tweedie_model_path = (
    MODEL_DIR / "lightgbm_tweedie_advanced_ca_2_fold3.txt"
)
tweedie_model.booster_.save_model(str(tweedie_model_path))

print("\nMODEL SAVED:", tweedie_model_path)


D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[50]	valid_0's rmse: 2.14624	valid_0's tweedie: 7.24483
[100]	valid_0's rmse: 2.09762	valid_0's tweedie: 7.2391
Early stopping, best iteration is:
[91]	valid_0's rmse: 2.10402	valid_0's tweedie: 7.23874

CA_2 FOLD 3 - LIGHTGBM TWEEDIE RESULTS
Best iteration: 91
MAE:       1.1193
RMSE:      2.1045
WAPE:      76.4277%
Bias:      -10.6288%

MODEL SAVED: D:\Retail-Demand\models\lightgbm_tweedie_advanced_ca_2_fold3.txt


In [8]:
# STEP 25: Save CA_2 Fold 3 LightGBM Predictions

validation_metadata = (
    pl.scan_parquet(FEATURE_FILES["CA_2"])
    .filter(
        (pl.col("date") >= pl.lit(config["valid_start"]).str.to_date()) &
        (pl.col("date") <= pl.lit(config["valid_end"]).str.to_date())
    )
    .select(["item_id", "date", "sales"])
    .collect()
)

metadata_pd = validation_metadata.to_pandas()

assert len(metadata_pd) == len(ca2_fold3_l2_pred)
assert len(metadata_pd) == len(ca2_fold3_tweedie_pred)
assert np.array_equal(
    metadata_pd["sales"].to_numpy(),
    y_valid.to_numpy()
)

for model_name, predictions in [
    ("l2", ca2_fold3_l2_pred),
    ("tweedie", ca2_fold3_tweedie_pred)
]:
    output = metadata_pd.copy()

    output["store_id"] = "CA_2"
    output["actual"] = output.pop("sales")
    output["prediction"] = predictions
    output["fold"] = 3
    output["model"] = model_name

    output["horizon"] = (
        output["date"] - pd.Timestamp(config["valid_start"])
    ).dt.days + 1

    output_path = (
        PREDICTION_DIR /
        f"lightgbm_{model_name}_advanced_ca_2_fold3.parquet"
    )

    output.to_parquet(output_path, index=False)

    print(f"Saved: {output_path}")
    print(f"Rows: {len(output):,}")

print("\nCA_2 FOLD 3 PREDICTIONS SAVED AND VERIFIED.")


Saved: D:\Retail-Demand\predictions\lightgbm_l2_advanced_ca_2_fold3.parquet
Rows: 85,372
Saved: D:\Retail-Demand\predictions\lightgbm_tweedie_advanced_ca_2_fold3.parquet
Rows: 85,372

CA_2 FOLD 3 PREDICTIONS SAVED AND VERIFIED.


In [9]:
# STEP 26: Four-Model Comparison for CA_2 Fold 3

# Load LightGBM predictions
l2_df = pd.read_parquet(
    PREDICTION_DIR / "lightgbm_l2_advanced_ca_2_fold3.parquet"
)

tweedie_df = pd.read_parquet(
    PREDICTION_DIR / "lightgbm_tweedie_advanced_ca_2_fold3.parquet"
)

# Load baseline predictions
seasonal_df = pd.read_parquet(
    PREDICTION_DIR / "seasonal_naive_fold_3.parquet"
)

croston_df = pd.read_parquet(
    PREDICTION_DIR / "croston_sba_fold_3.parquet"
)

# Filter baseline predictions for CA_2
seasonal_df = seasonal_df[
    seasonal_df["store_id"] == "CA_2"
].copy()

croston_df = croston_df[
    croston_df["store_id"] == "CA_2"
].copy()

# Standardize prediction columns
seasonal_df = seasonal_df.rename(
    columns={"forecast": "prediction"}
)

croston_df = croston_df.rename(
    columns={"forecast": "prediction"}
)

# Standardize date column
seasonal_df = seasonal_df.rename(
    columns={"forecast_date": "date"}
)

croston_df = croston_df.rename(
    columns={"forecast_date": "date"}
)

# Define evaluation function
def calculate_metrics(actual, predicted):
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    wape = (
        np.abs(actual - predicted).sum() /
        actual.sum() * 100
    )
    bias = (
        (predicted - actual).sum() /
        actual.sum() * 100
    )

    return {
        "MAE": mae,
        "RMSE": rmse,
        "WAPE (%)": wape,
        "Bias (%)": bias
    }

# Validate row counts and actual values
assert len(l2_df) == 85372
assert len(tweedie_df) == 85372
assert len(seasonal_df) == 85372
assert len(croston_df) == 85372

# Match all models on item and date
comparison_data = l2_df[
    ["item_id", "date", "actual", "prediction"]
].copy()

comparison_data = comparison_data.rename(
    columns={"prediction": "l2_prediction"}
)

comparison_data = comparison_data.merge(
    tweedie_df[["item_id", "date", "actual", "prediction"]]
    .rename(columns={"prediction": "tweedie_prediction",
                     "actual": "tweedie_actual"}),
    on=["item_id", "date"],
    validate="one_to_one"
)

comparison_data = comparison_data.merge(
    seasonal_df[["item_id", "date", "actual", "prediction"]]
    .rename(columns={"prediction": "seasonal_prediction",
                     "actual": "seasonal_actual"}),
    on=["item_id", "date"],
    validate="one_to_one"
)

comparison_data = comparison_data.merge(
    croston_df[["item_id", "date", "actual", "prediction"]]
    .rename(columns={"prediction": "croston_prediction",
                     "actual": "croston_actual"}),
    on=["item_id", "date"],
    validate="one_to_one"
)

# Verify actual values agree across models
assert np.array_equal(
    comparison_data["actual"].to_numpy(),
    comparison_data["tweedie_actual"].to_numpy()
)
assert np.array_equal(
    comparison_data["actual"].to_numpy(),
    comparison_data["seasonal_actual"].to_numpy()
)
assert np.array_equal(
    comparison_data["actual"].to_numpy(),
    comparison_data["croston_actual"].to_numpy()
)

actual = comparison_data["actual"].to_numpy()

# Calculate metrics
model_predictions = {
    "LightGBM Tweedie": comparison_data["tweedie_prediction"].to_numpy(),
    "LightGBM L2": comparison_data["l2_prediction"].to_numpy(),
    "Croston SBA": comparison_data["croston_prediction"].to_numpy(),
    "Seasonal Naive": comparison_data["seasonal_prediction"].to_numpy()
}

comparison_results = []

for model_name, predictions in model_predictions.items():
    metrics = calculate_metrics(actual, predictions)

    comparison_results.append({
        "Store": "CA_2",
        "Fold": 3,
        "Model": model_name,
        **metrics
    })

comparison_df = pd.DataFrame(comparison_results)

# Sort by WAPE
comparison_df = comparison_df.sort_values(
    "WAPE (%)"
).reset_index(drop=True)

# Save results
comparison_path = (
    RESULTS_DIR / "ca_2_fold3_four_model_comparison.csv"
)

comparison_df.to_csv(comparison_path, index=False)

print("CA_2 FOLD 3 - FOUR MODEL COMPARISON")
display(comparison_df)

print("\nBEST MODEL:", comparison_df.iloc[0]["Model"])
print("SAVED:", comparison_path)
print("Rows evaluated:", len(comparison_data))

ValueError: You are trying to merge on datetime64[ms] and object columns for key 'date'. If you wish to proceed you should use pd.concat

In [10]:
# STEP 26 (FIXED): Four-Model Comparison for CA_2 Fold 3

# Load saved predictions
l2_df = pd.read_parquet(
    PREDICTION_DIR / "lightgbm_l2_advanced_ca_2_fold3.parquet"
)

tweedie_df = pd.read_parquet(
    PREDICTION_DIR / "lightgbm_tweedie_advanced_ca_2_fold3.parquet"
)

seasonal_df = pd.read_parquet(
    PREDICTION_DIR / "seasonal_naive_fold_3.parquet"
)

croston_df = pd.read_parquet(
    PREDICTION_DIR / "croston_sba_fold_3.parquet"
)

# Filter baseline predictions for CA_2
seasonal_df = seasonal_df[
    seasonal_df["store_id"] == "CA_2"
].copy()

croston_df = croston_df[
    croston_df["store_id"] == "CA_2"
].copy()

# Standardize column names
seasonal_df = seasonal_df.rename(
    columns={
        "forecast_date": "date",
        "forecast": "prediction"
    }
)

croston_df = croston_df.rename(
    columns={
        "forecast_date": "date",
        "forecast": "prediction"
    }
)

# FIX: Convert all date columns to datetime
for df in [l2_df, tweedie_df, seasonal_df, croston_df]:
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()

# Check row counts
assert all(
    len(df) == 85372
    for df in [l2_df, tweedie_df, seasonal_df, croston_df]
)

# Build comparison using LightGBM L2 as the reference
comparison_data = l2_df[
    ["item_id", "date", "actual", "prediction"]
].rename(
    columns={"prediction": "l2_prediction"}
)

# Merge Tweedie
comparison_data = comparison_data.merge(
    tweedie_df[
        ["item_id", "date", "actual", "prediction"]
    ].rename(
        columns={
            "actual": "tweedie_actual",
            "prediction": "tweedie_prediction"
        }
    ),
    on=["item_id", "date"],
    validate="one_to_one"
)

# Merge Seasonal Naive
comparison_data = comparison_data.merge(
    seasonal_df[
        ["item_id", "date", "actual", "prediction"]
    ].rename(
        columns={
            "actual": "seasonal_actual",
            "prediction": "seasonal_prediction"
        }
    ),
    on=["item_id", "date"],
    validate="one_to_one"
)

# Merge Croston SBA
comparison_data = comparison_data.merge(
    croston_df[
        ["item_id", "date", "actual", "prediction"]
    ].rename(
        columns={
            "actual": "croston_actual",
            "prediction": "croston_prediction"
        }
    ),
    on=["item_id", "date"],
    validate="one_to_one"
)

# Verify actual values match
for actual_column in [
    "tweedie_actual",
    "seasonal_actual",
    "croston_actual"
]:
    assert np.array_equal(
        comparison_data["actual"].to_numpy(),
        comparison_data[actual_column].to_numpy()
    ), f"Actual values do not match: {actual_column}"

# Calculate evaluation metrics
actual = comparison_data["actual"].to_numpy()

model_predictions = {
    "LightGBM Tweedie": comparison_data["tweedie_prediction"].to_numpy(),
    "LightGBM L2": comparison_data["l2_prediction"].to_numpy(),
    "Croston SBA": comparison_data["croston_prediction"].to_numpy(),
    "Seasonal Naive": comparison_data["seasonal_prediction"].to_numpy()
}

comparison_results = []

for model_name, predictions in model_predictions.items():
    comparison_results.append({
        "Store": "CA_2",
        "Fold": 3,
        "Model": model_name,
        **calculate_metrics(actual, predictions)
    })

comparison_df = pd.DataFrame(comparison_results)

comparison_df = comparison_df.sort_values(
    "WAPE (%)"
).reset_index(drop=True)

# Save comparison
comparison_path = (
    RESULTS_DIR / "ca_2_fold3_four_model_comparison.csv"
)

comparison_df.to_csv(comparison_path, index=False)

print("CA_2 FOLD 3 - FOUR MODEL COMPARISON")
display(comparison_df)

print("\nBEST MODEL:", comparison_df.iloc[0]["Model"])
print("ROWS EVALUATED:", len(comparison_data))
print("SAVED:", comparison_path)

CA_2 FOLD 3 - FOUR MODEL COMPARISON


,Store,Fold,Model,MAE,RMSE,WAPE (%),Bias (%)
0,CA_2,3,LightGBM L2,1.115103,2.066308,76.139955,-10.111935
1,CA_2,3,LightGBM Tweedie,1.119317,2.104512,76.427701,-10.628837
2,CA_2,3,Croston SBA,1.163371,2.155290,79.435751,-7.792296
3,CA_2,3,Seasonal Naive,1.355913,2.728292,92.582640,0.176756



BEST MODEL: LightGBM L2
ROWS EVALUATED: 85372
SAVED: D:\Retail-Demand\results\ca_2_fold3_four_model_comparison.csv


In [11]:
# STEP 27: Consolidate CA_2 Three-Fold Results

fold_files = [
    RESULTS_DIR / "ca_2_fold1_four_model_comparison.csv",
    RESULTS_DIR / "ca_2_fold2_four_model_comparison.csv",
    RESULTS_DIR / "ca_2_fold3_four_model_comparison.csv"
]

# Verify all three files exist
for file in fold_files:
    assert file.exists(), f"Missing file: {file}"

# Load and combine fold results
fold_results = pd.concat(
    [pd.read_csv(file) for file in fold_files],
    ignore_index=True
)

# Save combined fold comparison
all_folds_path = RESULTS_DIR / "ca_2_all_fold_comparisons.csv"
fold_results.to_csv(all_folds_path, index=False)

# Calculate average metrics for each model
summary = (
    fold_results
    .groupby(["Store", "Model"], as_index=False)
    [["MAE", "RMSE", "WAPE (%)", "Bias (%)"]]
    .mean()
    .sort_values("WAPE (%)")
    .reset_index(drop=True)
)

summary_path = RESULTS_DIR / "ca_2_three_fold_model_summary.csv"
summary.to_csv(summary_path, index=False)

print("CA_2 THREE-FOLD CONSOLIDATED RESULTS")
display(summary)

print("\nBEST MODEL ACROSS THREE FOLDS:")
print(summary.iloc[0]["Model"])

print("\nFILES SAVED:")
print(all_folds_path)
print(summary_path)

print("\nFOLD RECORDS:", len(fold_results))
print("MODEL SUMMARY RECORDS:", len(summary))

CA_2 THREE-FOLD CONSOLIDATED RESULTS


,Store,Model,MAE,RMSE,WAPE (%),Bias (%)
0,CA_2,LightGBM Tweedie,1.091834,2.077829,78.159591,-7.691713
1,CA_2,LightGBM L2,1.100675,2.099818,78.808825,-7.227782
2,CA_2,Croston SBA,1.146956,2.140591,82.114758,-1.175739
3,CA_2,Seasonal Naive,1.303007,2.629521,93.236405,-0.134099



BEST MODEL ACROSS THREE FOLDS:
LightGBM Tweedie

FILES SAVED:
D:\Retail-Demand\results\ca_2_all_fold_comparisons.csv
D:\Retail-Demand\results\ca_2_three_fold_model_summary.csv

FOLD RECORDS: 12
MODEL SUMMARY RECORDS: 4


In [13]:
# STEP 28: Load CA_3 Advanced Features and Verify Fold 1

store = "CA_3"
fold = 1
config = FOLD_CONFIG[fold]

feature_path = FEATURE_FILES[store]

assert feature_path.exists(), f"Feature file missing: {feature_path}"

print("STORE:", store)
print("FEATURE FILE:", feature_path)

# Read only the date and target columns to verify partitions
store_data = pl.scan_parquet(feature_path)

train_count = (
    store_data
    .filter(pl.col("date") <= pl.lit(config["train_end"]).str.to_date())
    .select(pl.len())
    .collect()
    .item()
)

early_count = (
    store_data
    .filter(
        (pl.col("date") >= pl.lit(config["early_start"]).str.to_date()) &
        (pl.col("date") <= pl.lit(config["early_end"]).str.to_date())
    )
    .select(pl.len())
    .collect()
    .item()
)

valid_count = (
    store_data
    .filter(
        (pl.col("date") >= pl.lit(config["valid_start"]).str.to_date()) &
        (pl.col("date") <= pl.lit(config["valid_end"]).str.to_date())
    )
    .select(pl.len())
    .collect()
    .item()
)

print("\nCA_3 FOLD 1 PARTITION CHECK")
print("Training rows:", f"{train_count:,}")
print("Early stopping rows:", f"{early_count:,}")
print("Validation rows:", f"{valid_count:,}")

assert early_count == 85372
assert valid_count == 85372

print("\nCA_3 FOLD 1 PARTITIONS VERIFIED.")

STORE: CA_3
FEATURE FILE: D:\Retail-Demand\data\processed\ca_3_advanced_features.parquet

CA_3 FOLD 1 PARTITION CHECK
Training rows: 5,491,249
Early stopping rows: 85,372
Validation rows: 85,372

CA_3 FOLD 1 PARTITIONS VERIFIED.


In [14]:
# STEP 29: Load CA_3 Fold 1 Training and Validation Data

columns_to_load = ["date", "sales"] + feature_columns

# Training partition
train_data = (
    pl.scan_parquet(FEATURE_FILES[store])
    .filter(
        pl.col("date") <=
        pl.lit(config["train_end"]).str.to_date()
    )
    .select(columns_to_load)
    .collect()
)

# Early stopping partition
early_data = (
    pl.scan_parquet(FEATURE_FILES[store])
    .filter(
        (pl.col("date") >= pl.lit(config["early_start"]).str.to_date()) &
        (pl.col("date") <= pl.lit(config["early_end"]).str.to_date())
    )
    .select(columns_to_load)
    .collect()
)

# Validation partition
valid_data = (
    pl.scan_parquet(FEATURE_FILES[store])
    .filter(
        (pl.col("date") >= pl.lit(config["valid_start"]).str.to_date()) &
        (pl.col("date") <= pl.lit(config["valid_end"]).str.to_date())
    )
    .select(columns_to_load)
    .collect()
)

# Convert to Pandas
train_pd = train_data.to_pandas()
early_pd = early_data.to_pandas()
valid_pd = valid_data.to_pandas()

# Separate features and targets
X_train = train_pd[feature_columns]
y_train = train_pd["sales"]

X_early_stop = early_pd[feature_columns]
y_early_stop = early_pd["sales"]

X_valid = valid_pd[feature_columns]
y_valid = valid_pd["sales"]

# Set categorical columns consistently
for col in categorical_features:
    categories = sorted(
        set(X_train[col].dropna().unique()) |
        set(X_early_stop[col].dropna().unique()) |
        set(X_valid[col].dropna().unique())
    )

    for X in [X_train, X_early_stop, X_valid]:
        X[col] = pd.Categorical(
            X[col],
            categories=categories
        )

print("CA_3 FOLD 1 DATA LOADED")

print("\nTraining features:", X_train.shape)
print("Early stopping features:", X_early_stop.shape)
print("Validation features:", X_valid.shape)

print("\nTraining target:", y_train.shape)
print("Early stopping target:", y_early_stop.shape)
print("Validation target:", y_valid.shape)

print("\nAvailable RAM:",
      round(psutil.virtual_memory().available / (1024**3), 2),
      "GB")

assert X_train.shape[1] == 26
assert X_early_stop.shape[1] == 26
assert X_valid.shape[1] == 26

print("\nCA_3 FOLD 1 DATA READY FOR MODEL TRAINING.")

CA_3 FOLD 1 DATA LOADED

Training features: (5491249, 26)
Early stopping features: (85372, 26)
Validation features: (85372, 26)

Training target: (5491249,)
Early stopping target: (85372,)
Validation target: (85372,)

Available RAM: 6.19 GB

CA_3 FOLD 1 DATA READY FOR MODEL TRAINING.


In [15]:
# STEP 30: Train LightGBM L2 for CA_3 Fold 1

l2_model = lgb.LGBMRegressor(
    objective="regression_l2",
    **COMMON_PARAMS
)

l2_model.fit(
    X_train,
    y_train,
    eval_set=[(X_early_stop, y_early_stop)],
    eval_metric="rmse",
    categorical_feature=categorical_features,
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=50)
    ]
)

# Generate validation predictions
ca3_fold1_l2_pred = np.maximum(
    l2_model.predict(X_valid),
    0
)

# Calculate metrics
actual = y_valid.to_numpy()
predicted = ca3_fold1_l2_pred

mae = mean_absolute_error(actual, predicted)
rmse = np.sqrt(mean_squared_error(actual, predicted))
wape = np.abs(actual - predicted).sum() / actual.sum() * 100
bias = (predicted - actual).sum() / actual.sum() * 100

print("\nCA_3 FOLD 1 - LIGHTGBM L2 RESULTS")
print("Best iteration:", l2_model.best_iteration_)
print(f"MAE:       {mae:.4f}")
print(f"RMSE:      {rmse:.4f}")
print(f"WAPE:      {wape:.4f}%")
print(f"Bias:      {bias:.4f}%")

# Save model
model_path = MODEL_DIR / "lightgbm_l2_advanced_ca_3_fold1.txt"
l2_model.booster_.save_model(str(model_path))

print("\nMODEL SAVED:", model_path)

D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[50]	valid_0's rmse: 3.11064	valid_0's l2: 9.67611
[100]	valid_0's rmse: 3.03886	valid_0's l2: 9.23467
[150]	valid_0's rmse: 3.0128	valid_0's l2: 9.07696
[200]	valid_0's rmse: 3.00067	valid_0's l2: 9.00403
[250]	valid_0's rmse: 2.99539	valid_0's l2: 8.97237
[300]	valid_0's rmse: 2.99391	valid_0's l2: 8.9635
[350]	valid_0's rmse: 2.99147	valid_0's l2: 8.94888
[400]	valid_0's rmse: 2.9863	valid_0's l2: 8.91797
[450]	valid_0's rmse: 2.98545	valid_0's l2: 8.9129
[500]	valid_0's rmse: 2.98335	valid_0's l2: 8.90039
Early stopping, best iteration is:
[499]	valid_0's rmse: 2.98287	valid_0's l2: 8.89751

CA_3 FOLD 1 - LIGHTGBM L2 RESULTS
Best iteration: 499
MAE:       1.4383
RMSE:      3.1609
WAPE:      68.3970%
Bias:      -3.9877%

MODEL SAVED: D:\Retail-Demand\models\lightgbm_l2_advanced_ca_3_fold1.txt


In [16]:
# STEP 31: Train LightGBM Tweedie for CA_3 Fold 1

tweedie_model = lgb.LGBMRegressor(
    objective="tweedie",
    tweedie_variance_power=1.2,
    **COMMON_PARAMS
)

tweedie_model.fit(
    X_train,
    y_train,
    eval_set=[(X_early_stop, y_early_stop)],
    eval_metric="rmse",
    categorical_feature=categorical_features,
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=50)
    ]
)

# Generate validation predictions
ca3_fold1_tweedie_pred = np.maximum(
    tweedie_model.predict(X_valid),
    0
)

# Calculate metrics
actual = y_valid.to_numpy()
predicted = ca3_fold1_tweedie_pred

mae = mean_absolute_error(actual, predicted)
rmse = np.sqrt(mean_squared_error(actual, predicted))
wape = np.abs(actual - predicted).sum() / actual.sum() * 100
bias = (predicted - actual).sum() / actual.sum() * 100

print("\nCA_3 FOLD 1 - LIGHTGBM TWEEDIE RESULTS")
print("Best iteration:", tweedie_model.best_iteration_)
print(f"MAE:       {mae:.4f}")
print(f"RMSE:      {rmse:.4f}")
print(f"WAPE:      {wape:.4f}%")
print(f"Bias:      {bias:.4f}%")

# Save model
model_path = MODEL_DIR / "lightgbm_tweedie_advanced_ca_3_fold1.txt"
tweedie_model.booster_.save_model(str(model_path))

print("\nMODEL SAVED:", model_path)

D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[50]	valid_0's rmse: 3.27375	valid_0's tweedie: 9.639
[100]	valid_0's rmse: 3.18425	valid_0's tweedie: 9.60332
[150]	valid_0's rmse: 3.17222	valid_0's tweedie: 9.59232
[200]	valid_0's rmse: 3.16764	valid_0's tweedie: 9.58835
[250]	valid_0's rmse: 3.16888	valid_0's tweedie: 9.58321
Early stopping, best iteration is:
[212]	valid_0's rmse: 3.16591	valid_0's tweedie: 9.58653

CA_3 FOLD 1 - LIGHTGBM TWEEDIE RESULTS
Best iteration: 212
MAE:       1.4203
RMSE:      2.8879
WAPE:      67.5375%
Bias:      -2.1350%

MODEL SAVED: D:\Retail-Demand\models\lightgbm_tweedie_advanced_ca_3_fold1.txt


In [20]:

# STEP 32: Define Automated Store Forecasting Pipeline

import gc
import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb

from sklearn.metrics import mean_absolute_error, mean_squared_error


def run_store_pipeline(store):

    print("\n" + "=" * 65)
    print(f"STARTING AUTOMATED PIPELINE: {store}")
    print("=" * 65)

    feature_path = FEATURE_FILES[store]
    assert feature_path.exists(), f"Missing feature file: {feature_path}"

    store_code = store.lower()

    fold_results = []

    for fold in [1, 2, 3]:

        print(f"\n{'-' * 55}")
        print(f"{store} - FOLD {fold}")
        print(f"{'-' * 55}")

        config = FOLD_CONFIG[fold]

        # 1. Load only the required columns
        columns = ["date", "sales"] + feature_columns

        source = pl.scan_parquet(feature_path)

        train_pd = (
            source
            .filter(pl.col("date") <= pl.lit(config["train_end"]).str.to_date())
            .select(columns)
            .collect()
            .to_pandas()
        )

        early_pd = (
            source
            .filter(
                (pl.col("date") >= pl.lit(config["early_start"]).str.to_date()) &
                (pl.col("date") <= pl.lit(config["early_end"]).str.to_date())
            )
            .select(columns)
            .collect()
            .to_pandas()
        )

        valid_pd = (
            source
            .filter(
                (pl.col("date") >= pl.lit(config["valid_start"]).str.to_date()) &
                (pl.col("date") <= pl.lit(config["valid_end"]).str.to_date())
            )
            .select(columns)
            .collect()
            .to_pandas()
        )

        assert len(early_pd) == 85372
        assert len(valid_pd) == 85372

        # 2. Separate features and targets
        X_train = train_pd[feature_columns].copy()
        y_train = train_pd["sales"]

        X_early = early_pd[feature_columns].copy()
        y_early = early_pd["sales"]

        X_valid = valid_pd[feature_columns].copy()
        y_valid = valid_pd["sales"]

        # Match categorical levels across all partitions
        for col in categorical_features:

            categories = sorted(
                set(X_train[col].dropna().unique()) |
                set(X_early[col].dropna().unique()) |
                set(X_valid[col].dropna().unique())
            )

            for X in [X_train, X_early, X_valid]:
                X[col] = pd.Categorical(
                    X[col],
                    categories=categories
                )

        actual = y_valid.to_numpy()

        # 3. Generate and save model predictions
        model_predictions = {}

        for model_name in ["l2", "tweedie"]:

            model_path = (
                MODEL_DIR /
                f"lightgbm_{model_name}_advanced_{store_code}_fold{fold}.txt"
            )

            prediction_path = (
                PREDICTION_DIR /
                f"lightgbm_{model_name}_advanced_{store_code}_fold{fold}.parquet"
            )

            if prediction_path.exists():

                print(f"{model_name.upper()}: Existing prediction found.")

                saved_predictions = pd.read_parquet(prediction_path)

                saved_predictions["date"] = pd.to_datetime(
                    saved_predictions["date"]
                ).dt.normalize()

                # Verify saved prediction keys and actuals
                expected_keys = valid_pd[["date"]].copy()
                expected_keys["item_id"] = valid_pd["item_id"].to_numpy()

                check = expected_keys.merge(
                    saved_predictions[["item_id", "date", "actual"]],
                    on=["item_id", "date"],
                    how="left",
                    validate="one_to_one"
                )

                assert check["actual"].notna().all()
                assert np.array_equal(
                    check["actual"].to_numpy(),
                    actual
                )

                prediction_lookup = saved_predictions.set_index(
                    ["item_id", "date"]
                )["prediction"]

                keys = pd.MultiIndex.from_frame(
                    valid_pd[["item_id", "date"]]
                )

                predictions = prediction_lookup.reindex(keys).to_numpy()

                assert not np.isnan(predictions).any()

            else:

    if model_path.exists():

        print(f"{model_name.upper()}: Loading saved model.")

        booster = lgb.Booster(model_file=str(model_path))

        predictions = np.maximum(
            booster.predict(X_valid),
            0
        )

        del booster

    else:

        print(f"{model_name.upper()}: Training new model.")

        if model_name == "l2":

            model = lgb.LGBMRegressor(
                objective="regression_l2",
                **COMMON_PARAMS
            )

        else:

            model = lgb.LGBMRegressor(
                objective="tweedie",
                tweedie_variance_power=1.2,
                **COMMON_PARAMS
            )

        model.fit(
            X_train,
            y_train,
            eval_set=[(X_early, y_early)],
            eval_metric="rmse",
            categorical_feature=categorical_features,
            callbacks=[
                lgb.early_stopping(stopping_rounds=50),
                lgb.log_evaluation(period=100)
            ]
        )

        predictions = np.maximum(
            model.predict(X_valid),
            0
        )

        model.booster_.save_model(str(model_path))

        print("MODEL SAVED:", model_path)

        del model

            model_predictions[model_name] = predictions

            # Save or refresh prediction file
            output = valid_pd[
                ["item_id", "date", "sales"]
            ].copy()

            output["store_id"] = store
            output["actual"] = output.pop("sales")
            output["prediction"] = predictions
            output["fold"] = fold
            output["model"] = model_name

            output["horizon"] = (
                output["date"] -
                pd.Timestamp(config["valid_start"])
            ).dt.days + 1

            output.to_parquet(prediction_path, index=False)

            print(f"{model_name.upper()} predictions saved.")

            # Calculate metrics
            mae = mean_absolute_error(actual, predictions)
            rmse = np.sqrt(mean_squared_error(actual, predictions))
            wape = np.abs(actual - predictions).sum() / actual.sum() * 100
            bias = (predictions - actual).sum() / actual.sum() * 100

            fold_results.append({
                "Store": store,
                "Fold": fold,
                "Model": f"LightGBM {model_name.title()}",
                "MAE": mae,
                "RMSE": rmse,
                "WAPE (%)": wape,
                "Bias (%)": bias
            })

            print(
                f"{model_name.upper()} | "
                f"MAE: {mae:.4f} | "
                f"RMSE: {rmse:.4f} | "
                f"WAPE: {wape:.4f}% | "
                f"Bias: {bias:.4f}%"
            )

            del predictions, output
            gc.collect()

        # 4. Load and prepare baseline predictions
        seasonal = pd.read_parquet(
            PREDICTION_DIR / f"seasonal_naive_fold_{fold}.parquet"
        )

        croston = pd.read_parquet(
            PREDICTION_DIR / f"croston_sba_fold_{fold}.parquet"
        )

        seasonal = seasonal[
            seasonal["store_id"] == store
        ].copy()

        croston = croston[
            croston["store_id"] == store
        ].copy()

        seasonal = seasonal.rename(
            columns={
                "forecast_date": "date",
                "forecast": "prediction"
            }
        )

        croston = croston.rename(
            columns={
                "forecast_date": "date",
                "forecast": "prediction"
            }
        )

        seasonal["date"] = pd.to_datetime(
            seasonal["date"]
        ).dt.normalize()

        croston["date"] = pd.to_datetime(
            croston["date"]
        ).dt.normalize()

        # 5. Build aligned comparison
        comparison = valid_pd[
            ["item_id", "date", "sales"]
        ].copy()

        comparison["date"] = pd.to_datetime(
            comparison["date"]
        ).dt.normalize()

        comparison = comparison.rename(
            columns={"sales": "actual"}
        )

        for name, prediction_df in [
            ("l2", None),
            ("tweedie", None),
            ("croston", croston),
            ("seasonal", seasonal)
        ]:

            if name in ["l2", "tweedie"]:

                values = model_predictions[name]

                comparison[f"{name}_prediction"] = values

            else:

                baseline = prediction_df[
                    ["item_id", "date", "actual", "prediction"]
                ].copy()

                comparison = comparison.merge(
                    baseline[
                        ["item_id", "date", "actual", "prediction"]
                    ].rename(
                        columns={
                            "actual": f"{name}_actual",
                            "prediction": f"{name}_prediction"
                        }
                    ),
                    on=["item_id", "date"],
                    validate="one_to_one"
                )

                assert np.array_equal(
                    comparison["actual"].to_numpy(),
                    comparison[f"{name}_actual"].to_numpy()
                )

        # 6. Calculate all four model metrics
        four_model_results = []

        for name, label in [
            ("l2", "LightGBM L2"),
            ("tweedie", "LightGBM Tweedie"),
            ("croston", "Croston SBA"),
            ("seasonal", "Seasonal Naive")
        ]:

            predictions = comparison[
                f"{name}_prediction"
            ].to_numpy()

            four_model_results.append({
                "Store": store,
                "Fold": fold,
                "Model": label,
                "MAE": mean_absolute_error(actual, predictions),
                "RMSE": np.sqrt(mean_squared_error(actual, predictions)),
                "WAPE (%)": (
                    np.abs(actual - predictions).sum() /
                    actual.sum() * 100
                ),
                "Bias (%)": (
                    (predictions - actual).sum() /
                    actual.sum() * 100
                )
            })

        comparison_df = pd.DataFrame(
            four_model_results
        ).sort_values("WAPE (%)")

        comparison_path = (
            RESULTS_DIR /
            f"{store_code}_fold{fold}_four_model_comparison.csv"
        )

        comparison_df.to_csv(
            comparison_path,
            index=False
        )

        print(f"\n{store} FOLD {fold} FOUR-MODEL COMPARISON")
        display(comparison_df)

        print("Comparison saved:", comparison_path)

        # 7. Clean up before next fold
        del (
            train_pd, early_pd, valid_pd,
            X_train, X_early, X_valid,
            y_train, y_early, y_valid,
            seasonal, croston, comparison,
            comparison_df, model_predictions
        )

        gc.collect()

        print(
            "Available RAM:",
            round(
                psutil.virtual_memory().available / (1024 ** 3),
                2
            ),
            "GB"
        )

    # 8. Consolidate three-fold results
    all_fold_files = [
        RESULTS_DIR / f"{store_code}_fold{i}_four_model_comparison.csv"
        for i in [1, 2, 3]
    ]

    all_folds = pd.concat(
        [pd.read_csv(path) for path in all_fold_files],
        ignore_index=True
    )

    all_folds_path = (
        RESULTS_DIR / f"{store_code}_all_fold_comparisons.csv"
    )

    all_folds.to_csv(all_folds_path, index=False)

    summary = (
        all_folds
        .groupby(["Store", "Model"], as_index=False)
        [["MAE", "RMSE", "WAPE (%)", "Bias (%)"]]
        .mean()
        .sort_values("WAPE (%)")
        .reset_index(drop=True)
    )

    summary_path = (
        RESULTS_DIR / f"{store_code}_three_fold_model_summary.csv"
    )

    summary.to_csv(summary_path, index=False)

    print("\n" + "=" * 65)
    print(f"{store} - THREE-FOLD FINAL SUMMARY")
    print("=" * 65)

    display(summary)

    print("Overall best model:", summary.iloc[0]["Model"])
    print("Summary saved:", summary_path)

    return summary


IndentationError: expected an indented block after 'else' statement on line 152 (3472705856.py, line 154)

In [4]:

# ============================================================
# STEP 32: AUTOMATED MULTI-STORE THREE-FOLD FORECASTING PIPELINE
# ============================================================

import gc
import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb
import psutil

from sklearn.metrics import mean_absolute_error, mean_squared_error


def run_store_pipeline(store):

    print("\n" + "=" * 70)
    print(f"STARTING AUTOMATED PIPELINE: {store}")
    print("=" * 70)

    feature_path = FEATURE_FILES[store]
    assert feature_path.exists(), f"Missing feature file: {feature_path}"

    store_code = store.lower()

    # Keep all four model comparisons for this store
    all_fold_results = []

    # --------------------------------------------------------
    # PROCESS THREE FOLDS
    # --------------------------------------------------------

    for fold in [1, 2, 3]:

        print("\n" + "-" * 65)
        print(f"{store} - FOLD {fold}")
        print("-" * 65)

        config = FOLD_CONFIG[fold]

        print("Available RAM:",
              round(psutil.virtual_memory().available / (1024**3), 2),
              "GB")

        # ----------------------------------------------------
        # 1. LOAD DATA
        # ----------------------------------------------------

        columns = ["date", "sales"] + feature_columns

        source = pl.scan_parquet(feature_path)

        train_pd = (
            source
            .filter(
                pl.col("date") <=
                pl.lit(config["train_end"]).str.to_date()
            )
            .select(columns)
            .collect()
            .to_pandas()
        )

        early_pd = (
            source
            .filter(
                (pl.col("date") >=
                 pl.lit(config["early_start"]).str.to_date()) &
                (pl.col("date") <=
                 pl.lit(config["early_end"]).str.to_date())
            )
            .select(columns)
            .collect()
            .to_pandas()
        )

        valid_pd = (
            source
            .filter(
                (pl.col("date") >=
                 pl.lit(config["valid_start"]).str.to_date()) &
                (pl.col("date") <=
                 pl.lit(config["valid_end"]).str.to_date())
            )
            .select(columns)
            .collect()
            .to_pandas()
        )

        assert len(early_pd) == 85372
        assert len(valid_pd) == 85372

        print("Training rows:", f"{len(train_pd):,}")
        print("Early stopping rows:", f"{len(early_pd):,}")
        print("Validation rows:", f"{len(valid_pd):,}")

        # ----------------------------------------------------
        # 2. PREPARE FEATURES
        # ----------------------------------------------------

        X_train = train_pd[feature_columns].copy()
        y_train = train_pd["sales"]

        X_early = early_pd[feature_columns].copy()
        y_early = early_pd["sales"]

        X_valid = valid_pd[feature_columns].copy()
        y_valid = valid_pd["sales"]

        # Consistent categorical levels
        for col in categorical_features:

            categories = sorted(
                set(X_train[col].dropna().unique()) |
                set(X_early[col].dropna().unique()) |
                set(X_valid[col].dropna().unique())
            )

            for X in [X_train, X_early, X_valid]:

                X[col] = pd.Categorical(
                    X[col],
                    categories=categories
                )

        actual = y_valid.to_numpy()

        # ----------------------------------------------------
        # 3. TRAIN OR REUSE LIGHTGBM MODELS
        # ----------------------------------------------------

        model_predictions = {}

        for model_name in ["l2", "tweedie"]:

            model_path = (
                MODEL_DIR /
                f"lightgbm_{model_name}_advanced_{store_code}_fold{fold}.txt"
            )

            prediction_path = (
                PREDICTION_DIR /
                f"lightgbm_{model_name}_advanced_{store_code}_fold{fold}.parquet"
            )

            predictions = None

            # Reuse a saved prediction only if its keys and
            # actual values match this validation partition.
            if prediction_path.exists():

                print(f"\n{model_name.upper()}: Checking saved predictions.")

                saved = pd.read_parquet(prediction_path)

                saved["date"] = pd.to_datetime(
                    saved["date"]
                ).dt.normalize()

                expected = valid_pd[
                    ["item_id", "date", "sales"]
                ].copy()

                expected["date"] = pd.to_datetime(
                    expected["date"]
                ).dt.normalize()

                expected = expected.rename(
                    columns={"sales": "actual"}
                )

                saved_keys = saved[
                    ["item_id", "date"]
                ].drop_duplicates()

                expected_keys = expected[
                    ["item_id", "date"]
                ].drop_duplicates()

                keys_match = (
                    len(saved) == len(expected) and
                    len(saved_keys) == len(expected_keys) and
                    saved_keys.merge(
                        expected_keys,
                        on=["item_id", "date"],
                        how="inner"
                    ).shape[0] == len(expected)
                )

                if keys_match:

                    checked = expected.merge(
                        saved[
                            ["item_id", "date", "actual", "prediction"]
                        ],
                        on=["item_id", "date"],
                        validate="one_to_one"
                    )

                    actual_matches = np.array_equal(
                        checked["actual_x"].to_numpy(),
                        checked["actual_y"].to_numpy()
                    )

                    if actual_matches:
                        predictions = checked[
                            "prediction"
                        ].to_numpy()

                        print("Existing predictions verified.")

                del saved

            # Train or load model if valid predictions
            # were not available.
            if predictions is None:

                if model_path.exists():

                    print(f"\nLoading saved {model_name.upper()} model.")

                    booster = lgb.Booster(
                        model_file=str(model_path)
                    )

                    predictions = np.maximum(
                        booster.predict(X_valid),
                        0
                    )

                    del booster

                else:

                    print(f"\nTraining new {model_name.upper()} model.")

                    if model_name == "l2":

                        model = lgb.LGBMRegressor(
                            objective="regression_l2",
                            **COMMON_PARAMS
                        )

                    else:

                        model = lgb.LGBMRegressor(
                            objective="tweedie",
                            tweedie_variance_power=1.2,
                            **COMMON_PARAMS
                        )

                    model.fit(
                        X_train,
                        y_train,
                        eval_set=[(X_early, y_early)],
                        eval_metric="rmse",
                        categorical_feature=categorical_features,
                        callbacks=[
                            lgb.early_stopping(
                                stopping_rounds=50
                            ),
                            lgb.log_evaluation(period=100)
                        ]
                    )

                    predictions = np.maximum(
                        model.predict(X_valid),
                        0
                    )

                    model.booster_.save_model(
                        str(model_path)
                    )

                    print("MODEL SAVED:", model_path)

                    del model

                # Save prediction file
                output = valid_pd[
                    ["item_id", "date", "sales"]
                ].copy()

                output["store_id"] = store
                output["actual"] = output.pop("sales")
                output["prediction"] = predictions
                output["fold"] = fold
                output["model"] = model_name

                output["horizon"] = (
                    output["date"] -
                    pd.Timestamp(config["valid_start"])
                ).dt.days + 1

                output.to_parquet(
                    prediction_path,
                    index=False
                )

                print("PREDICTIONS SAVED:", prediction_path)

                del output

            # Ensure correct prediction dimensions
            assert len(predictions) == len(actual)
            assert np.isfinite(predictions).all()

            model_predictions[model_name] = predictions

            # Model metrics
            mae = mean_absolute_error(actual, predictions)

            rmse = np.sqrt(
                mean_squared_error(actual, predictions)
            )

            wape = (
                np.abs(actual - predictions).sum() /
                actual.sum() * 100
            )

            bias = (
                (predictions - actual).sum() /
                actual.sum() * 100
            )

            print(f"\n{model_name.upper()} RESULTS")
            print(f"MAE:  {mae:.4f}")
            print(f"RMSE: {rmse:.4f}")
            print(f"WAPE: {wape:.4f}%")
            print(f"Bias: {bias:.4f}%")

            all_fold_results.append({
                "Store": store,
                "Fold": fold,
                "Model": (
                    "LightGBM L2"
                    if model_name == "l2"
                    else "LightGBM Tweedie"
                ),
                "MAE": mae,
                "RMSE": rmse,
                "WAPE (%)": wape,
                "Bias (%)": bias
            })

            gc.collect()

        # ----------------------------------------------------
        # 4. LOAD BASELINE PREDICTIONS
        # ----------------------------------------------------

        seasonal = pd.read_parquet(
            PREDICTION_DIR /
            f"seasonal_naive_fold_{fold}.parquet"
        )

        croston = pd.read_parquet(
            PREDICTION_DIR /
            f"croston_sba_fold_{fold}.parquet"
        )

        seasonal = seasonal[
            seasonal["store_id"] == store
        ].copy()

        croston = croston[
            croston["store_id"] == store
        ].copy()

        seasonal = seasonal.rename(
            columns={
                "forecast_date": "date",
                "forecast": "prediction"
            }
        )

        croston = croston.rename(
            columns={
                "forecast_date": "date",
                "forecast": "prediction"
            }
        )

        for df in [seasonal, croston]:
            df["date"] = pd.to_datetime(
                df["date"]
            ).dt.normalize()

        assert len(seasonal) == 85372
        assert len(croston) == 85372

        # ----------------------------------------------------
        # 5. ALIGN ALL FOUR MODELS
        # ----------------------------------------------------

        comparison = valid_pd[
            ["item_id", "date", "sales"]
        ].copy()

        comparison["date"] = pd.to_datetime(
            comparison["date"]
        ).dt.normalize()

        comparison = comparison.rename(
            columns={"sales": "actual"}
        )

        comparison["l2_prediction"] = (
            model_predictions["l2"]
        )

        comparison["tweedie_prediction"] = (
            model_predictions["tweedie"]
        )

        for name, baseline in [
            ("croston", croston),
            ("seasonal", seasonal)
        ]:

            baseline = baseline[
                ["item_id", "date", "actual", "prediction"]
            ].rename(
                columns={
                    "actual": f"{name}_actual",
                    "prediction": f"{name}_prediction"
                }
            )

            comparison = comparison.merge(
                baseline,
                on=["item_id", "date"],
                validate="one_to_one"
            )

            assert np.array_equal(
                comparison["actual"].to_numpy(),
                comparison[f"{name}_actual"].to_numpy()
            )

        # ----------------------------------------------------
        # 6. CALCULATE FOUR-MODEL COMPARISON
        # ----------------------------------------------------

        comparison_results = []

        models = {
            "LightGBM L2": "l2_prediction",
            "LightGBM Tweedie": "tweedie_prediction",
            "Croston SBA": "croston_prediction",
            "Seasonal Naive": "seasonal_prediction"
        }

        for label, prediction_column in models.items():

            predictions = comparison[
                prediction_column
            ].to_numpy()

            comparison_results.append({
                "Store": store,
                "Fold": fold,
                "Model": label,
                "MAE": mean_absolute_error(
                    actual, predictions
                ),
                "RMSE": np.sqrt(
                    mean_squared_error(
                        actual, predictions
                    )
                ),
                "WAPE (%)": (
                    np.abs(actual - predictions).sum() /
                    actual.sum() * 100
                ),
                "Bias (%)": (
                    (predictions - actual).sum() /
                    actual.sum() * 100
                )
            })

        comparison_df = pd.DataFrame(
            comparison_results
        ).sort_values(
            "WAPE (%)"
        ).reset_index(drop=True)

        # Save each fold immediately
        comparison_path = (
            RESULTS_DIR /
            f"{store_code}_fold{fold}_four_model_comparison.csv"
        )

        comparison_df.to_csv(
            comparison_path,
            index=False
        )

        print(f"\n{store} FOLD {fold} FOUR-MODEL RESULTS")
        display(comparison_df)

        print("COMPARISON SAVED:", comparison_path)

        # ----------------------------------------------------
        # 7. MEMORY CLEANUP
        # ----------------------------------------------------

        del (
            train_pd, early_pd, valid_pd,
            X_train, X_early, X_valid,
            y_train, y_early, y_valid,
            seasonal, croston, comparison,
            comparison_df, model_predictions
        )

        gc.collect()

        available_ram = round(
            psutil.virtual_memory().available /
            (1024 ** 3),
            2
        )

        print("\nAvailable RAM after fold:", available_ram, "GB")

    # --------------------------------------------------------
    # 8. CONSOLIDATE ALL THREE FOLDS
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print(f"{store} - THREE-FOLD CONSOLIDATION")
    print("=" * 70)

    fold_files = [
        RESULTS_DIR /
        f"{store_code}_fold{i}_four_model_comparison.csv"
        for i in [1, 2, 3]
    ]

    for path in fold_files:
        assert path.exists(), f"Missing comparison file: {path}"

    all_folds = pd.concat(
        [pd.read_csv(path) for path in fold_files],
        ignore_index=True
    )

    all_folds_path = (
        RESULTS_DIR /
        f"{store_code}_all_fold_comparisons.csv"
    )

    all_folds.to_csv(
        all_folds_path,
        index=False
    )

    summary = (
        all_folds
        .groupby(
            ["Store", "Model"],
            as_index=False
        )[
            ["MAE", "RMSE", "WAPE (%)", "Bias (%)"]
        ]
        .mean()
        .sort_values(
            "WAPE (%)"
        )
        .reset_index(drop=True)
    )

    summary_path = (
        RESULTS_DIR /
        f"{store_code}_three_fold_model_summary.csv"
    )

    summary.to_csv(
        summary_path,
        index=False
    )

    print("\nTHREE-FOLD MODEL SUMMARY")
    display(summary)

    print("\nOVERALL BEST MODEL:", summary.iloc[0]["Model"])

    print("\nSAVED FILES:")
    print(all_folds_path)
    print(summary_path)

    print("\nFOLD RECORDS:", len(all_folds))
    print("SUMMARY RECORDS:", len(summary))

    print("\n" + "=" * 70)
    print(f"{store} PIPELINE COMPLETED SUCCESSFULLY")
    print("=" * 70)

    return summary


In [22]:
# STEP 33: Verify the automated pipeline

print("Pipeline ready:", callable(run_store_pipeline))
print("Current RAM:", round(
    psutil.virtual_memory().available / (1024**3), 2
), "GB")

Pipeline ready: True
Current RAM: 6.58 GB


In [23]:
# STEP 34: Execute the complete CA_3 pipeline

ca3_summary = run_store_pipeline("CA_3")


STARTING AUTOMATED PIPELINE: CA_3

-----------------------------------------------------------------
CA_3 - FOLD 1
-----------------------------------------------------------------
Available RAM: 6.76 GB
Training rows: 5,491,249
Early stopping rows: 85,372
Validation rows: 85,372

Loading saved L2 model.
PREDICTIONS SAVED: D:\Retail-Demand\predictions\lightgbm_l2_advanced_ca_3_fold1.parquet

L2 RESULTS
MAE:  1.4383
RMSE: 3.1609
WAPE: 68.3970%
Bias: -3.9877%

Loading saved TWEEDIE model.
PREDICTIONS SAVED: D:\Retail-Demand\predictions\lightgbm_tweedie_advanced_ca_3_fold1.parquet

TWEEDIE RESULTS
MAE:  1.4203
RMSE: 2.8879
WAPE: 67.5375%
Bias: -2.1350%

CA_3 FOLD 1 FOUR-MODEL RESULTS


,Store,Fold,Model,MAE,RMSE,WAPE (%),Bias (%)
0,CA_3,1,LightGBM Tweedie,1.420263,2.887879,67.537464,-2.135045
1,CA_3,1,Croston SBA,1.427388,2.983394,67.876267,-6.464853
2,CA_3,1,LightGBM L2,1.438338,3.160911,68.396987,-3.987743
3,CA_3,1,Seasonal Naive,1.675643,3.598363,79.681503,-3.898491


COMPARISON SAVED: D:\Retail-Demand\results\ca_3_fold1_four_model_comparison.csv

Available RAM after fold: 4.67 GB

-----------------------------------------------------------------
CA_3 - FOLD 2
-----------------------------------------------------------------
Available RAM: 4.67 GB
Training rows: 5,576,621
Early stopping rows: 85,372
Validation rows: 85,372

Training new L2 model.


D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 3.07938	valid_0's l2: 9.48259
Early stopping, best iteration is:
[78]	valid_0's rmse: 3.06726	valid_0's l2: 9.40806
MODEL SAVED: D:\Retail-Demand\models\lightgbm_l2_advanced_ca_3_fold2.txt
PREDICTIONS SAVED: D:\Retail-Demand\predictions\lightgbm_l2_advanced_ca_3_fold2.parquet

L2 RESULTS
MAE:  1.4818
RMSE: 3.2295
WAPE: 69.8985%
Bias: -0.9624%

Training new TWEEDIE model.


D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


... (7 outputs left out)

,Store,Fold,Model,MAE,RMSE,WAPE (%),Bias (%)
0,CA_3,3,LightGBM Tweedie,1.436257,2.865333,69.285635,3.378272
1,CA_3,3,Croston SBA,1.441868,2.959378,69.556268,-0.157222
2,CA_3,3,LightGBM L2,1.447646,2.864965,69.835001,3.120692
3,CA_3,3,Seasonal Naive,1.677833,3.602721,80.939358,1.717786


COMPARISON SAVED: D:\Retail-Demand\results\ca_3_fold3_four_model_comparison.csv

Available RAM after fold: 5.41 GB

CA_3 - THREE-FOLD CONSOLIDATION

THREE-FOLD MODEL SUMMARY


,Store,Model,MAE,RMSE,WAPE (%),Bias (%)
0,CA_3,LightGBM Tweedie,1.438330,2.958146,68.540071,0.274646
1,CA_3,Croston SBA,1.448283,3.069156,69.012464,-3.258165
2,CA_3,LightGBM L2,1.455934,3.085116,69.376818,-0.609806
3,CA_3,Seasonal Naive,1.688868,3.676246,80.476773,-1.636185



OVERALL BEST MODEL: LightGBM Tweedie

SAVED FILES:
D:\Retail-Demand\results\ca_3_all_fold_comparisons.csv
D:\Retail-Demand\results\ca_3_three_fold_model_summary.csv

FOLD RECORDS: 12
SUMMARY RECORDS: 4

CA_3 PIPELINE COMPLETED SUCCESSFULLY


In [24]:
# STEP 35: Run automated pipeline for CA_4

ca4_summary = run_store_pipeline("CA_4")


STARTING AUTOMATED PIPELINE: CA_4

-----------------------------------------------------------------
CA_4 - FOLD 1
-----------------------------------------------------------------
Available RAM: 7.37 GB
Training rows: 5,491,249
Early stopping rows: 85,372
Validation rows: 85,372

Training new L2 model.


D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 1.39855	valid_0's l2: 1.95595
[200]	valid_0's rmse: 1.39642	valid_0's l2: 1.94998
Early stopping, best iteration is:
[186]	valid_0's rmse: 1.39598	valid_0's l2: 1.94877
MODEL SAVED: D:\Retail-Demand\models\lightgbm_l2_advanced_ca_4_fold1.txt
PREDICTIONS SAVED: D:\Retail-Demand\predictions\lightgbm_l2_advanced_ca_4_fold1.parquet

L2 RESULTS
MAE:  0.7582
RMSE: 1.4550
WAPE: 93.2892%
Bias: -2.8732%

Training new TWEEDIE model.


D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 1.40716	valid_0's tweedie: 4.74035
Early stopping, best iteration is:
[111]	valid_0's rmse: 1.40582	valid_0's tweedie: 4.74006
MODEL SAVED: D:\Retail-Demand\models\lightgbm_tweedie_advanced_ca_4_fold1.txt
PREDICTIONS SAVED: D:\Retail-Demand\predictions\lightgbm_tweedie_advanced_ca_4_fold1.parquet

TWEEDIE RESULTS
MAE:  0.7512
RMSE: 1.4325
WAPE: 92.4262%
Bias: -3.4177%

CA_4 FOLD 1 FOUR-MODEL RESULTS


,Store,Fold,Model,MAE,RMSE,WAPE (%),Bias (%)
0,CA_4,1,LightGBM Tweedie,0.751150,1.432455,92.426250,-3.417695
1,CA_4,1,LightGBM L2,0.758164,1.454974,93.289246,-2.873202
2,CA_4,1,Croston SBA,0.782693,1.472844,96.307459,2.359930
3,CA_4,1,Seasonal Naive,0.873261,1.891631,107.451500,1.167450


... (11 outputs left out)

,Store,Fold,Model,MAE,RMSE,WAPE (%),Bias (%)
0,CA_4,3,LightGBM Tweedie,0.786261,1.411061,90.448657,-2.583420
1,CA_4,3,LightGBM L2,0.790896,1.411039,90.981895,-2.856882
2,CA_4,3,Croston SBA,0.800178,1.432289,92.049645,-2.079923
3,CA_4,3,Seasonal Naive,0.933503,1.941397,107.386846,-0.384030


COMPARISON SAVED: D:\Retail-Demand\results\ca_4_fold3_four_model_comparison.csv

Available RAM after fold: 5.79 GB

CA_4 - THREE-FOLD CONSOLIDATION

THREE-FOLD MODEL SUMMARY


,Store,Model,MAE,RMSE,WAPE (%),Bias (%)
0,CA_4,LightGBM Tweedie,0.770146,1.429947,91.144254,-3.416116
1,CA_4,LightGBM L2,0.777912,1.441613,92.064472,-3.382550
2,CA_4,Croston SBA,0.794928,1.466715,94.099593,-0.676509
3,CA_4,Seasonal Naive,0.902158,1.913060,106.745519,-1.562963



OVERALL BEST MODEL: LightGBM Tweedie

SAVED FILES:
D:\Retail-Demand\results\ca_4_all_fold_comparisons.csv
D:\Retail-Demand\results\ca_4_three_fold_model_summary.csv

FOLD RECORDS: 12
SUMMARY RECORDS: 4

CA_4 PIPELINE COMPLETED SUCCESSFULLY


In [25]:
# STEP 36: Run automated pipeline for TX_1

tx1_summary = run_store_pipeline("TX_1")


STARTING AUTOMATED PIPELINE: TX_1

-----------------------------------------------------------------
TX_1 - FOLD 1
-----------------------------------------------------------------
Available RAM: 6.0 GB
Training rows: 5,491,249
Early stopping rows: 85,372
Validation rows: 85,372

Training new L2 model.


D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 1.8311	valid_0's l2: 3.35294
[200]	valid_0's rmse: 1.8053	valid_0's l2: 3.25912
[300]	valid_0's rmse: 1.79783	valid_0's l2: 3.23219
[400]	valid_0's rmse: 1.79282	valid_0's l2: 3.2142
[500]	valid_0's rmse: 1.79025	valid_0's l2: 3.205
[600]	valid_0's rmse: 1.7888	valid_0's l2: 3.19981
Early stopping, best iteration is:
[550]	valid_0's rmse: 1.78775	valid_0's l2: 3.19605
MODEL SAVED: D:\Retail-Demand\models\lightgbm_l2_advanced_tx_1_fold1.txt
PREDICTIONS SAVED: D:\Retail-Demand\predictions\lightgbm_l2_advanced_tx_1_fold1.parquet

L2 RESULTS
MAE:  0.8830
RMSE: 1.9035
WAPE: 83.1175%
Bias: -2.8706%

Training new TWEEDIE model.


D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 1.91773	valid_0's tweedie: 5.45976
Early stopping, best iteration is:
[104]	valid_0's rmse: 1.9143	valid_0's tweedie: 5.45937
MODEL SAVED: D:\Retail-Demand\models\lightgbm_tweedie_advanced_tx_1_fold1.txt
PREDICTIONS SAVED: D:\Retail-Demand\predictions\lightgbm_tweedie_advanced_tx_1_fold1.parquet

TWEEDIE RESULTS
MAE:  0.8774
RMSE: 1.8430
WAPE: 82.5947%
Bias: -3.8698%

TX_1 FOLD 1 FOUR-MODEL RESULTS


,Store,Fold,Model,MAE,RMSE,WAPE (%),Bias (%)
0,TX_1,1,LightGBM Tweedie,0.877397,1.843008,82.594676,-3.869761
1,TX_1,1,LightGBM L2,0.882951,1.903478,83.117548,-2.870636
2,TX_1,1,Croston SBA,0.900172,1.917203,84.738681,-7.458434
3,TX_1,1,Seasonal Naive,0.953685,2.164059,89.776161,-13.053258


... (11 outputs left out)

,Store,Fold,Model,MAE,RMSE,WAPE (%),Bias (%)
0,TX_1,3,LightGBM Tweedie,0.871821,1.823041,80.224479,-0.992283
1,TX_1,3,Croston SBA,0.883750,1.797573,81.322252,0.978837
2,TX_1,3,LightGBM L2,0.884018,1.835833,81.346882,-1.393113
3,TX_1,3,Seasonal Naive,1.005622,2.310899,92.536863,-0.168147


COMPARISON SAVED: D:\Retail-Demand\results\tx_1_fold3_four_model_comparison.csv

Available RAM after fold: 6.21 GB

TX_1 - THREE-FOLD CONSOLIDATION

THREE-FOLD MODEL SUMMARY


,Store,Model,MAE,RMSE,WAPE (%),Bias (%)
0,TX_1,LightGBM Tweedie,0.874315,1.821304,81.105985,-2.364716
1,TX_1,LightGBM L2,0.884741,1.857591,82.069845,-2.726720
2,TX_1,Croston SBA,0.890274,1.854831,82.591125,-2.887273
3,TX_1,Seasonal Naive,0.978955,2.229084,90.792664,-7.367716



OVERALL BEST MODEL: LightGBM Tweedie

SAVED FILES:
D:\Retail-Demand\results\tx_1_all_fold_comparisons.csv
D:\Retail-Demand\results\tx_1_three_fold_model_summary.csv

FOLD RECORDS: 12
SUMMARY RECORDS: 4

TX_1 PIPELINE COMPLETED SUCCESSFULLY


In [26]:
# STEP 37: Run automated pipeline for TX_2
tx2_summary = run_store_pipeline("TX_2")


STARTING AUTOMATED PIPELINE: TX_2

-----------------------------------------------------------------
TX_2 - FOLD 1
-----------------------------------------------------------------
Available RAM: 6.69 GB
Training rows: 5,491,249
Early stopping rows: 85,372
Validation rows: 85,372

Training new L2 model.


D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 2.02777	valid_0's l2: 4.11183
[200]	valid_0's rmse: 2.00674	valid_0's l2: 4.02702
[300]	valid_0's rmse: 1.99727	valid_0's l2: 3.9891
[400]	valid_0's rmse: 1.99298	valid_0's l2: 3.97197
Early stopping, best iteration is:
[394]	valid_0's rmse: 1.99267	valid_0's l2: 3.97074
MODEL SAVED: D:\Retail-Demand\models\lightgbm_l2_advanced_tx_2_fold1.txt
PREDICTIONS SAVED: D:\Retail-Demand\predictions\lightgbm_l2_advanced_tx_2_fold1.parquet

L2 RESULTS
MAE:  1.0478
RMSE: 2.2300
WAPE: 76.0011%
Bias: -6.1520%

Training new TWEEDIE model.


D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 2.15936	valid_0's tweedie: 6.53778
Early stopping, best iteration is:
[70]	valid_0's rmse: 2.14665	valid_0's tweedie: 6.54519
MODEL SAVED: D:\Retail-Demand\models\lightgbm_tweedie_advanced_tx_2_fold1.txt
PREDICTIONS SAVED: D:\Retail-Demand\predictions\lightgbm_tweedie_advanced_tx_2_fold1.parquet

TWEEDIE RESULTS
MAE:  1.0490
RMSE: 2.2320
WAPE: 76.0839%
Bias: -9.6085%

TX_2 FOLD 1 FOUR-MODEL RESULTS


,Store,Fold,Model,MAE,RMSE,WAPE (%),Bias (%)
0,TX_2,1,LightGBM L2,1.047842,2.230020,76.001132,-6.152035
1,TX_2,1,LightGBM Tweedie,1.048983,2.232047,76.083880,-9.608503
2,TX_2,1,Croston SBA,1.074277,2.371256,77.918471,-11.448375
3,TX_2,1,Seasonal Naive,1.157171,2.614290,83.930877,-13.759940


... (11 outputs left out)

,Store,Fold,Model,MAE,RMSE,WAPE (%),Bias (%)
0,TX_2,3,LightGBM Tweedie,0.993704,1.916253,77.048062,5.394095
1,TX_2,3,LightGBM L2,1.001490,1.909799,77.651712,5.835535
2,TX_2,3,Croston SBA,1.019507,2.034802,79.048688,5.394614
3,TX_2,3,Seasonal Naive,1.175866,2.515961,91.172143,5.334859


COMPARISON SAVED: D:\Retail-Demand\results\tx_2_fold3_four_model_comparison.csv

Available RAM after fold: 6.1 GB

TX_2 - THREE-FOLD CONSOLIDATION

THREE-FOLD MODEL SUMMARY


,Store,Model,MAE,RMSE,WAPE (%),Bias (%)
0,TX_2,LightGBM Tweedie,1.020716,2.054898,76.012223,-1.713486
1,TX_2,LightGBM L2,1.026449,2.057551,76.444373,-0.914327
2,TX_2,Croston SBA,1.043274,2.166797,77.696667,-1.715534
3,TX_2,Seasonal Naive,1.169821,2.556464,87.180328,-3.862937



OVERALL BEST MODEL: LightGBM Tweedie

SAVED FILES:
D:\Retail-Demand\results\tx_2_all_fold_comparisons.csv
D:\Retail-Demand\results\tx_2_three_fold_model_summary.csv

FOLD RECORDS: 12
SUMMARY RECORDS: 4

TX_2 PIPELINE COMPLETED SUCCESSFULLY


In [27]:
# STEP 38: Run automated pipeline for TX_3
tx3_summary = run_store_pipeline("TX_3")


STARTING AUTOMATED PIPELINE: TX_3

-----------------------------------------------------------------
TX_3 - FOLD 1
-----------------------------------------------------------------
Available RAM: 6.16 GB
Training rows: 5,491,249
Early stopping rows: 85,372
Validation rows: 85,372

Training new L2 model.


D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 1.91788	valid_0's l2: 3.67826
[200]	valid_0's rmse: 1.89221	valid_0's l2: 3.58047
[300]	valid_0's rmse: 1.88441	valid_0's l2: 3.55099
[400]	valid_0's rmse: 1.87946	valid_0's l2: 3.53236
[500]	valid_0's rmse: 1.87797	valid_0's l2: 3.52678
[600]	valid_0's rmse: 1.87693	valid_0's l2: 3.52286
Early stopping, best iteration is:
[592]	valid_0's rmse: 1.87621	valid_0's l2: 3.52018
MODEL SAVED: D:\Retail-Demand\models\lightgbm_l2_advanced_tx_3_fold1.txt
PREDICTIONS SAVED: D:\Retail-Demand\predictions\lightgbm_l2_advanced_tx_3_fold1.parquet

L2 RESULTS
MAE:  1.0137
RMSE: 2.2157
WAPE: 78.8534%
Bias: -5.6093%

Training new TWEEDIE model.


D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 1.98282	valid_0's tweedie: 6.30675
Early stopping, best iteration is:
[136]	valid_0's rmse: 1.9709	valid_0's tweedie: 6.30316
MODEL SAVED: D:\Retail-Demand\models\lightgbm_tweedie_advanced_tx_3_fold1.txt
PREDICTIONS SAVED: D:\Retail-Demand\predictions\lightgbm_tweedie_advanced_tx_3_fold1.parquet

TWEEDIE RESULTS
MAE:  1.0017
RMSE: 2.0837
WAPE: 77.9170%
Bias: -5.4706%

TX_3 FOLD 1 FOUR-MODEL RESULTS


,Store,Fold,Model,MAE,RMSE,WAPE (%),Bias (%)
0,TX_3,1,LightGBM Tweedie,1.001689,2.083732,77.916959,-5.470644
1,TX_3,1,LightGBM L2,1.013728,2.215700,78.853405,-5.609259
2,TX_3,1,Croston SBA,1.021327,2.197713,79.444534,-8.877594
3,TX_3,1,Seasonal Naive,1.094281,2.442807,85.119313,-9.582426


... (11 outputs left out)

,Store,Fold,Model,MAE,RMSE,WAPE (%),Bias (%)
0,TX_3,3,LightGBM Tweedie,0.969238,1.872608,77.752528,0.438138
1,TX_3,3,LightGBM L2,0.977355,1.879460,78.403685,0.740463
2,TX_3,3,Croston SBA,0.991111,1.996903,79.507198,0.633425
3,TX_3,3,Seasonal Naive,1.136626,2.587254,91.180395,-0.543121


COMPARISON SAVED: D:\Retail-Demand\results\tx_3_fold3_four_model_comparison.csv

Available RAM after fold: 6.48 GB

TX_3 - THREE-FOLD CONSOLIDATION

THREE-FOLD MODEL SUMMARY


,Store,Model,MAE,RMSE,WAPE (%),Bias (%)
0,TX_3,LightGBM Tweedie,0.983197,1.981025,77.671323,-1.623317
1,TX_3,LightGBM L2,0.993878,2.026491,78.513626,-1.842971
2,TX_3,Croston SBA,0.998347,2.062156,78.869162,-2.798905
3,TX_3,Seasonal Naive,1.106034,2.475540,87.407118,-6.074582



OVERALL BEST MODEL: LightGBM Tweedie

SAVED FILES:
D:\Retail-Demand\results\tx_3_all_fold_comparisons.csv
D:\Retail-Demand\results\tx_3_three_fold_model_summary.csv

FOLD RECORDS: 12
SUMMARY RECORDS: 4

TX_3 PIPELINE COMPLETED SUCCESSFULLY


In [28]:
# STEP 39: Run automated pipeline for WI_1
wi1_summary = run_store_pipeline("WI_1")


STARTING AUTOMATED PIPELINE: WI_1

-----------------------------------------------------------------
WI_1 - FOLD 1
-----------------------------------------------------------------
Available RAM: 7.09 GB
Training rows: 5,491,249
Early stopping rows: 85,372
Validation rows: 85,372

Training new L2 model.


D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 1.75615	valid_0's l2: 3.08405
[200]	valid_0's rmse: 1.75149	valid_0's l2: 3.06771
[300]	valid_0's rmse: 1.75117	valid_0's l2: 3.06659
Early stopping, best iteration is:
[279]	valid_0's rmse: 1.74979	valid_0's l2: 3.06176
MODEL SAVED: D:\Retail-Demand\models\lightgbm_l2_advanced_wi_1_fold1.txt
PREDICTIONS SAVED: D:\Retail-Demand\predictions\lightgbm_l2_advanced_wi_1_fold1.parquet

L2 RESULTS
MAE:  0.9838
RMSE: 1.7979
WAPE: 79.1113%
Bias: -5.4363%

Training new TWEEDIE model.


D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 1.76405	valid_0's tweedie: 6.47484
Early stopping, best iteration is:
[118]	valid_0's rmse: 1.76192	valid_0's tweedie: 6.47259
MODEL SAVED: D:\Retail-Demand\models\lightgbm_tweedie_advanced_wi_1_fold1.txt
PREDICTIONS SAVED: D:\Retail-Demand\predictions\lightgbm_tweedie_advanced_wi_1_fold1.parquet

TWEEDIE RESULTS
MAE:  0.9808
RMSE: 1.8028
WAPE: 78.8676%
Bias: -5.4205%

WI_1 FOLD 1 FOUR-MODEL RESULTS


,Store,Fold,Model,MAE,RMSE,WAPE (%),Bias (%)
0,WI_1,1,LightGBM Tweedie,0.980820,1.802760,78.867607,-5.420507
1,WI_1,1,LightGBM L2,0.983850,1.797888,79.111275,-5.436278
2,WI_1,1,Croston SBA,1.038476,1.911187,83.503760,-0.342953
3,WI_1,1,Seasonal Naive,1.148280,2.283135,92.333123,-3.348372


... (11 outputs left out)

,Store,Fold,Model,MAE,RMSE,WAPE (%),Bias (%)
0,WI_1,3,LightGBM Tweedie,0.962335,1.707844,76.580614,-4.119980
1,WI_1,3,LightGBM L2,0.967994,1.708647,77.030989,-4.026907
2,WI_1,3,Croston SBA,1.031667,1.877611,82.097921,0.729000
3,WI_1,3,Seasonal Naive,1.177283,2.379069,93.685741,-1.026277


COMPARISON SAVED: D:\Retail-Demand\results\wi_1_fold3_four_model_comparison.csv

Available RAM after fold: 6.38 GB

WI_1 - THREE-FOLD CONSOLIDATION

THREE-FOLD MODEL SUMMARY


,Store,Model,MAE,RMSE,WAPE (%),Bias (%)
0,WI_1,LightGBM Tweedie,0.977082,1.787145,78.235718,-4.550565
1,WI_1,LightGBM L2,0.982816,1.788746,78.694726,-4.598372
2,WI_1,Croston SBA,1.037578,1.916272,83.077427,0.792185
3,WI_1,Seasonal Naive,1.162770,2.328425,93.095893,-2.687819



OVERALL BEST MODEL: LightGBM Tweedie

SAVED FILES:
D:\Retail-Demand\results\wi_1_all_fold_comparisons.csv
D:\Retail-Demand\results\wi_1_three_fold_model_summary.csv

FOLD RECORDS: 12
SUMMARY RECORDS: 4

WI_1 PIPELINE COMPLETED SUCCESSFULLY


In [1]:
# STEP 40: Run automated pipeline for WI_2
wi2_summary = run_store_pipeline("WI_2")

NameError: name 'run_store_pipeline' is not defined

In [2]:
def run_store_pipeline

SyntaxError: expected '(' (2499556196.py, line 1)

In [3]:

# STEP 40A: Verify restored environment

print("Pipeline:", callable(run_store_pipeline))
print("WI_2 feature file:", FEATURE_FILES["WI_2"].exists())
print("Model directory:", MODEL_DIR.exists())
print("Prediction directory:", PREDICTION_DIR.exists())
print("Results directory:", RESULTS_DIR.exists())


NameError: name 'run_store_pipeline' is not defined

In [5]:
# STEP 40A: Verify restored pipeline

print("Pipeline:", callable(run_store_pipeline))
print("WI_2 feature file:", FEATURE_FILES["WI_2"].exists())
print("Model directory:", MODEL_DIR.exists())
print("Prediction directory:", PREDICTION_DIR.exists())
print("Results directory:", RESULTS_DIR.exists())

Pipeline: True


NameError: name 'FEATURE_FILES' is not defined

In [7]:
# STEP 40A: Verify restored configuration

print("Pipeline:", callable(run_store_pipeline))
print("Feature files:", list(FEATURE_FILES.keys()))
print("WI_2 file exists:", FEATURE_FILES["WI_2"].exists())
print("Model directory:", MODEL_DIR.exists())
print("Prediction directory:", PREDICTION_DIR.exists())
print("Results directory:", RESULTS_DIR.exists())

Pipeline: True
Feature files: ['CA_2', 'CA_3', 'CA_4', 'TX_1', 'TX_2', 'TX_3', 'WI_1', 'WI_2', 'WI_3']
WI_2 file exists: True
Model directory: True
Prediction directory: True
Results directory: True


In [8]:
# STEP 40B: Resume WI_2 automated pipeline

wi2_summary = run_store_pipeline("WI_2")


STARTING AUTOMATED PIPELINE: WI_2

-----------------------------------------------------------------
WI_2 - FOLD 1
-----------------------------------------------------------------
Available RAM: 7.34 GB
Training rows: 5,491,249
Early stopping rows: 85,372
Validation rows: 85,372

Training new L2 model.


D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 3.06976	valid_0's l2: 9.42344
Early stopping, best iteration is:
[105]	valid_0's rmse: 3.06837	valid_0's l2: 9.41491
MODEL SAVED: D:\Retail-Demand\models\lightgbm_l2_advanced_wi_2_fold1.txt
PREDICTIONS SAVED: D:\Retail-Demand\predictions\lightgbm_l2_advanced_wi_2_fold1.parquet

L2 RESULTS
MAE:  1.3304
RMSE: 3.4656
WAPE: 75.8568%
Bias: -16.1266%

Training new TWEEDIE model.


D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 3.072	valid_0's tweedie: 7.44432
[200]	valid_0's rmse: 3.06418	valid_0's tweedie: 7.42923
Early stopping, best iteration is:
[190]	valid_0's rmse: 3.06236	valid_0's tweedie: 7.42967
MODEL SAVED: D:\Retail-Demand\models\lightgbm_tweedie_advanced_wi_2_fold1.txt
PREDICTIONS SAVED: D:\Retail-Demand\predictions\lightgbm_tweedie_advanced_wi_2_fold1.parquet

TWEEDIE RESULTS
MAE:  1.3179
RMSE: 3.4173
WAPE: 75.1463%
Bias: -12.6532%

WI_2 FOLD 1 FOUR-MODEL RESULTS


,Store,Fold,Model,MAE,RMSE,WAPE (%),Bias (%)
0,WI_2,1,LightGBM Tweedie,1.317894,3.417287,75.146293,-12.653162
1,WI_2,1,LightGBM L2,1.330356,3.465557,75.856844,-16.126609
2,WI_2,1,Croston SBA,1.362484,3.538561,77.688770,-17.028150
3,WI_2,1,Seasonal Naive,1.473282,4.046344,84.006465,-21.941185


... (11 outputs left out)

,Store,Fold,Model,MAE,RMSE,WAPE (%),Bias (%)
0,WI_2,3,LightGBM Tweedie,1.241389,3.093443,70.157435,-9.037541
1,WI_2,3,LightGBM L2,1.243845,3.029339,70.296255,-7.579819
2,WI_2,3,Croston SBA,1.311529,3.303143,74.121440,-8.652827
3,WI_2,3,Seasonal Naive,1.418685,3.652242,80.177413,-17.550642


COMPARISON SAVED: D:\Retail-Demand\results\wi_2_fold3_four_model_comparison.csv

Available RAM after fold: 3.99 GB

WI_2 - THREE-FOLD CONSOLIDATION

THREE-FOLD MODEL SUMMARY


,Store,Model,MAE,RMSE,WAPE (%),Bias (%)
0,WI_2,LightGBM Tweedie,1.270549,3.269429,72.552765,-11.286343
1,WI_2,LightGBM L2,1.284284,3.256230,73.342126,-11.137977
2,WI_2,Croston SBA,1.325347,3.371424,75.677992,-10.308087
3,WI_2,Seasonal Naive,1.446747,3.818582,82.618978,-17.791015



OVERALL BEST MODEL: LightGBM Tweedie

SAVED FILES:
D:\Retail-Demand\results\wi_2_all_fold_comparisons.csv
D:\Retail-Demand\results\wi_2_three_fold_model_summary.csv

FOLD RECORDS: 12
SUMMARY RECORDS: 4

WI_2 PIPELINE COMPLETED SUCCESSFULLY


In [9]:
# STEP 41: Run automated pipeline for WI_3
wi3_summary = run_store_pipeline("WI_3")


STARTING AUTOMATED PIPELINE: WI_3

-----------------------------------------------------------------
WI_3 - FOLD 1
-----------------------------------------------------------------
Available RAM: 4.22 GB
Training rows: 5,491,249
Early stopping rows: 85,372
Validation rows: 85,372

Training new L2 model.


D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 2.15774	valid_0's l2: 4.65584
[200]	valid_0's rmse: 2.12628	valid_0's l2: 4.52106
[300]	valid_0's rmse: 2.11675	valid_0's l2: 4.48065
Early stopping, best iteration is:
[290]	valid_0's rmse: 2.11538	valid_0's l2: 4.47482
MODEL SAVED: D:\Retail-Demand\models\lightgbm_l2_advanced_wi_3_fold1.txt
PREDICTIONS SAVED: D:\Retail-Demand\predictions\lightgbm_l2_advanced_wi_3_fold1.parquet

L2 RESULTS
MAE:  1.0138
RMSE: 2.5628
WAPE: 77.1727%
Bias: -8.8865%

Training new TWEEDIE model.


D:\Retail-Demand\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 2.26019	valid_0's tweedie: 6.16266
[200]	valid_0's rmse: 2.19739	valid_0's tweedie: 6.14938
[300]	valid_0's rmse: 2.1546	valid_0's tweedie: 6.14368
[400]	valid_0's rmse: 2.12412	valid_0's tweedie: 6.13663
Early stopping, best iteration is:
[439]	valid_0's rmse: 2.12248	valid_0's tweedie: 6.13437
MODEL SAVED: D:\Retail-Demand\models\lightgbm_tweedie_advanced_wi_3_fold1.txt
PREDICTIONS SAVED: D:\Retail-Demand\predictions\lightgbm_tweedie_advanced_wi_3_fold1.parquet

TWEEDIE RESULTS
MAE:  1.0017
RMSE: 2.4040
WAPE: 76.2504%
Bias: -6.2505%

WI_3 FOLD 1 FOUR-MODEL RESULTS


,Store,Fold,Model,MAE,RMSE,WAPE (%),Bias (%)
0,WI_3,1,LightGBM Tweedie,1.001717,2.403997,76.250358,-6.250506
1,WI_3,1,LightGBM L2,1.013834,2.562764,77.172689,-8.886512
2,WI_3,1,Croston SBA,1.038665,2.519803,79.062806,-12.275339
3,WI_3,1,Seasonal Naive,1.095289,2.719145,83.373011,-16.643930


... (11 outputs left out)

,Store,Fold,Model,MAE,RMSE,WAPE (%),Bias (%)
0,WI_3,3,LightGBM Tweedie,0.963529,2.068919,73.569806,-6.813531
1,WI_3,3,LightGBM L2,0.978748,2.068716,74.731809,-5.035578
2,WI_3,3,Croston SBA,1.015628,2.254376,77.547796,-4.165962
3,WI_3,3,Seasonal Naive,1.111957,2.591257,84.902960,-10.931044


COMPARISON SAVED: D:\Retail-Demand\results\wi_3_fold3_four_model_comparison.csv

Available RAM after fold: 5.03 GB

WI_3 - THREE-FOLD CONSOLIDATION

THREE-FOLD MODEL SUMMARY


,Store,Model,MAE,RMSE,WAPE (%),Bias (%)
0,WI_3,LightGBM Tweedie,0.976391,2.217476,74.745687,-6.389635
1,WI_3,LightGBM L2,0.991956,2.278992,75.938643,-6.190084
2,WI_3,Croston SBA,1.015987,2.347672,77.774501,-6.211156
3,WI_3,Seasonal Naive,1.095597,2.619113,83.872933,-12.920682



OVERALL BEST MODEL: LightGBM Tweedie

SAVED FILES:
D:\Retail-Demand\results\wi_3_all_fold_comparisons.csv
D:\Retail-Demand\results\wi_3_three_fold_model_summary.csv

FOLD RECORDS: 12
SUMMARY RECORDS: 4

WI_3 PIPELINE COMPLETED SUCCESSFULLY
